# Temporal EEG state knowledge graph experiments

This notebook contains the preprocessing, chronological cohort construction, GNN baselines, continual-learning experiments, temporal knowledge-graph experiments, and paper-level result generation.

Run `01_protocol_audit.ipynb` first. The CHB-MIT EDF files are not included in this repository. Change the path variables in the setup cells to match the local or Google Drive location of the dataset and project directory.


In [ ]:

!pip install -q "mne>=1.8,<2"

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib
import json
import re
import time

import mne
import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt, welch

mne.set_log_level("ERROR")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

DATA_ROOT = Path("/content/drive/MyDrive/chb-mit-scalp-eeg-database-1.0.0")
PROTOCOL_ROOT = Path(
    "/content/drive/MyDrive/EEG_Research/"
    "continual_graph_forecasting/protocol_v2"
)
CACHE_ROOT = Path(
    "/content/drive/MyDrive/EEG_Research/"
    "continual_graph_forecasting/graph_cache_v1"
)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

required = [
    PROTOCOL_ROOT / "protocol.json",
    PROTOCOL_ROOT / "fixed_channels.json",
    PROTOCOL_ROOT / "file_manifest_v2.csv",
    PROTOCOL_ROOT / "window_manifest_v2.pkl",
    PROTOCOL_ROOT / "seizure_manifest_v2.csv",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing protocol_v2 files:\n" + "\n".join(missing)
    )

protocol_payload = json.loads((PROTOCOL_ROOT / "protocol.json").read_text())
protocol_v2_hash = protocol_payload["protocol_hash"]
protocol = protocol_payload["protocol"]

FIXED_CHANNELS = tuple(
    json.loads((PROTOCOL_ROOT / "fixed_channels.json").read_text())
)

files_v2 = pd.read_csv(PROTOCOL_ROOT / "file_manifest_v2.csv")
windows_v2 = pd.read_pickle(PROTOCOL_ROOT / "window_manifest_v2.pkl")
events_v2 = pd.read_csv(PROTOCOL_ROOT / "seizure_manifest_v2.csv")

assert len(FIXED_CHANNELS) == 22
assert files_v2["edf_sfreq_hz"].eq(256.0).all()

print("Protocol v2 hash:", protocol_v2_hash)
print("Fixed channels:", len(FIXED_CHANNELS))
print("Usable EDF files:", int(files_v2["montage_compatible"].sum()))
print("Window rows:", len(windows_v2))
print("CACHE_ROOT:", CACHE_ROOT)


In [ ]:

PREPROCESS_CONFIG = {
    "source_protocol_v2_hash": protocol_v2_hash,
    "sampling_rate_hz": 256.0,
    "filter": {
        "type": "Butterworth bandpass",
        "order": 4,
        "low_hz": 0.5,
        "high_hz": 45.0,
        "phase": "zero_phase_sosfiltfilt",
        "filter_scope": "whole_edf_independently",
    },
    "window_sec": float(protocol["windowing"]["window_sec"]),
    "stride_sec": float(protocol["windowing"]["stride_sec"]),
    "bands_hz": {
        "delta": [0.5, 4.0],
        "theta": [4.0, 8.0],
        "alpha": [8.0, 13.0],
        "beta": [13.0, 30.0],
        "gamma": [30.0, 45.0],
    },
    "node_features": "log10_absolute_bandpower",
    "connectivity": "signed_pearson_correlation",
    "connectivity_diagonal_saved": False,
    "hard_qc": {
        "reject_nonfinite": True,
        "reject_zero_variance_channel": True,
        "amplitude_threshold_rejection": False,
    },
    "normalization": "none_in_cache; fit later on allowed training data only",
}

def canonical_json(obj):
    return json.dumps(obj, sort_keys=True, separators=(",", ":"))

preprocess_hash = hashlib.sha256(
    canonical_json(PREPROCESS_CONFIG).encode("utf-8")
).hexdigest()

config_path = CACHE_ROOT / "preprocess_config.json"
payload = {
    "preprocess_hash": preprocess_hash,
    "config": PREPROCESS_CONFIG,
}

if config_path.exists():
    existing = json.loads(config_path.read_text())
    if existing.get("preprocess_hash") != preprocess_hash:
        raise RuntimeError(
            "graph_cache_v1 already contains a different preprocessing "
            "configuration. Do not overwrite it silently."
        )
else:
    config_path.write_text(json.dumps(payload, indent=2))

print("Preprocess hash:", preprocess_hash)
print(json.dumps(PREPROCESS_CONFIG, indent=2))


In [ ]:

SFREQ = 256.0
WINDOW_SEC = PREPROCESS_CONFIG["window_sec"]
WINDOW_SAMPLES = int(round(WINDOW_SEC * SFREQ))

BANDS = PREPROCESS_CONFIG["bands_hz"]
BAND_NAMES = tuple(BANDS.keys())

SOS = butter(
    N=4,
    Wn=[0.5, 45.0],
    btype="bandpass",
    fs=SFREQ,
    output="sos",
)

TRIU_I, TRIU_J = np.triu_indices(len(FIXED_CHANNELS), k=1)
N_CONNECTIVITY = len(TRIU_I)
assert N_CONNECTIVITY == 231

def canonical_channel_name(name):
    x = name.strip().upper()
    x = re.sub(r"-(\d+)$", "", x)
    return x

def build_channel_pick_map(raw_names):
    canon_to_actual = {}

    for actual in raw_names:
        canonical = canonical_channel_name(actual)
        if canonical in FIXED_CHANNELS:
            canon_to_actual.setdefault(canonical, []).append(actual)

    missing = [ch for ch in FIXED_CHANNELS if ch not in canon_to_actual]
    ambiguous = {
        ch: names
        for ch, names in canon_to_actual.items()
        if len(names) != 1
    }

    if missing:
        raise RuntimeError(f"Missing frozen channels: {missing}")
    if ambiguous:
        raise RuntimeError(f"Ambiguous channel mappings: {ambiguous}")

    return [canon_to_actual[ch][0] for ch in FIXED_CHANNELS]

def compute_window_features(x):
    x = np.asarray(x, dtype=np.float64)

    qc_finite = np.isfinite(x).all(axis=(1, 2))

    channel_std = np.nanstd(x, axis=2)
    qc_nonzero_variance = np.all(channel_std > 0.0, axis=1)

    max_abs_uv = (
        np.nanmax(np.abs(x), axis=(1, 2)) * 1e6
    ).astype(np.float32)

    min_channel_std_uv = (
        np.nanmin(channel_std, axis=1) * 1e6
    ).astype(np.float32)

    nperseg = int(2 * SFREQ)
    noverlap = nperseg // 2

    freqs, psd = welch(
        x,
        fs=SFREQ,
        window="hann",
        nperseg=nperseg,
        noverlap=noverlap,
        detrend="constant",
        scaling="density",
        axis=-1,
    )

    band_power = []

    for band_name in BAND_NAMES:
        low, high = BANDS[band_name]

        if band_name != BAND_NAMES[-1]:
            mask = (freqs >= low) & (freqs < high)
        else:
            mask = (freqs >= low) & (freqs <= high)

        if mask.sum() < 2:
            raise RuntimeError(
                f"Too few Welch bins for {band_name}: {mask.sum()}"
            )

        power = np.trapezoid(
              psd[..., mask],
              freqs[mask],
              axis=-1,
         )
        band_power.append(power)

    band_power = np.stack(band_power, axis=-1)

    eps = np.finfo(np.float64).tiny
    node_features = np.log10(
        np.maximum(band_power, eps)
    ).astype(np.float32)

    centered = x - np.mean(x, axis=2, keepdims=True)
    norm = np.linalg.norm(centered, axis=2, keepdims=True)
    norm = np.maximum(norm, np.finfo(np.float64).eps)
    z = centered / norm

    corr = np.einsum(
        "wcs,wds->wcd",
        z,
        z,
        optimize=True,
    )
    corr = np.clip(corr, -1.0, 1.0)

    connectivity = corr[:, TRIU_I, TRIU_J].astype(np.float32)

    return (
        node_features,
        connectivity,
        qc_finite,
        qc_nonzero_variance,
        max_abs_uv,
        min_channel_std_uv,
    )

def extract_windows_from_file(filtered, starts_rel_sec):
    starts = np.rint(
        np.asarray(starts_rel_sec) * SFREQ
    ).astype(np.int64)

    offsets = np.arange(WINDOW_SAMPLES, dtype=np.int64)
    sample_idx = starts[:, None] + offsets[None, :]

    if sample_idx.max() >= filtered.shape[1]:
        raise RuntimeError(
            "Manifest window exceeds filtered EDF samples."
        )

    return np.transpose(
        filtered[:, sample_idx],
        (1, 0, 2),
    )


In [ ]:
# ============================================================
# Preprocess usable EDF files and build graph-feature cache
# ============================================================
# Duplicate channel labels are resolved deterministically.
# Windows are processed in batches to limit memory use.

import gc
import re
import time
from pathlib import Path

import mne
import numpy as np
import pandas as pd
from scipy.signal import sosfiltfilt, welch


# ------------------------------------------------------------
# Fixed batch size
# ------------------------------------------------------------

WINDOW_BATCH_SIZE = 128


# ------------------------------------------------------------
# Canonical-channel helper
# ------------------------------------------------------------

def canonical_channel_name_v2(name):

    x = str(name).strip().upper()

    # MNE suffixes duplicate EDF names:
    # T8-P8-0, T8-P8-1 -> T8-P8
    x = re.sub(
        r"-(\d+)$",
        "",
        x
    )

    return x


# ------------------------------------------------------------
# Fixed channel selector
# ------------------------------------------------------------

def select_fixed_channels(raw):

    mapping = {}

    for actual in raw.ch_names:

        canonical = (
            canonical_channel_name_v2(
                actual
            )
        )

        if canonical in FIXED_CHANNELS:

            mapping.setdefault(
                canonical,
                []
            ).append(actual)


    missing = [
        ch
        for ch in FIXED_CHANNELS
        if ch not in mapping
    ]


    if missing:

        raise RuntimeError(
            f"Missing required channels: "
            f"{missing}"
        )


    selected = []

    duplicate_records = []


    for canonical in FIXED_CHANNELS:

        candidates = mapping[
            canonical
        ]

        # Deterministic rule:
        # use first occurrence in EDF order.
        selected.append(
            candidates[0]
        )


        if len(candidates) > 1:

            duplicate_records.append(
                {
                    "canonical_channel":
                        canonical,

                    "candidates":
                        "|".join(
                            candidates
                        ),

                    "selected":
                        candidates[0],

                    "n_occurrences":
                        len(candidates)
                }
            )


    if len(selected) != 22:

        raise RuntimeError(
            f"Expected 22 channels, "
            f"obtained {len(selected)}"
        )


    return (
        selected,
        duplicate_records
    )


# ------------------------------------------------------------
# Feature extraction
# ------------------------------------------------------------

def compute_features_batch(x):

    """
    x:
        (n_windows, 22, 2560)

    returns:
        node_features:
            (n_windows, 22, 5)

        connectivity:
            (n_windows, 231)
    """

    x = np.asarray(
        x,
        dtype=np.float64
    )


    # --------------------------------------------------------
    # QC
    # --------------------------------------------------------

    qc_finite = np.isfinite(
        x
    ).all(
        axis=(1, 2)
    )


    channel_std = np.nanstd(
        x,
        axis=2
    )


    qc_nonzero_variance = np.all(
        channel_std > 0.0,
        axis=1
    )


    max_abs_uv = (
        np.nanmax(
            np.abs(x),
            axis=(1, 2)
        )
        * 1e6
    ).astype(
        np.float32
    )


    min_channel_std_uv = (
        np.nanmin(
            channel_std,
            axis=1
        )
        * 1e6
    ).astype(
        np.float32
    )


    # --------------------------------------------------------
    # Welch PSD
    # --------------------------------------------------------

    nperseg = int(
        2 * SFREQ
    )

    noverlap = (
        nperseg // 2
    )


    freqs, psd = welch(

        x,

        fs=SFREQ,

        window="hann",

        nperseg=nperseg,

        noverlap=noverlap,

        detrend="constant",

        scaling="density",

        axis=-1
    )


    band_power = []


    for band_name in BAND_NAMES:

        low, high = BANDS[
            band_name
        ]


        if (
            band_name
            != BAND_NAMES[-1]
        ):

            mask = (
                (freqs >= low)
                &
                (freqs < high)
            )

        else:

            mask = (
                (freqs >= low)
                &
                (freqs <= high)
            )


        if mask.sum() < 2:

            raise RuntimeError(
                f"Insufficient frequency "
                f"bins for {band_name}"
            )


        # np.trapezoid avoids deprecated np.trapz
        power = np.trapezoid(

            psd[..., mask],

            freqs[mask],

            axis=-1
        )


        band_power.append(
            power
        )


    band_power = np.stack(
        band_power,
        axis=-1
    )


    eps = np.finfo(
        np.float64
    ).tiny


    node_features = np.log10(

        np.maximum(
            band_power,
            eps
        )

    ).astype(
        np.float32
    )


    # --------------------------------------------------------
    # Signed Pearson connectivity
    # --------------------------------------------------------

    centered = (
        x
        -
        np.mean(
            x,
            axis=2,
            keepdims=True
        )
    )


    norm = np.linalg.norm(
        centered,
        axis=2,
        keepdims=True
    )


    norm = np.maximum(
        norm,
        np.finfo(
            np.float64
        ).eps
    )


    z = (
        centered
        / norm
    )


    corr = np.einsum(

        "wcs,wds->wcd",

        z,
        z,

        optimize=True
    )


    corr = np.clip(
        corr,
        -1.0,
        1.0
    )


    connectivity = corr[
        :,
        TRIU_I,
        TRIU_J
    ].astype(
        np.float32
    )


    return (
        node_features,
        connectivity,
        qc_finite,
        qc_nonzero_variance,
        max_abs_uv,
        min_channel_std_uv
    )


# ------------------------------------------------------------
# Process ONE EDF safely
# ------------------------------------------------------------

def process_edf_production(
    patient,
    fname
):

    edf_path = (
        DATA_ROOT
        / patient
        / fname
    )


    patient_cache = (
        CACHE_ROOT
        / patient
    )

    patient_cache.mkdir(
        parents=True,
        exist_ok=True
    )


    out_path = (
        patient_cache
        / f"{Path(fname).stem}.npz"
    )


    # --------------------------------------------------------
    # Resume / validate existing cache
    # --------------------------------------------------------

    if out_path.exists():

        try:

            with np.load(
                out_path,
                allow_pickle=False
            ) as z:

                required_keys = {
                    "preprocess_hash",
                    "node_features",
                    "connectivity",
                    "hard_qc_pass"
                }


                valid_keys = (
                    required_keys
                    .issubset(
                        set(z.files)
                    )
                )


                valid_hash = (
                    str(
                        z[
                            "preprocess_hash"
                        ].item()
                    )
                    == preprocess_hash
                )


            if (
                valid_keys
                and valid_hash
            ):

                return {
                    "patient":
                        patient,

                    "source_file":
                        fname,

                    "status":
                        "cached",

                    "n_windows":
                        np.nan,

                    "n_qc_fail":
                        np.nan,

                    "cache_path":
                        str(out_path)
                }


        except Exception:

            pass


        # Existing cache is incomplete/corrupt.
        out_path.unlink(
            missing_ok=True
        )


    # --------------------------------------------------------
    # Frozen manifest windows
    # --------------------------------------------------------

    fw = windows_v2[
        (
            windows_v2[
                "patient"
            ]
            == patient
        )
        &
        (
            windows_v2[
                "file"
            ]
            == fname
        )
    ].copy()


    fw = (
        fw
        .sort_values(
            "window_index_in_file"
        )
        .reset_index(
            drop=True
        )
    )


    n_windows = len(
        fw
    )


    if n_windows == 0:

        return {
            "patient":
                patient,

            "source_file":
                fname,

            "status":
                "no_windows",

            "n_windows":
                0,

            "n_qc_fail":
                0,

            "cache_path":
                ""
        }


    # --------------------------------------------------------
    # EDF header first
    # --------------------------------------------------------

    raw = mne.io.read_raw_edf(

        edf_path,

        preload=False,

        verbose="ERROR"
    )


    sfreq = float(
        raw.info[
            "sfreq"
        ]
    )


    if not np.isclose(
        sfreq,
        SFREQ,
        atol=1e-9,
        rtol=0.0
    ):

        raw.close()

        raise RuntimeError(
            f"{patient}/{fname}: "
            f"unexpected sampling rate "
            f"{sfreq}"
        )


    (
        selected_channels,
        duplicate_records
    ) = select_fixed_channels(
        raw
    )


    # Load only selected 22 channels.
    data = raw.get_data(
        picks=selected_channels
    ).astype(
        np.float64,
        copy=False
    )


    raw.close()
    del raw


    if data.shape[0] != 22:

        raise RuntimeError(
            f"{patient}/{fname}: "
            f"selected EEG shape "
            f"{data.shape}"
        )


    if not np.isfinite(
        data
    ).all():

        raise RuntimeError(
            f"{patient}/{fname}: "
            f"non-finite raw EEG detected"
        )


    # --------------------------------------------------------
    # Filter entire EDF independently
    # --------------------------------------------------------

    filtered = sosfiltfilt(

        SOS,

        data,

        axis=1
    )


    del data


    # --------------------------------------------------------
    # Preallocate compact outputs
    # --------------------------------------------------------

    all_node_features = np.empty(
        (
            n_windows,
            22,
            len(BAND_NAMES)
        ),
        dtype=np.float32
    )


    all_connectivity = np.empty(
        (
            n_windows,
            len(TRIU_I)
        ),
        dtype=np.float32
    )


    all_qc_finite = np.empty(
        n_windows,
        dtype=bool
    )


    all_qc_nonzero = np.empty(
        n_windows,
        dtype=bool
    )


    all_max_abs_uv = np.empty(
        n_windows,
        dtype=np.float32
    )


    all_min_std_uv = np.empty(
        n_windows,
        dtype=np.float32
    )


    # --------------------------------------------------------
    # Chunked window extraction
    # --------------------------------------------------------

    for start in range(
        0,
        n_windows,
        WINDOW_BATCH_SIZE
    ):

        stop = min(
            start
            + WINDOW_BATCH_SIZE,

            n_windows
        )


        batch_rows = fw.iloc[
            start:stop
        ]


        x = extract_windows_from_file(

            filtered,

            batch_rows[
                "start_rel_sec"
            ].to_numpy()
        )


        (
            node_features,
            connectivity,
            qc_finite,
            qc_nonzero_variance,
            max_abs_uv,
            min_channel_std_uv
        ) = compute_features_batch(
            x
        )


        all_node_features[
            start:stop
        ] = node_features


        all_connectivity[
            start:stop
        ] = connectivity


        all_qc_finite[
            start:stop
        ] = qc_finite


        all_qc_nonzero[
            start:stop
        ] = (
            qc_nonzero_variance
        )


        all_max_abs_uv[
            start:stop
        ] = max_abs_uv


        all_min_std_uv[
            start:stop
        ] = (
            min_channel_std_uv
        )


        del (
            x,
            node_features,
            connectivity
        )


    del filtered


    hard_qc_pass = (
        all_qc_finite
        &
        all_qc_nonzero
    )


    # --------------------------------------------------------
    # Save compact EDF cache
    # --------------------------------------------------------

    np.savez_compressed(

        str(out_path),

        preprocess_hash=np.array(
            preprocess_hash
        ),

        patient=np.array(
            patient
        ),

        source_file=np.array(
            fname
        ),

        selected_channels=np.asarray(
            selected_channels
        ),

        window_index_in_file=fw[
            "window_index_in_file"
        ].to_numpy(
            dtype=np.int32
        ),

        start_rel_sec=fw[
            "start_rel_sec"
        ].to_numpy(
            dtype=np.float32
        ),

        start_abs_sec=fw[
            "start_abs_sec"
        ].to_numpy(
            dtype=np.float64
        ),

        zone=fw[
            "zone"
        ].astype(
            str
        ).to_numpy(),

        event_id=fw[
            "event_id"
        ].astype(
            str
        ).to_numpy(),

        node_features=(
            all_node_features
        ),

        connectivity=(
            all_connectivity
        ),

        qc_finite=(
            all_qc_finite
        ),

        qc_nonzero_variance=(
            all_qc_nonzero
        ),

        hard_qc_pass=(
            hard_qc_pass
        ),

        max_abs_uv=(
            all_max_abs_uv
        ),

        min_channel_std_uv=(
            all_min_std_uv
        )
    )


    n_qc_fail = int(
        (
            ~hard_qc_pass
        ).sum()
    )


    result = {

        "patient":
            patient,

        "source_file":
            fname,

        "status":
            "processed",

        "n_windows":
            n_windows,

        "n_qc_fail":
            n_qc_fail,

        "p99_max_abs_uv":
            float(
                np.nanpercentile(
                    all_max_abs_uv,
                    99
                )
            ),

        "p01_min_channel_std_uv":
            float(
                np.nanpercentile(
                    all_min_std_uv,
                    1
                )
            ),

        "n_duplicate_labels":
            len(
                duplicate_records
            ),

        "cache_path":
            str(out_path)
    }


    del (
        all_node_features,
        all_connectivity,
        all_qc_finite,
        all_qc_nonzero,
        all_max_abs_uv,
        all_min_std_uv,
        hard_qc_pass
    )


    gc.collect()


    return result


# ============================================================
# BUILD THE 98-FILE LIST
# ============================================================

usable_files = files_v2[
    files_v2[
        "montage_compatible"
    ].astype(bool)
][
    [
        "patient",
        "file"
    ]
].copy()


usable_files = (
    usable_files
    .sort_values(
        [
            "patient",
            "file"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "Usable EDF files:",
    len(usable_files)
)


assert (
    len(usable_files)
    == 98
)


# ============================================================
# RUN
# ============================================================

results = []

start_time = time.time()


for i, row in usable_files.iterrows():

    patient = row[
        "patient"
    ]

    fname = row[
        "file"
    ]


    print(
        f"[{i+1:03d}/"
        f"{len(usable_files):03d}] "
        f"{patient}/{fname}"
    )


    result = process_edf_production(
        patient,
        fname
    )


    results.append(
        result
    )


    if result[
        "status"
    ] == "processed":

        print(
            "    "
            f"windows="
            f"{result['n_windows']:,} | "
            f"qc_fail="
            f"{result['n_qc_fail']} | "
            f"p99|uV|="
            f"{result['p99_max_abs_uv']:.1f}"
        )

    else:

        print(
            "    "
            f"{result['status']}"
        )


    # Save progress after EVERY EDF.
    pd.DataFrame(
        results
    ).to_csv(

        CACHE_ROOT
        / "batch_progress.csv",

        index=False
    )


runtime_min = (
    time.time()
    - start_time
) / 60.0


batch_results = pd.DataFrame(
    results
)


batch_results.to_csv(

    CACHE_ROOT
    / "cache_run_manifest.csv",

    index=False
)


print(
    "\n======================================"
)

print(
    "BATCH PREPROCESSING COMPLETE"
)

print(
    "======================================"
)

print(
    f"Runtime this session: "
    f"{runtime_min:.1f} min"
)


display(
    batch_results[
        "status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "status"
    )
    .reset_index(
        name="count"
    )
)


print(
    "\nQC failures by patient:"
)


display(
    batch_results
    .groupby(
        "patient",
        as_index=False
    )
    .agg(
        files=(
            "source_file",
            "count"
        ),

        windows=(
            "n_windows",
            "sum"
        ),

        hard_qc_failures=(
            "n_qc_fail",
            "sum"
        )
    )
)


## Full-cohort metadata screen

This intermediate screen creates the file and seizure-event manifests used by the final stream-level cohort screen. Its provisional patient-qualification fields are not used in the paper. The final inclusion rule is applied in the following cell.


In [ ]:
# ============================================================
# Full CHB-MIT metadata and provisional stream screen
# ============================================================
# This cell creates full-cohort file and event manifests.
# Final patient inclusion is determined by the next screening cell.

from pathlib import Path
import re
import json
import hashlib

import mne
import numpy as np
import pandas as pd


# ============================================================
# 1. PATHS
# ============================================================

DATA_ROOT = Path(
    "/content/drive/MyDrive/"
    "chb-mit-scalp-eeg-database-1.0.0"
)

SCREEN_ROOT = Path(
    "/content/drive/MyDrive/EEG_Research/"
    "continual_graph_forecasting/cohort_screen_v1"
)

SCREEN_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. FROZEN SETTINGS
# ============================================================

PATIENTS = [
    f"chb{i:02d}"
    for i in range(1, 25)
]

FIXED_CHANNELS = [
    "C3-P3",
    "C4-P4",
    "CZ-PZ",
    "F3-C3",
    "F4-C4",
    "F7-T7",
    "F8-T8",
    "FP1-F3",
    "FP1-F7",
    "FP2-F4",
    "FP2-F8",
    "FT10-T8",
    "FT9-FT10",
    "FZ-CZ",
    "P3-O1",
    "P4-O2",
    "P7-O1",
    "P7-T7",
    "P8-O2",
    "T7-FT9",
    "T7-P7",
    "T8-P8",
]

WINDOW_SEC = 10.0
STRIDE_SEC = 5.0

SOP_SEC = 30 * 60
SPH_SEC = 5 * 60

# preictal = onset - 35 min to onset - 5 min
PREICTAL_BEFORE_START = SOP_SEC + SPH_SEC
PREICTAL_BEFORE_END = SPH_SEC

CLEAN_BUFFER_SEC = 4 * 60 * 60

MIN_PREICTAL_COVERAGE = 0.95

MIN_WARMUP_EVENTS = 2

# Provisional screen parameter.
MIN_FUTURE_EVENTS = 2

# Provisional criterion; not used for final paper inclusion.
MIN_VALID_FUTURE_EPISODES = 2

# Same retention requirements used in latest split.
RETENTION_POS_PER_EVENT = 30
RETENTION_NEG_TOTAL = 60

# After reserving retention negatives, require useful
# causal clean-interictal training data.
MIN_INITIAL_NEGATIVE_WINDOWS = 120

SEED = 2026


SCREEN_CONFIG = {
    "patients": PATIENTS,
    "fixed_channels": FIXED_CHANNELS,
    "window_sec": WINDOW_SEC,
    "stride_sec": STRIDE_SEC,
    "sop_sec": SOP_SEC,
    "sph_sec": SPH_SEC,
    "clean_interictal_buffer_sec": CLEAN_BUFFER_SEC,
    "min_preictal_coverage": MIN_PREICTAL_COVERAGE,
    "minimum_warmup_events": MIN_WARMUP_EVENTS,
    "minimum_future_events": MIN_FUTURE_EVENTS,
    "minimum_valid_future_episodes":
        MIN_VALID_FUTURE_EPISODES,
    "retention_positive_windows_per_event":
        RETENTION_POS_PER_EVENT,
    "retention_negative_windows":
        RETENTION_NEG_TOTAL,
    "minimum_initial_negative_windows":
        MIN_INITIAL_NEGATIVE_WINDOWS,
    "seed": SEED,
}


# ============================================================
# 3. HELPERS
# ============================================================

def natural_file_key(name):

    match = re.search(
        r"_(\d+)\.edf$",
        str(name),
        flags=re.I
    )

    if match:
        return int(
            match.group(1)
        )

    return 10**9


def canonical_channel_name(name):

    name = (
        str(name)
        .strip()
        .upper()
    )

    # MNE duplicate labels:
    # T8-P8-0 / T8-P8-1 -> T8-P8
    name = re.sub(
        r"-(\d+)$",
        "",
        name
    )

    return name


def montage_compatible(ch_names):

    canonical = {
        canonical_channel_name(ch)
        for ch in ch_names
    }

    return all(
        ch in canonical
        for ch in FIXED_CHANNELS
    )


def hms_to_seconds(text):

    match = re.search(
        r"(\d{1,2}):"
        r"(\d{2}):"
        r"(\d{2})",
        str(text)
    )

    if not match:
        return None

    h, m, s = map(
        int,
        match.groups()
    )

    return (
        h * 3600
        + m * 60
        + s
    )


def parse_patient_summary(path):

    text = path.read_text(
        errors="ignore"
    )

    file_matches = list(
        re.finditer(
            r"(?im)^File Name:\s*(\S+)\s*$",
            text
        )
    )

    rows = []
    seizure_rows = []


    for i, match in enumerate(
        file_matches
    ):

        block_start = match.start()

        if i + 1 < len(file_matches):
            block_end = (
                file_matches[
                    i + 1
                ].start()
            )
        else:
            block_end = len(text)

        block = text[
            block_start:block_end
        ]

        fname = (
            match.group(1)
            .strip()
        )


        start_match = re.search(
            r"(?im)^File Start Time:\s*"
            r"([^\r\n]+)",
            block
        )

        start_tod = None

        if start_match:
            start_tod = hms_to_seconds(
                start_match.group(1)
            )


        seizure_starts = re.findall(
            r"(?im)^Seizure"
            r"(?:\s+\d+)?"
            r"\s+Start Time:\s*"
            r"(\d+)\s*seconds?",
            block
        )

        seizure_ends = re.findall(
            r"(?im)^Seizure"
            r"(?:\s+\d+)?"
            r"\s+End Time:\s*"
            r"(\d+)\s*seconds?",
            block
        )


        if (
            len(seizure_starts)
            != len(seizure_ends)
        ):

            raise RuntimeError(
                f"{path.name}/{fname}: "
                "seizure start/end count mismatch"
            )


        rows.append({
            "file": fname,
            "summary_start_tod":
                start_tod,
        })


        for seizure_number, (
            start_rel,
            end_rel
        ) in enumerate(
            zip(
                seizure_starts,
                seizure_ends
            ),
            start=1
        ):

            seizure_rows.append({
                "file":
                    fname,

                "seizure_in_file":
                    seizure_number,

                "start_rel_sec":
                    float(
                        start_rel
                    ),

                "end_rel_sec":
                    float(
                        end_rel
                    ),
            })


    return (
        pd.DataFrame(rows),
        pd.DataFrame(
            seizure_rows
        )
    )


def merge_intervals(intervals):

    if not intervals:
        return []

    intervals = sorted(
        (
            float(s),
            float(e)
        )
        for s, e in intervals
    )

    merged = [
        list(
            intervals[0]
        )
    ]


    for start, end in (
        intervals[1:]
    ):

        if (
            start
            <= merged[-1][1]
            + 1e-6
        ):

            merged[-1][1] = max(
                merged[-1][1],
                end
            )

        else:

            merged.append(
                [
                    start,
                    end
                ]
            )


    return [
        (
            start,
            end
        )
        for start, end
        in merged
    ]


def interval_coverage(
    target_start,
    target_end,
    recording_intervals
):

    coverage = 0.0

    for rec_start, rec_end in (
        recording_intervals
    ):

        overlap = max(
            0.0,
            min(
                target_end,
                rec_end
            )
            -
            max(
                target_start,
                rec_start
            )
        )

        coverage += overlap


    duration = (
        target_end
        - target_start
    )

    if duration <= 0:
        return 0.0

    return min(
        1.0,
        coverage / duration
    )


def select_nonoverlap_positions(
    starts,
    ends,
    n,
    seed
):

    starts = np.asarray(
        starts,
        dtype=float
    )

    ends = np.asarray(
        ends,
        dtype=float
    )

    if len(starts) < n:
        return None

    rng = np.random.default_rng(
        seed
    )

    order = rng.permutation(
        len(starts)
    )

    selected = []

    selected_intervals = []


    for idx in order:

        start = starts[idx]
        end = ends[idx]

        overlap = any(
            start < old_end
            and
            end > old_start

            for old_start, old_end
            in selected_intervals
        )

        if overlap:
            continue

        selected.append(idx)

        selected_intervals.append(
            (
                start,
                end
            )
        )

        if len(selected) == n:
            return np.array(
                selected,
                dtype=int
            )


    return None


def count_after_excluding_intervals(
    starts,
    ends,
    intervals
):

    keep = np.ones(
        len(starts),
        dtype=bool
    )

    for start, end in intervals:

        keep &= ~(
            (starts < end)
            &
            (ends > start)
        )

    return int(
        keep.sum()
    )


# ============================================================
# 4. SCREEN EACH PATIENT
# ============================================================

patient_results = []

event_results = []

future_results = []

file_results = []


for patient in PATIENTS:

    print(
        "\n"
        + "=" * 80
    )

    print(
        "SCREENING:",
        patient
    )

    print(
        "=" * 80
    )


    patient_dir = (
        DATA_ROOT
        / patient
    )


    if not patient_dir.exists():

        print(
            "  MISSING PATIENT DIRECTORY"
        )

        patient_results.append({
            "patient":
                patient,

            "qualifies":
                False,

            "reason":
                "patient_directory_missing"
        })

        continue


    summary_path = (
        patient_dir
        / f"{patient}-summary.txt"
    )


    if not summary_path.exists():

        print(
            "  MISSING SUMMARY FILE"
        )

        patient_results.append({
            "patient":
                patient,

            "qualifies":
                False,

            "reason":
                "summary_file_missing"
        })

        continue


    # --------------------------------------------------------
    # Parse summary
    # --------------------------------------------------------

    summary_files, seizures_rel = (
        parse_patient_summary(
            summary_path
        )
    )


    summary_start_map = dict(
        zip(
            summary_files["file"],
            summary_files[
                "summary_start_tod"
            ]
        )
    )


    # --------------------------------------------------------
    # Read EDF HEADERS only
    # --------------------------------------------------------

    edf_paths = sorted(
        patient_dir.glob(
            "*.edf"
        ),
        key=lambda p:
            natural_file_key(
                p.name
            )
    )


    header_rows = []


    for edf_path in edf_paths:

        raw = mne.io.read_raw_edf(
            edf_path,
            preload=False,
            verbose="ERROR"
        )


        sfreq = float(
            raw.info[
                "sfreq"
            ]
        )

        duration = float(
            raw.n_times
            / sfreq
        )


        meas_date = raw.info.get(
            "meas_date",
            None
        )


        if meas_date is None:
            header_timestamp = None
        else:
            header_timestamp = float(
                meas_date.timestamp()
            )


        compatible = montage_compatible(
            raw.ch_names
        )


        header_rows.append({

            "patient":
                patient,

            "file":
                edf_path.name,

            "sfreq":
                sfreq,

            "duration_sec":
                duration,

            "header_timestamp":
                header_timestamp,

            "summary_start_tod":
                summary_start_map.get(
                    edf_path.name,
                    None
                ),

            "montage_compatible":
                compatible,

            "n_channels":
                len(
                    raw.ch_names
                ),
        })


        raw.close()


    files_df = pd.DataFrame(
        header_rows
    )


    if files_df.empty:

        patient_results.append({
            "patient":
                patient,

            "qualifies":
                False,

            "reason":
                "no_edf_files"
        })

        continue


    # --------------------------------------------------------
    # Construct chronological file starts
    #
    # Prefer EDF absolute header dates.
    # Fall back to summary clock + rollover.
    # --------------------------------------------------------

    if files_df[
        "header_timestamp"
    ].notna().all():

        base = float(
            files_df[
                "header_timestamp"
            ].min()
        )

        files_df[
            "start_abs_sec"
        ] = (
            files_df[
                "header_timestamp"
            ]
            - base
        )


    else:

        ordered = (
            files_df
            .sort_values(
                "file",
                key=lambda s:
                    s.map(
                        natural_file_key
                    )
            )
            .copy()
        )


        starts = []

        day_offset = 0.0
        previous_tod = None


        for _, row in ordered.iterrows():

            tod = row[
                "summary_start_tod"
            ]


            if pd.isna(tod):

                raise RuntimeError(
                    f"{patient}/{row['file']}: "
                    "no EDF date and no summary "
                    "start time"
                )


            tod = float(
                tod
            )


            if (
                previous_tod is not None
                and
                tod < previous_tod
            ):

                day_offset += 86400.0


            starts.append(
                day_offset
                + tod
            )

            previous_tod = tod


        first = min(
            starts
        )

        ordered[
            "start_abs_sec"
        ] = (
            np.asarray(
                starts
            )
            - first
        )


        files_df = (
            ordered
            .sort_index()
        )


    files_df[
        "end_abs_sec"
    ] = (
        files_df[
            "start_abs_sec"
        ]
        +
        files_df[
            "duration_sec"
        ]
    )


    file_results.extend(
        files_df.to_dict(
            "records"
        )
    )


    start_map = dict(
        zip(
            files_df[
                "file"
            ],
            files_df[
                "start_abs_sec"
            ]
        )
    )


    # --------------------------------------------------------
    # Build absolute seizure events
    # --------------------------------------------------------

    seizure_rows = []


    for _, seizure in (
        seizures_rel.iterrows()
    ):

        fname = seizure[
            "file"
        ]


        if fname not in start_map:
            continue


        onset = (
            float(
                start_map[
                    fname
                ]
            )
            +
            float(
                seizure[
                    "start_rel_sec"
                ]
            )
        )


        offset = (
            float(
                start_map[
                    fname
                ]
            )
            +
            float(
                seizure[
                    "end_rel_sec"
                ]
            )
        )


        seizure_rows.append({
            "patient":
                patient,

            "file":
                fname,

            "onset_abs_sec":
                onset,

            "offset_abs_sec":
                offset,
        })


    all_events = (
        pd.DataFrame(
            seizure_rows
        )
        .sort_values(
            "onset_abs_sec"
        )
        .reset_index(
            drop=True
        )
    )


    if all_events.empty:

        patient_results.append({

            "patient":
                patient,

            "n_edf":
                len(
                    files_df
                ),

            "compatible_edf":
                int(
                    files_df[
                        "montage_compatible"
                    ].sum()
                ),

            "total_seizures":
                0,

            "eligible_events":
                0,

            "qualifies":
                False,

            "reason":
                "no_seizures"
        })

        print(
            "  No seizures"
        )

        continue


    all_events[
        "event_id"
    ] = [
        f"{patient}_E{i:02d}"
        for i in range(
            1,
            len(all_events)
            + 1
        )
    ]


    all_events[
        "preictal_start_abs_sec"
    ] = (
        all_events[
            "onset_abs_sec"
        ]
        -
        PREICTAL_BEFORE_START
    )


    all_events[
        "preictal_end_abs_sec"
    ] = (
        all_events[
            "onset_abs_sec"
        ]
        -
        PREICTAL_BEFORE_END
    )


    # --------------------------------------------------------
    # Compatible recording intervals
    # --------------------------------------------------------

    compatible_files = files_df[
        files_df[
            "montage_compatible"
        ]
    ].copy()


    compatible_intervals = (
        merge_intervals(
            list(
                zip(
                    compatible_files[
                        "start_abs_sec"
                    ],
                    compatible_files[
                        "end_abs_sec"
                    ]
                )
            )
        )
    )


    # --------------------------------------------------------
    # Event eligibility
    # --------------------------------------------------------

    eligible_flags = []

    coverage_values = []

    previous_overlap_flags = []


    for event_idx, event in (
        all_events.iterrows()
    ):

        pre_start = float(
            event[
                "preictal_start_abs_sec"
            ]
        )

        pre_end = float(
            event[
                "preictal_end_abs_sec"
            ]
        )


        coverage = interval_coverage(
            pre_start,
            pre_end,
            compatible_intervals
        )


        previous_overlap = False


        if event_idx > 0:

            previous_offset = float(
                all_events.iloc[
                    event_idx - 1
                ][
                    "offset_abs_sec"
                ]
            )

            previous_overlap = (
                previous_offset
                > pre_start
            )


        eligible = (
            coverage
            >= MIN_PREICTAL_COVERAGE
            and
            not previous_overlap
        )


        coverage_values.append(
            coverage
        )

        previous_overlap_flags.append(
            previous_overlap
        )

        eligible_flags.append(
            eligible
        )


    all_events[
        "preictal_coverage"
    ] = coverage_values

    all_events[
        "previous_seizure_overlap"
    ] = previous_overlap_flags

    all_events[
        "eligible"
    ] = eligible_flags


    eligible_events = (
        all_events[
            all_events[
                "eligible"
            ]
        ]
        .sort_values(
            "onset_abs_sec"
        )
        .reset_index(
            drop=True
        )
    )


    event_results.extend(
        all_events.to_dict(
            "records"
        )
    )


    # --------------------------------------------------------
    # Generate ALL windows from compatible EDFs
    # --------------------------------------------------------

    starts_list = []
    ends_list = []


    for _, file_row in (
        compatible_files.iterrows()
    ):

        duration = float(
            file_row[
                "duration_sec"
            ]
        )


        n_windows = int(
            np.floor(
                (
                    duration
                    - WINDOW_SEC
                )
                /
                STRIDE_SEC
            )
            + 1
        )


        if n_windows <= 0:
            continue


        starts = (
            float(
                file_row[
                    "start_abs_sec"
                ]
            )
            +
            np.arange(
                n_windows,
                dtype=float
            )
            * STRIDE_SEC
        )


        starts_list.append(
            starts
        )

        ends_list.append(
            starts
            + WINDOW_SEC
        )


    if len(starts_list) == 0:

        patient_results.append({

            "patient":
                patient,

            "n_edf":
                len(
                    files_df
                ),

            "compatible_edf":
                0,

            "total_seizures":
                len(
                    all_events
                ),

            "eligible_events":
                len(
                    eligible_events
                ),

            "qualifies":
                False,

            "reason":
                "no_compatible_windows"
        })

        continue


    window_starts = np.concatenate(
        starts_list
    )

    window_ends = np.concatenate(
        ends_list
    )


    # --------------------------------------------------------
    # Clean-interictal mask
    #
    # A complete 10 s window must NOT overlap the
    # +/- 4 h exclusion region around ANY seizure.
    # --------------------------------------------------------

    clean_mask = np.ones(
        len(
            window_starts
        ),
        dtype=bool
    )


    for _, event in (
        all_events.iterrows()
    ):

        exclusion_start = (
            float(
                event[
                    "onset_abs_sec"
                ]
            )
            -
            CLEAN_BUFFER_SEC
        )


        exclusion_end = (
            float(
                event[
                    "offset_abs_sec"
                ]
            )
            +
            CLEAN_BUFFER_SEC
        )


        overlap = (
            (
                window_starts
                < exclusion_end
            )
            &
            (
                window_ends
                > exclusion_start
            )
        )


        clean_mask &= ~overlap


    # --------------------------------------------------------
    # Basic eligibility checks
    # --------------------------------------------------------

    patient_number = int(
        patient[
            3:
        ]
    )


    reason = None
    qualifies = False

    chosen_k = None

    chosen_clean_total = 0

    chosen_clean_after_retention = 0

    chosen_future_count = 0

    chosen_valid_future = 0

    chosen_zero_negative_future = 0


    if len(
        eligible_events
    ) < (
        MIN_WARMUP_EVENTS
        +
        MIN_FUTURE_EVENTS
    ):

        reason = (
            "insufficient_eligible_seizures"
        )


    else:

        # ----------------------------------------------------
        # Retention-positive feasibility from first
        # TWO eligible seizures.
        # ----------------------------------------------------

        retention_positive_ok = True


        for retention_event_idx in (
            range(2)
        ):

            event = (
                eligible_events.iloc[
                    retention_event_idx
                ]
            )


            positive_mask = (
                (
                    window_starts
                    >= float(
                        event[
                            "preictal_start_abs_sec"
                        ]
                    )
                )
                &
                (
                    window_ends
                    <= float(
                        event[
                            "preictal_end_abs_sec"
                        ]
                    )
                )
            )


            pos_idx = np.flatnonzero(
                positive_mask
            )


            selection = (
                select_nonoverlap_positions(

                    window_starts[
                        pos_idx
                    ],

                    window_ends[
                        pos_idx
                    ],

                    RETENTION_POS_PER_EVENT,

                    seed=(
                        SEED
                        +
                        patient_number
                        * 10000
                        +
                        100
                        +
                        retention_event_idx
                    )
                )
            )


            if selection is None:

                retention_positive_ok = (
                    False
                )

                break


        if not retention_positive_ok:

            reason = (
                "insufficient_retention_"
                "positive_windows"
            )


        else:

            # =================================================
            # Search earliest acceptable warm-up.
            # Must preserve >=2 future events.
            # =================================================

            max_warmup_k = (
                len(
                    eligible_events
                )
                -
                MIN_FUTURE_EVENTS
            )


            for k in range(
                MIN_WARMUP_EVENTS,
                max_warmup_k + 1
            ):

                warmup_last = (
                    eligible_events.iloc[
                        k - 1
                    ]
                )


                cutoff = float(
                    warmup_last[
                        "offset_abs_sec"
                    ]
                )


                causal_clean_mask = (
                    clean_mask
                    &
                    (
                        window_ends
                        <= cutoff
                    )
                )


                clean_indices = (
                    np.flatnonzero(
                        causal_clean_mask
                    )
                )


                if len(
                    clean_indices
                ) == 0:
                    continue


                # ---------------------------------------------
                # Can we reserve 60 non-overlapping
                # retention-negative windows?
                # ---------------------------------------------

                chosen_positions = (
                    select_nonoverlap_positions(

                        window_starts[
                            clean_indices
                        ],

                        window_ends[
                            clean_indices
                        ],

                        RETENTION_NEG_TOTAL,

                        seed=(
                            SEED
                            +
                            patient_number
                            * 10000
                            +
                            k
                        )
                    )
                )


                if chosen_positions is None:
                    continue


                selected_global = (
                    clean_indices[
                        chosen_positions
                    ]
                )


                retention_intervals = list(
                    zip(
                        window_starts[
                            selected_global
                        ],
                        window_ends[
                            selected_global
                        ]
                    )
                )


                clean_after_retention = (
                    count_after_excluding_intervals(

                        window_starts[
                            clean_indices
                        ],

                        window_ends[
                            clean_indices
                        ],

                        retention_intervals
                    )
                )


                if (
                    clean_after_retention
                    <
                    MIN_INITIAL_NEGATIVE_WINDOWS
                ):

                    continue


                # ---------------------------------------------
                # Examine future episodes
                # ---------------------------------------------

                future_events = (
                    eligible_events.iloc[
                        k:
                    ]
                    .reset_index(
                        drop=True
                    )
                )


                previous_event = (
                    eligible_events.iloc[
                        k - 1
                    ]
                )


                valid_future = 0
                zero_negative = 0


                temp_future_rows = []


                for future_idx, (
                    _,
                    current_event
                ) in enumerate(
                    future_events.iterrows(),
                    start=1
                ):

                    block_start = float(
                        previous_event[
                            "offset_abs_sec"
                        ]
                    )


                    block_end = float(
                        current_event[
                            "preictal_end_abs_sec"
                        ]
                    )


                    in_block = (
                        (
                            window_starts
                            >= block_start
                        )
                        &
                        (
                            window_ends
                            <= block_end
                        )
                    )


                    current_preictal = (
                        (
                            window_starts
                            >= float(
                                current_event[
                                    "preictal_start_abs_sec"
                                ]
                            )
                        )
                        &
                        (
                            window_ends
                            <= float(
                                current_event[
                                    "preictal_end_abs_sec"
                                ]
                            )
                        )
                    )


                    n_positive = int(
                        current_preictal.sum()
                    )


                    n_negative = int(
                        (
                            in_block
                            &
                            clean_mask
                        ).sum()
                    )


                    both_classes = (
                        n_positive > 0
                        and
                        n_negative > 0
                    )


                    if both_classes:
                        valid_future += 1


                    if n_negative == 0:
                        zero_negative += 1


                    temp_future_rows.append({

                        "patient":
                            patient,

                        "candidate_warmup_events":
                            k,

                        "episode_index":
                            future_idx,

                        "event_id":
                            current_event[
                                "event_id"
                            ],

                        "n_preictal_windows":
                            n_positive,

                        "n_clean_interictal_windows":
                            n_negative,

                        "both_classes":
                            both_classes,
                    })


                    previous_event = (
                        current_event
                    )


                # ---------------------------------------------
                # FINAL QUALIFICATION CONDITION
                # ---------------------------------------------

                if (
                    len(
                        future_events
                    )
                    >=
                    MIN_FUTURE_EVENTS

                    and

                    valid_future
                    >=
                    MIN_VALID_FUTURE_EPISODES
                ):

                    chosen_k = k

                    chosen_clean_total = (
                        len(
                            clean_indices
                        )
                    )

                    chosen_clean_after_retention = (
                        clean_after_retention
                    )

                    chosen_future_count = (
                        len(
                            future_events
                        )
                    )

                    chosen_valid_future = (
                        valid_future
                    )

                    chosen_zero_negative_future = (
                        zero_negative
                    )

                    future_results.extend(
                        temp_future_rows
                    )

                    qualifies = True

                    reason = "qualified"

                    break


            if chosen_k is None:

                if reason is None:

                    reason = (
                        "no_warmup_with_"
                        "sufficient_training_"
                        "and_future_both_class_"
                        "episodes"
                    )


    # --------------------------------------------------------
    # Save patient screen result
    # --------------------------------------------------------

    result = {

        "patient":
            patient,

        "n_edf":
            len(
                files_df
            ),

        "compatible_edf":
            int(
                files_df[
                    "montage_compatible"
                ].sum()
            ),

        "total_seizures":
            len(
                all_events
            ),

        "eligible_events":
            len(
                eligible_events
            ),

        "warmup_events":
            chosen_k,

        "future_events":
            chosen_future_count,

        "future_both_class_episodes":
            chosen_valid_future,

        "future_zero_negative_episodes":
            chosen_zero_negative_future,

        "causal_clean_windows_at_warmup":
            chosen_clean_total,

        "initial_clean_after_retention":
            chosen_clean_after_retention,

        "qualifies":
            qualifies,

        "reason":
            reason,
    }


    patient_results.append(
        result
    )


    print(
        "  EDFs:",
        result[
            "n_edf"
        ],

        "| compatible:",
        result[
            "compatible_edf"
        ]
    )


    print(
        "  seizures:",
        result[
            "total_seizures"
        ],

        "| eligible:",
        result[
            "eligible_events"
        ]
    )


    print(
        "  QUALIFIES:",
        qualifies
    )


    if qualifies:

        print(
            "  warm-up:",
            chosen_k,

            "| future:",
            chosen_future_count,

            "| future both-class:",
            chosen_valid_future
        )

    else:

        print(
            "  reason:",
            reason
        )


# ============================================================
# 5. RESULTS
# ============================================================

screen_df = pd.DataFrame(
    patient_results
)

event_df = pd.DataFrame(
    event_results
)

future_df = pd.DataFrame(
    future_results
)

file_df = pd.DataFrame(
    file_results
)


# ============================================================
# 6. FREEZE SCREEN HASH
# ============================================================

hash_cols = [
    "patient",
    "compatible_edf",
    "total_seizures",
    "eligible_events",
    "warmup_events",
    "future_events",
    "future_both_class_episodes",
    "qualifies",
    "reason",
]


screen_hash = hashlib.sha256(
    screen_df[
        hash_cols
    ]
    .fillna(
        ""
    )
    .to_csv(
        index=False
    )
    .encode(
        "utf-8"
    )
).hexdigest()


# ============================================================
# 7. SAVE
# ============================================================

screen_df.to_csv(
    SCREEN_ROOT
    / "patient_screen.csv",
    index=False
)

event_df.to_csv(
    SCREEN_ROOT
    / "all_events_screen.csv",
    index=False
)

future_df.to_csv(
    SCREEN_ROOT
    / "qualified_future_episodes.csv",
    index=False
)

file_df.to_csv(
    SCREEN_ROOT
    / "file_montage_screen.csv",
    index=False
)


(
    SCREEN_ROOT
    / "screen_config.json"
).write_text(

    json.dumps(
        {
            "screen_hash":
                screen_hash,

            "config":
                SCREEN_CONFIG,
        },
        indent=2
    )
)


# ============================================================
# 8. FINAL REPORT
# ============================================================

qualified = (
    screen_df[
        screen_df[
            "qualifies"
        ]
    ]
    .copy()
)


print(
    "\n"
    + "=" * 90
)

print(
    "FULL CHB-MIT COHORT SCREEN COMPLETE"
)

print(
    "=" * 90
)


print(
    "Screen hash:",
    screen_hash
)


print(
    "\nQUALIFIED PATIENTS"
)

display(
    qualified[
        [
            "patient",
            "compatible_edf",
            "total_seizures",
            "eligible_events",
            "warmup_events",
            "future_events",
            "future_both_class_episodes",
            "future_zero_negative_episodes",
            "causal_clean_windows_at_warmup",
            "initial_clean_after_retention",
        ]
    ]
)


print(
    "\nALL PATIENTS"
)

display(
    screen_df[
        [
            "patient",
            "compatible_edf",
            "total_seizures",
            "eligible_events",
            "warmup_events",
            "future_events",
            "future_both_class_episodes",
            "qualifies",
            "reason",
        ]
    ]
)


print(
    "\nNUMBER QUALIFIED:",
    int(
        screen_df[
            "qualifies"
        ].sum()
    )
)


print(
    "\nQUALIFIED PATIENT IDS:"
)

print(
    qualified[
        "patient"
    ].tolist()
)


if not future_df.empty:

    print(
        "\nFUTURE EPISODES FOR QUALIFIED PATIENTS"
    )

    display(
        future_df
    )


print(
    "\nSaved to:",
    SCREEN_ROOT
)


In [ ]:
# ============================================================
# Final stream-level cohort screen used in the paper
# ============================================================
# Uses the fixed 4-hour clean-interictal definition, 30-min SOP,
# and 5-min SPH. Individual future seizure episodes may be
# preictal-only; clean exposure is evaluated across the full
# chronological future stream.

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd


# ============================================================
# 1. PATHS
# ============================================================

OLD_SCREEN_ROOT = Path(
    "/content/drive/MyDrive/EEG_Research/"
    "continual_graph_forecasting/cohort_screen_v1"
)

NEW_SCREEN_ROOT = Path(
    "/content/drive/MyDrive/EEG_Research/"
    "continual_graph_forecasting/cohort_screen_v2"
)

NEW_SCREEN_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. LOAD PREVIOUS OBJECTIVE SCREEN DATA
# ============================================================

files = pd.read_csv(
    OLD_SCREEN_ROOT
    / "file_montage_screen.csv"
)

events = pd.read_csv(
    OLD_SCREEN_ROOT
    / "all_events_screen.csv"
)


def to_bool(series):

    if series.dtype == bool:
        return series

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .eq("true")
    )


files[
    "montage_compatible"
] = to_bool(
    files[
        "montage_compatible"
    ]
)


events[
    "eligible"
] = to_bool(
    events[
        "eligible"
    ]
)


# ============================================================
# 3. FROZEN PARAMETERS
# ============================================================

WINDOW_SEC = 10.0
STRIDE_SEC = 5.0

CLEAN_BUFFER_SEC = (
    4 * 60 * 60
)

MIN_WARMUP_EVENTS = 2

# Require at least two genuinely future seizure events.
MIN_FUTURE_EVENTS = 2

# Minimum clean training data after retention selection.
RETENTION_NEG_TOTAL = 60

MIN_INITIAL_NEGATIVE_WINDOWS = 120

SEED = 2026


SCREEN_CONFIG = {

    "source_screen":
        "cohort_screen_v1",

    "window_sec":
        WINDOW_SEC,

    "stride_sec":
        STRIDE_SEC,

    "clean_interictal_buffer_sec":
        CLEAN_BUFFER_SEC,

    "minimum_warmup_events":
        MIN_WARMUP_EVENTS,

    "minimum_future_seizures":
        MIN_FUTURE_EVENTS,

    "retention_negative_windows":
        RETENTION_NEG_TOTAL,

    "minimum_initial_negative_windows_after_retention":
        MIN_INITIAL_NEGATIVE_WINDOWS,

    "future_inclusion_rule":
        (
            "at least two future seizures and at least "
            "one clean-interictal window somewhere in "
            "the complete future stream"
        ),

    "future_episode_rule":
        (
            "individual future seizure episodes may be "
            "preictal-only; both classes are not required "
            "inside every episode"
        ),

    "primary_future_metrics":
        (
            "seizure-level sensitivity across future events; "
            "false alarms per hour across pooled future "
            "clean-interictal exposure; pooled future AUPRC"
        ),

    "seed":
        SEED
}


print(
    json.dumps(
        SCREEN_CONFIG,
        indent=2
    )
)


# ============================================================
# 4. HELPERS
# ============================================================

def select_nonoverlap_positions(
    starts,
    ends,
    n,
    seed
):

    starts = np.asarray(
        starts,
        dtype=float
    )

    ends = np.asarray(
        ends,
        dtype=float
    )


    if len(starts) < n:

        return None


    rng = np.random.default_rng(
        seed
    )

    order = rng.permutation(
        len(starts)
    )


    selected = []

    intervals = []


    for idx in order:

        start = float(
            starts[idx]
        )

        end = float(
            ends[idx]
        )


        overlap = any(

            start < old_end
            and
            end > old_start

            for old_start, old_end
            in intervals
        )


        if overlap:

            continue


        selected.append(
            idx
        )

        intervals.append(
            (
                start,
                end
            )
        )


        if len(
            selected
        ) == n:

            return np.asarray(
                selected,
                dtype=int
            )


    return None


def count_after_excluding_intervals(
    starts,
    ends,
    intervals
):

    starts = np.asarray(
        starts,
        dtype=float
    )

    ends = np.asarray(
        ends,
        dtype=float
    )


    keep = np.ones(
        len(starts),
        dtype=bool
    )


    for start, end in intervals:

        keep &= ~(
            (starts < end)
            &
            (ends > start)
        )


    return int(
        keep.sum()
    )


# ============================================================
# 5. SCREEN
# ============================================================

patient_rows = []

future_rows = []


patients = sorted(
    events[
        "patient"
    ].unique()
)


for patient_number, patient in enumerate(
    patients,
    start=1
):

    print(
        "\n"
        + "=" * 80
    )

    print(
        "SCREENING:",
        patient
    )

    print(
        "=" * 80
    )


    pf = (
        files[
            (
                files[
                    "patient"
                ]
                == patient
            )
            &
            (
                files[
                    "montage_compatible"
                ]
            )
        ]
        .sort_values(
            "start_abs_sec"
        )
        .reset_index(
            drop=True
        )
    )


    all_events = (
        events[
            events[
                "patient"
            ]
            == patient
        ]
        .sort_values(
            "onset_abs_sec"
        )
        .reset_index(
            drop=True
        )
    )


    eligible_events = (
        all_events[
            all_events[
                "eligible"
            ]
        ]
        .sort_values(
            "onset_abs_sec"
        )
        .reset_index(
            drop=True
        )
    )


    # --------------------------------------------------------
    # Basic seizure-count requirement
    # --------------------------------------------------------

    if len(
        eligible_events
    ) < (
        MIN_WARMUP_EVENTS
        +
        MIN_FUTURE_EVENTS
    ):

        patient_rows.append({

            "patient":
                patient,

            "eligible_events":
                len(
                    eligible_events
                ),

            "warmup_events":
                np.nan,

            "future_events":
                0,

            "future_events_with_clean":
                0,

            "future_clean_windows":
                0,

            "future_clean_exposure_hours":
                0.0,

            "qualifies":
                False,

            "reason":
                "insufficient_eligible_seizures"
        })


        print(
            "QUALIFIES: False"
        )

        print(
            "Reason: insufficient eligible seizures"
        )

        continue


    # --------------------------------------------------------
    # Generate windows from compatible EDFs
    # --------------------------------------------------------

    starts_parts = []

    ends_parts = []


    for _, row in pf.iterrows():

        duration = float(
            row[
                "duration_sec"
            ]
        )


        n_windows = int(

            np.floor(
                (
                    duration
                    - WINDOW_SEC
                )
                /
                STRIDE_SEC
            )
            + 1
        )


        if n_windows <= 0:

            continue


        starts = (

            float(
                row[
                    "start_abs_sec"
                ]
            )

            +

            np.arange(
                n_windows,
                dtype=float
            )
            * STRIDE_SEC
        )


        starts_parts.append(
            starts
        )


        ends_parts.append(
            starts
            + WINDOW_SEC
        )


    if not starts_parts:

        patient_rows.append({

            "patient":
                patient,

            "eligible_events":
                len(
                    eligible_events
                ),

            "warmup_events":
                np.nan,

            "future_events":
                0,

            "future_events_with_clean":
                0,

            "future_clean_windows":
                0,

            "future_clean_exposure_hours":
                0.0,

            "qualifies":
                False,

            "reason":
                "no_compatible_windows"
        })


        print(
            "QUALIFIES: False"
        )

        continue


    window_starts = np.concatenate(
        starts_parts
    )

    window_ends = np.concatenate(
        ends_parts
    )


    # ========================================================
    # 6. FROZEN CLEAN-INTERICTAL MASK
    #
    # >= 4 hours away from ALL seizures.
    # ========================================================

    clean_mask = np.ones(
        len(
            window_starts
        ),
        dtype=bool
    )


    for _, event in (
        all_events.iterrows()
    ):

        exclusion_start = (

            float(
                event[
                    "onset_abs_sec"
                ]
            )

            -

            CLEAN_BUFFER_SEC
        )


        exclusion_end = (

            float(
                event[
                    "offset_abs_sec"
                ]
            )

            +

            CLEAN_BUFFER_SEC
        )


        clean_mask &= ~(
            (
                window_starts
                < exclusion_end
            )
            &
            (
                window_ends
                > exclusion_start
            )
        )


    # ========================================================
    # 7. FIND EARLIEST VALID WARM-UP
    # ========================================================

    chosen = None


    max_k = (

        len(
            eligible_events
        )

        -

        MIN_FUTURE_EVENTS
    )


    for k in range(
        MIN_WARMUP_EVENTS,
        max_k + 1
    ):

        warmup_last = (
            eligible_events.iloc[
                k - 1
            ]
        )


        cutoff = float(
            warmup_last[
                "offset_abs_sec"
            ]
        )


        # ----------------------------------------------------
        # Causal clean negatives available at this point
        # ----------------------------------------------------

        causal_clean_mask = (
            clean_mask
            &
            (
                window_ends
                <= cutoff
            )
        )


        causal_indices = np.flatnonzero(
            causal_clean_mask
        )


        if len(
            causal_indices
        ) == 0:

            continue


        # ----------------------------------------------------
        # Reserve retention negatives
        # ----------------------------------------------------

        selected = (
            select_nonoverlap_positions(

                window_starts[
                    causal_indices
                ],

                window_ends[
                    causal_indices
                ],

                n=RETENTION_NEG_TOTAL,

                seed=(
                    SEED
                    +
                    patient_number
                    * 10000
                    +
                    k
                )
            )
        )


        if selected is None:

            continue


        selected_global = (
            causal_indices[
                selected
            ]
        )


        retention_intervals = list(

            zip(
                window_starts[
                    selected_global
                ],

                window_ends[
                    selected_global
                ]
            )
        )


        initial_clean_remaining = (
            count_after_excluding_intervals(

                window_starts[
                    causal_indices
                ],

                window_ends[
                    causal_indices
                ],

                retention_intervals
            )
        )


        if (
            initial_clean_remaining
            <
            MIN_INITIAL_NEGATIVE_WINDOWS
        ):

            continue


        # ====================================================
        # 8. FUTURE STREAM
        # ====================================================

        future_events = (
            eligible_events.iloc[
                k:
            ]
            .reset_index(
                drop=True
            )
        )


        if (
            len(
                future_events
            )
            <
            MIN_FUTURE_EVENTS
        ):

            continue


        previous_event = (
            eligible_events.iloc[
                k - 1
            ]
        )


        episode_rows = []

        total_future_clean = 0

        events_with_clean = 0


        for episode_index, (
            _,
            current_event
        ) in enumerate(
            future_events.iterrows(),
            start=1
        ):

            block_start = float(
                previous_event[
                    "offset_abs_sec"
                ]
            )


            block_end = float(
                current_event[
                    "preictal_end_abs_sec"
                ]
            )


            current_preictal_mask = (
                (
                    window_starts
                    >=
                    float(
                        current_event[
                            "preictal_start_abs_sec"
                        ]
                    )
                )
                &
                (
                    window_ends
                    <=
                    float(
                        current_event[
                            "preictal_end_abs_sec"
                        ]
                    )
                )
            )


            block_mask = (
                (
                    window_starts
                    >= block_start
                )
                &
                (
                    window_ends
                    <= block_end
                )
            )


            future_clean_mask = (
                block_mask
                &
                clean_mask
            )


            n_preictal = int(
                current_preictal_mask.sum()
            )


            n_clean = int(
                future_clean_mask.sum()
            )


            total_future_clean += (
                n_clean
            )


            if n_clean > 0:

                events_with_clean += 1


            episode_rows.append({

                "patient":
                    patient,

                "warmup_events":
                    k,

                "episode_index":
                    episode_index,

                "event_id":
                    current_event[
                        "event_id"
                    ],

                "n_preictal_windows":
                    n_preictal,

                "n_clean_interictal_windows":
                    n_clean,

                "contains_clean_interictal":
                    n_clean > 0
            })


            previous_event = (
                current_event
            )


        # ----------------------------------------------------
        # CORRECTED QUALIFICATION:
        #
        # We only need some future clean interictal exposure
        # across the COMPLETE FUTURE STREAM.
        #
        # Individual seizure blocks may be positive-only.
        # ----------------------------------------------------

        if total_future_clean <= 0:

            continue


        chosen = {

            "k":
                k,

            "future_events":
                len(
                    future_events
                ),

            "events_with_clean":
                events_with_clean,

            "future_clean":
                total_future_clean,

            "initial_clean_remaining":
                initial_clean_remaining,

            "episodes":
                episode_rows
        }


        break


    # ========================================================
    # 9. PATIENT RESULT
    # ========================================================

    if chosen is None:

        patient_rows.append({

            "patient":
                patient,

            "eligible_events":
                len(
                    eligible_events
                ),

            "warmup_events":
                np.nan,

            "future_events":
                0,

            "future_events_with_clean":
                0,

            "future_clean_windows":
                0,

            "future_clean_exposure_hours":
                0.0,

            "qualifies":
                False,

            "reason":
                (
                    "no_valid_causal_warmup_"
                    "with_future_interictal_exposure"
                )
        })


        print(
            "QUALIFIES: False"
        )


        print(
            "Reason: no valid warm-up with "
            "future interictal exposure"
        )


    else:

        # With 5-s stride, this is the approximate
        # chronological exposure represented by
        # overlapping clean windows.

        future_hours = (

            chosen[
                "future_clean"
            ]

            *

            STRIDE_SEC

            /

            3600.0
        )


        patient_rows.append({

            "patient":
                patient,

            "eligible_events":
                len(
                    eligible_events
                ),

            "warmup_events":
                chosen[
                    "k"
                ],

            "future_events":
                chosen[
                    "future_events"
                ],

            "future_events_with_clean":
                chosen[
                    "events_with_clean"
                ],

            "future_clean_windows":
                chosen[
                    "future_clean"
                ],

            "future_clean_exposure_hours":
                future_hours,

            "initial_clean_after_retention":
                chosen[
                    "initial_clean_remaining"
                ],

            "qualifies":
                True,

            "reason":
                "qualified"
        })


        future_rows.extend(
            chosen[
                "episodes"
            ]
        )


        print(
            "QUALIFIES: True"
        )


        print(
            "Warm-up events:",
            chosen[
                "k"
            ]
        )


        print(
            "Future seizures:",
            chosen[
                "future_events"
            ]
        )


        print(
            "Future clean-interictal exposure:",
            f"{future_hours:.2f} h"
        )


# ============================================================
# 10. FINAL DATAFRAMES
# ============================================================

screen_v2 = pd.DataFrame(
    patient_rows
)


future_v2 = pd.DataFrame(
    future_rows
)


qualified = (
    screen_v2[
        screen_v2[
            "qualifies"
        ]
    ]
    .copy()
)


# ============================================================
# 11. HASH
# ============================================================

hash_columns = [

    "patient",

    "eligible_events",

    "warmup_events",

    "future_events",

    "future_clean_windows",

    "qualifies",

    "reason"
]


screen_hash = hashlib.sha256(

    screen_v2[
        hash_columns
    ]
    .fillna(
        ""
    )
    .to_csv(
        index=False
    )
    .encode(
        "utf-8"
    )

).hexdigest()


# ============================================================
# 12. SAVE
# ============================================================

screen_v2.to_csv(
    NEW_SCREEN_ROOT
    / "patient_screen_v2.csv",
    index=False
)


future_v2.to_csv(
    NEW_SCREEN_ROOT
    / "future_stream_v2.csv",
    index=False
)


(
    NEW_SCREEN_ROOT
    / "screen_config_v2.json"
).write_text(

    json.dumps(
        {
            "screen_hash":
                screen_hash,

            "config":
                SCREEN_CONFIG
        },
        indent=2
    )
)


# ============================================================
# 13. REPORT
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "CORRECTED STREAM-LEVEL COHORT SCREEN COMPLETE"
)

print(
    "=" * 90
)


print(
    "Screen hash:",
    screen_hash
)


print(
    "\nQUALIFIED PATIENTS"
)


display(
    qualified[
        [
            "patient",
            "eligible_events",
            "warmup_events",
            "future_events",
            "future_events_with_clean",
            "future_clean_windows",
            "future_clean_exposure_hours",
            "initial_clean_after_retention"
        ]
    ]
)


print(
    "\nALL PATIENTS"
)


display(
    screen_v2[
        [
            "patient",
            "eligible_events",
            "warmup_events",
            "future_events",
            "future_events_with_clean",
            "future_clean_exposure_hours",
            "qualifies",
            "reason"
        ]
    ]
)


print(
    "\nNUMBER QUALIFIED:",
    int(
        qualified.shape[
            0
        ]
    )
)


print(
    "\nQUALIFIED PATIENT IDS:"
)


print(
    qualified[
        "patient"
    ].tolist()
)


print(
    "\nTOTAL FUTURE SEIZURES:"
)


print(
    int(
        qualified[
            "future_events"
        ].sum()
    )
)


print(
    "\nTOTAL FUTURE CLEAN-INTERICTAL EXPOSURE:"
)


print(
    f"{qualified['future_clean_exposure_hours'].sum():.2f} hours"
)


if not future_v2.empty:

    print(
        "\nFUTURE EPISODES"
    )

    display(
        future_v2
    )


print(
    "\nSaved to:",
    NEW_SCREEN_ROOT
)


In [ ]:

# ============================================================
# Extend graph cache to chb09 and chb10
# ============================================================

from pathlib import Path
import gc
import hashlib
import json
import re
import time

import mne
import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt, welch

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

DATA_ROOT = Path(
    "/content/drive/MyDrive/chb-mit-scalp-eeg-database-1.0.0"
)

SCREEN_V1_ROOT = Path(
    "/content/drive/MyDrive/EEG_Research/"
    "continual_graph_forecasting/cohort_screen_v1"
)

SCREEN_V2_ROOT = Path(
    "/content/drive/MyDrive/EEG_Research/"
    "continual_graph_forecasting/cohort_screen_v2"
)

CACHE_ROOT = Path(
    "/content/drive/MyDrive/EEG_Research/"
    "continual_graph_forecasting/graph_cache_v1"
)

EXTENSION_ROOT = Path(
    "/content/drive/MyDrive/EEG_Research/"
    "continual_graph_forecasting/"
    "protocol_extension_chb09_chb10_v1"
)

EXTENSION_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_PATIENTS = ("chb09", "chb10")

# ------------------------------------------------------------
# 2. Frozen preprocessing
# ------------------------------------------------------------

SFREQ = 256.0
WINDOW_SEC = 10.0
STRIDE_SEC = 5.0
WINDOW_SAMPLES = int(WINDOW_SEC * SFREQ)
CLEAN_BUFFER_SEC = 4 * 60 * 60
WINDOW_BATCH_SIZE = 128

FIXED_CHANNELS = [
    "C3-P3",
    "C4-P4",
    "CZ-PZ",
    "F3-C3",
    "F4-C4",
    "F7-T7",
    "F8-T8",
    "FP1-F3",
    "FP1-F7",
    "FP2-F4",
    "FP2-F8",
    "FT10-T8",
    "FT9-FT10",
    "FZ-CZ",
    "P3-O1",
    "P4-O2",
    "P7-O1",
    "P7-T7",
    "P8-O2",
    "T7-FT9",
    "T7-P7",
    "T8-P8",
]

BANDS = {
    "delta": (0.5, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
    "gamma": (30.0, 45.0),
}

BAND_NAMES = list(BANDS.keys())

TRIU_I, TRIU_J = np.triu_indices(len(FIXED_CHANNELS), k=1)
assert len(TRIU_I) == 231

SOS = butter(
    4,
    [0.5, 45.0],
    btype="bandpass",
    fs=SFREQ,
    output="sos",
)

# ------------------------------------------------------------
# 3. Load frozen cohort-screen outputs
# ------------------------------------------------------------

files_screen = pd.read_csv(
    SCREEN_V1_ROOT / "file_montage_screen.csv"
)

events_screen = pd.read_csv(
    SCREEN_V1_ROOT / "all_events_screen.csv"
)

cohort_v2 = pd.read_csv(
    SCREEN_V2_ROOT / "patient_screen_v2.csv"
)


def to_bool(series):
    if series.dtype == bool:
        return series

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .eq("true")
    )


files_screen["montage_compatible"] = to_bool(
    files_screen["montage_compatible"]
)

events_screen["eligible"] = to_bool(
    events_screen["eligible"]
)

cohort_v2["qualifies"] = to_bool(
    cohort_v2["qualifies"]
)

for patient in TARGET_PATIENTS:
    row = cohort_v2[
        cohort_v2["patient"] == patient
    ]

    if len(row) != 1:
        raise RuntimeError(
            f"Missing cohort result for {patient}"
        )

    if not bool(row.iloc[0]["qualifies"]):
        raise RuntimeError(
            f"{patient} is not qualified in cohort_screen_v2."
        )

print("Confirmed qualified:", TARGET_PATIENTS)

# ------------------------------------------------------------
# 4. Reuse exact existing preprocessing hash from chb06
# ------------------------------------------------------------

chb06_cache_files = sorted(
    (CACHE_ROOT / "chb06").glob("*.npz")
)

if not chb06_cache_files:
    raise RuntimeError(
        "No existing chb06 graph cache found."
    )

with np.load(
    chb06_cache_files[0],
    allow_pickle=False,
) as z:
    EXISTING_PREPROCESS_HASH = str(
        z["preprocess_hash"].item()
    )

print(
    "Existing preprocessing hash:",
    EXISTING_PREPROCESS_HASH,
)

# ------------------------------------------------------------
# 5. Build exact window manifest for chb09 + chb10
# ------------------------------------------------------------

manifest_parts = []

for patient in TARGET_PATIENTS:
    print("\nBuilding manifest:", patient)

    patient_files = (
        files_screen[
            (files_screen["patient"] == patient)
            & files_screen["montage_compatible"]
        ]
        .sort_values("start_abs_sec")
        .reset_index(drop=True)
    )

    all_events = (
        events_screen[
            events_screen["patient"] == patient
        ]
        .sort_values("onset_abs_sec")
        .reset_index(drop=True)
    )

    eligible_events = (
        all_events[
            all_events["eligible"]
        ]
        .sort_values("onset_abs_sec")
        .reset_index(drop=True)
    )

    print(
        "  compatible EDFs:",
        len(patient_files),
    )
    print(
        "  all seizures:",
        len(all_events),
    )
    print(
        "  eligible seizures:",
        len(eligible_events),
    )

    for _, file_row in patient_files.iterrows():
        fname = str(file_row["file"])
        file_start = float(
            file_row["start_abs_sec"]
        )
        duration = float(
            file_row["duration_sec"]
        )

        n_windows = int(
            np.floor(
                (duration - WINDOW_SEC)
                / STRIDE_SEC
            )
            + 1
        )

        if n_windows <= 0:
            continue

        window_index = np.arange(
            n_windows,
            dtype=np.int32,
        )

        start_rel = (
            window_index.astype(float)
            * STRIDE_SEC
        )
        end_rel = start_rel + WINDOW_SEC

        start_abs = file_start + start_rel
        end_abs = file_start + end_rel

        zone = np.full(
            n_windows,
            "other_excluded",
            dtype="<U32",
        )

        event_id = np.full(
            n_windows,
            "",
            dtype="<U32",
        )

        # Frozen >=4-hour clean-interictal rule
        clean_mask = np.ones(
            n_windows,
            dtype=bool,
        )

        for _, ev in all_events.iterrows():
            exclusion_start = (
                float(ev["onset_abs_sec"])
                - CLEAN_BUFFER_SEC
            )
            exclusion_end = (
                float(ev["offset_abs_sec"])
                + CLEAN_BUFFER_SEC
            )

            overlaps_exclusion = (
                (start_abs < exclusion_end)
                & (end_abs > exclusion_start)
            )

            clean_mask &= ~overlaps_exclusion

        zone[clean_mask] = "clean_interictal"

        # Eligible preictal windows
        already_preictal = np.zeros(
            n_windows,
            dtype=bool,
        )

        for _, ev in eligible_events.iterrows():
            pre_start = float(
                ev["preictal_start_abs_sec"]
            )
            pre_end = float(
                ev["preictal_end_abs_sec"]
            )

            mask = (
                (start_abs >= pre_start)
                & (end_abs <= pre_end)
            )

            if np.any(
                mask & already_preictal
            ):
                raise RuntimeError(
                    f"{patient}/{fname}: "
                    "preictal intervals overlap."
                )

            zone[mask] = "preictal"
            event_id[mask] = str(
                ev["event_id"]
            )
            already_preictal |= mask

        manifest_parts.append(
            pd.DataFrame(
                {
                    "patient": patient,
                    "file": fname,
                    "window_index_in_file":
                        window_index,
                    "start_rel_sec":
                        start_rel,
                    "end_rel_sec":
                        end_rel,
                    "start_abs_sec":
                        start_abs,
                    "end_abs_sec":
                        end_abs,
                    "zone":
                        zone,
                    "event_id":
                        event_id,
                }
            )
        )

window_manifest = pd.concat(
    manifest_parts,
    ignore_index=True,
)

window_manifest["window_key"] = (
    window_manifest["patient"].astype(str)
    + "|"
    + window_manifest["file"].astype(str)
    + "|"
    + window_manifest[
        "window_index_in_file"
    ].astype(str)
)

for patient in TARGET_PATIENTS:
    temp = window_manifest[
        window_manifest["patient"] == patient
    ]

    n_pre = int(
        (temp["zone"] == "preictal").sum()
    )
    n_clean = int(
        (
            temp["zone"]
            == "clean_interictal"
        ).sum()
    )

    print(f"\n{patient}:")
    print(
        "  total windows:",
        f"{len(temp):,}",
    )
    print(
        "  preictal:",
        f"{n_pre:,}",
    )
    print(
        "  clean interictal:",
        f"{n_clean:,}",
    )

    if n_pre == 0:
        raise RuntimeError(
            f"{patient}: no preictal windows."
        )

    if n_clean == 0:
        raise RuntimeError(
            f"{patient}: no clean interictal windows."
        )

# ------------------------------------------------------------
# 6. Save extension manifest + exact hash
# ------------------------------------------------------------

window_manifest.to_pickle(
    EXTENSION_ROOT / "window_manifest.pkl"
)

window_manifest.to_csv(
    EXTENSION_ROOT / "window_manifest.csv",
    index=False,
)

target_events = (
    events_screen[
        events_screen["patient"].isin(
            TARGET_PATIENTS
        )
    ]
    .copy()
)

target_files = (
    files_screen[
        files_screen["patient"].isin(
            TARGET_PATIENTS
        )
        & files_screen[
            "montage_compatible"
        ]
    ]
    .copy()
)

target_events.to_csv(
    EXTENSION_ROOT / "events.csv",
    index=False,
)

target_files.to_csv(
    EXTENSION_ROOT / "files.csv",
    index=False,
)

manifest_hash_columns = [
    "window_key",
    "start_abs_sec",
    "end_abs_sec",
    "zone",
    "event_id",
]

manifest_hash = hashlib.sha256(
    window_manifest[
        manifest_hash_columns
    ]
    .sort_values("window_key")
    .to_csv(index=False)
    .encode("utf-8")
).hexdigest()

print(
    "\nExtension manifest hash:",
    manifest_hash,
)

# ------------------------------------------------------------
# 7. Channel helpers
# ------------------------------------------------------------

def canonical_channel_name(name):
    x = (
        str(name)
        .strip()
        .upper()
    )

    x = re.sub(
        r"-(\d+)$",
        "",
        x,
    )

    return x


def select_fixed_channels(raw):
    mapping = {}

    for actual in raw.ch_names:
        canonical = (
            canonical_channel_name(
                actual
            )
        )

        if canonical in FIXED_CHANNELS:
            mapping.setdefault(
                canonical,
                [],
            ).append(actual)

    missing = [
        ch
        for ch in FIXED_CHANNELS
        if ch not in mapping
    ]

    if missing:
        raise RuntimeError(
            f"Missing frozen channels: "
            f"{missing}"
        )

    selected = []
    duplicates = []

    for canonical in FIXED_CHANNELS:
        candidates = mapping[
            canonical
        ]

        chosen = candidates[0]
        selected.append(chosen)

        if len(candidates) > 1:
            duplicates.append(
                {
                    "canonical_channel":
                        canonical,
                    "candidates":
                        "|".join(
                            candidates
                        ),
                    "selected":
                        chosen,
                }
            )

    if len(selected) != 22:
        raise RuntimeError(
            "Expected exactly 22 "
            "selected channels."
        )

    return selected, duplicates


# ------------------------------------------------------------
# 8. Window extraction
# ------------------------------------------------------------

def extract_window_batch(
    filtered,
    starts_rel,
):
    starts_sample = np.rint(
        np.asarray(
            starts_rel,
            dtype=float,
        )
        * SFREQ
    ).astype(np.int64)

    batch = np.empty(
        (
            len(starts_sample),
            len(FIXED_CHANNELS),
            WINDOW_SAMPLES,
        ),
        dtype=np.float64,
    )

    for i, start in enumerate(
        starts_sample
    ):
        stop = (
            start
            + WINDOW_SAMPLES
        )

        if (
            start < 0
            or stop > filtered.shape[1]
        ):
            raise RuntimeError(
                f"Window [{start}:{stop}] "
                f"outside signal length "
                f"{filtered.shape[1]}"
            )

        batch[i] = filtered[
            :,
            start:stop,
        ]

    return batch


# ------------------------------------------------------------
# 9. Feature extraction
# ------------------------------------------------------------

def compute_features(x):
    x = np.asarray(
        x,
        dtype=np.float64,
    )

    qc_finite = np.isfinite(
        x
    ).all(
        axis=(1, 2)
    )

    channel_std = np.nanstd(
        x,
        axis=2,
    )

    qc_nonzero_variance = np.all(
        channel_std > 0.0,
        axis=1,
    )

    max_abs_uv = (
        np.nanmax(
            np.abs(x),
            axis=(1, 2),
        )
        * 1e6
    ).astype(np.float32)

    min_channel_std_uv = (
        np.nanmin(
            channel_std,
            axis=1,
        )
        * 1e6
    ).astype(np.float32)

    nperseg = int(
        2.0 * SFREQ
    )

    noverlap = nperseg // 2

    freqs, psd = welch(
        x,
        fs=SFREQ,
        window="hann",
        nperseg=nperseg,
        noverlap=noverlap,
        detrend="constant",
        scaling="density",
        axis=-1,
    )

    band_power = []

    for band_idx, band_name in enumerate(
        BAND_NAMES
    ):
        low, high = BANDS[
            band_name
        ]

        if band_idx < (
            len(BAND_NAMES) - 1
        ):
            mask = (
                (freqs >= low)
                & (freqs < high)
            )
        else:
            mask = (
                (freqs >= low)
                & (freqs <= high)
            )

        if mask.sum() < 2:
            raise RuntimeError(
                f"Insufficient frequency "
                f"bins for {band_name}"
            )

        power = np.trapezoid(
            psd[..., mask],
            freqs[mask],
            axis=-1,
        )

        band_power.append(power)

    band_power = np.stack(
        band_power,
        axis=-1,
    )

    eps = np.finfo(
        np.float64
    ).tiny

    node_features = np.log10(
        np.maximum(
            band_power,
            eps,
        )
    ).astype(np.float32)

    centered = (
        x
        - np.mean(
            x,
            axis=2,
            keepdims=True,
        )
    )

    norm = np.linalg.norm(
        centered,
        axis=2,
        keepdims=True,
    )

    norm = np.maximum(
        norm,
        np.finfo(
            np.float64
        ).eps,
    )

    normalized = (
        centered / norm
    )

    correlation = np.einsum(
        "wcs,wds->wcd",
        normalized,
        normalized,
        optimize=True,
    )

    correlation = np.clip(
        correlation,
        -1.0,
        1.0,
    )

    connectivity = correlation[
        :,
        TRIU_I,
        TRIU_J,
    ].astype(np.float32)

    return (
        node_features,
        connectivity,
        qc_finite,
        qc_nonzero_variance,
        max_abs_uv,
        min_channel_std_uv,
    )


# ------------------------------------------------------------
# 10. Process one EDF
# ------------------------------------------------------------

duplicate_audit_rows = []


def process_one_edf(
    patient,
    fname,
):
    edf_path = (
        DATA_ROOT
        / patient
        / fname
    )

    output_dir = (
        CACHE_ROOT
        / patient
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    out_path = (
        output_dir
        / f"{Path(fname).stem}.npz"
    )

    fw = (
        window_manifest[
            (
                window_manifest["patient"]
                == patient
            )
            & (
                window_manifest["file"]
                == fname
            )
        ]
        .sort_values(
            "window_index_in_file"
        )
        .reset_index(drop=True)
    )

    if fw.empty:
        return {
            "patient": patient,
            "source_file": fname,
            "status": "no_windows",
            "n_windows": 0,
            "n_qc_fail": 0,
        }

    # Resume only if exact cache is valid.
    if out_path.exists():
        try:
            with np.load(
                out_path,
                allow_pickle=False,
            ) as z:
                required = {
                    "preprocess_hash",
                    "extension_manifest_hash",
                    "node_features",
                    "connectivity",
                    "hard_qc_pass",
                    "source_file",
                }

                valid_keys = (
                    required
                    .issubset(
                        set(z.files)
                    )
                )

                same_preprocess_hash = (
                    valid_keys
                    and str(
                        z[
                            "preprocess_hash"
                        ].item()
                    )
                    == EXISTING_PREPROCESS_HASH
                )

                same_manifest_hash = (
                    valid_keys
                    and str(
                        z[
                            "extension_manifest_hash"
                        ].item()
                    )
                    == manifest_hash
                )

                same_windows = (
                    valid_keys
                    and z[
                        "node_features"
                    ].shape[0]
                    == len(fw)
                )

            if (
                valid_keys
                and same_preprocess_hash
                and same_manifest_hash
                and same_windows
            ):
                return {
                    "patient":
                        patient,
                    "source_file":
                        fname,
                    "status":
                        "cached",
                    "n_windows":
                        len(fw),
                    "n_qc_fail":
                        0,
                }

        except Exception:
            pass

        out_path.unlink(
            missing_ok=True
        )

    raw = mne.io.read_raw_edf(
        edf_path,
        preload=False,
        verbose="ERROR",
    )

    sfreq = float(
        raw.info["sfreq"]
    )

    if not np.isclose(
        sfreq,
        SFREQ,
        atol=1e-9,
        rtol=0.0,
    ):
        raw.close()

        raise RuntimeError(
            f"{patient}/{fname}: "
            f"sampling rate={sfreq}, "
            f"expected={SFREQ}"
        )

    selected_channels, duplicates = (
        select_fixed_channels(raw)
    )

    for duplicate in duplicates:
        duplicate_audit_rows.append(
            {
                "patient":
                    patient,
                "file":
                    fname,
                **duplicate,
            }
        )

    data = raw.get_data(
        picks=selected_channels
    ).astype(
        np.float64,
        copy=False,
    )

    raw.close()
    del raw

    if data.shape[0] != 22:
        raise RuntimeError(
            f"{patient}/{fname}: "
            f"unexpected channel count "
            f"{data.shape[0]}"
        )

    if not np.isfinite(data).all():
        raise RuntimeError(
            f"{patient}/{fname}: "
            "non-finite raw EEG."
        )

    filtered = sosfiltfilt(
        SOS,
        data,
        axis=1,
    )

    del data

    n_windows = len(fw)

    node_all = np.empty(
        (
            n_windows,
            22,
            5,
        ),
        dtype=np.float32,
    )

    conn_all = np.empty(
        (
            n_windows,
            231,
        ),
        dtype=np.float32,
    )

    qc_finite_all = np.empty(
        n_windows,
        dtype=bool,
    )

    qc_nonzero_all = np.empty(
        n_windows,
        dtype=bool,
    )

    max_abs_all = np.empty(
        n_windows,
        dtype=np.float32,
    )

    min_std_all = np.empty(
        n_windows,
        dtype=np.float32,
    )

    for batch_start in range(
        0,
        n_windows,
        WINDOW_BATCH_SIZE,
    ):
        batch_stop = min(
            batch_start
            + WINDOW_BATCH_SIZE,
            n_windows,
        )

        batch_rows = fw.iloc[
            batch_start:batch_stop
        ]

        x = extract_window_batch(
            filtered,
            batch_rows[
                "start_rel_sec"
            ].to_numpy(),
        )

        (
            node_features,
            connectivity,
            qc_finite,
            qc_nonzero,
            max_abs_uv,
            min_std_uv,
        ) = compute_features(x)

        node_all[
            batch_start:batch_stop
        ] = node_features

        conn_all[
            batch_start:batch_stop
        ] = connectivity

        qc_finite_all[
            batch_start:batch_stop
        ] = qc_finite

        qc_nonzero_all[
            batch_start:batch_stop
        ] = qc_nonzero

        max_abs_all[
            batch_start:batch_stop
        ] = max_abs_uv

        min_std_all[
            batch_start:batch_stop
        ] = min_std_uv

        del (
            x,
            node_features,
            connectivity,
        )

    del filtered

    hard_qc_pass = (
        qc_finite_all
        & qc_nonzero_all
    )

    n_qc_fail = int(
        (~hard_qc_pass).sum()
    )

    np.savez_compressed(
        str(out_path),

        preprocess_hash=np.array(
            EXISTING_PREPROCESS_HASH
        ),

        extension_manifest_hash=np.array(
            manifest_hash
        ),

        patient=np.array(patient),

        source_file=np.array(fname),

        selected_channels=np.asarray(
            selected_channels
        ),

        window_index_in_file=fw[
            "window_index_in_file"
        ].to_numpy(
            dtype=np.int32
        ),

        start_rel_sec=fw[
            "start_rel_sec"
        ].to_numpy(
            dtype=np.float32
        ),

        start_abs_sec=fw[
            "start_abs_sec"
        ].to_numpy(
            dtype=np.float64
        ),

        end_abs_sec=fw[
            "end_abs_sec"
        ].to_numpy(
            dtype=np.float64
        ),

        zone=fw[
            "zone"
        ].astype(str).to_numpy(),

        event_id=fw[
            "event_id"
        ].astype(str).to_numpy(),

        node_features=node_all,

        connectivity=conn_all,

        qc_finite=qc_finite_all,

        qc_nonzero_variance=(
            qc_nonzero_all
        ),

        hard_qc_pass=hard_qc_pass,

        max_abs_uv=max_abs_all,

        min_channel_std_uv=(
            min_std_all
        ),
    )

    p99 = float(
        np.nanpercentile(
            max_abs_all,
            99,
        )
    )

    del (
        node_all,
        conn_all,
        qc_finite_all,
        qc_nonzero_all,
        max_abs_all,
        min_std_all,
        hard_qc_pass,
    )

    gc.collect()

    return {
        "patient":
            patient,
        "source_file":
            fname,
        "status":
            "processed",
        "n_windows":
            n_windows,
        "n_qc_fail":
            n_qc_fail,
        "p99_max_abs_uv":
            p99,
        "cache_path":
            str(out_path),
    }


# ------------------------------------------------------------
# 11. Exact file list — FIXED
# Sort BEFORE dropping start_abs_sec.
# ------------------------------------------------------------

usable_files = (
    target_files
    .sort_values(
        [
            "patient",
            "start_abs_sec",
        ]
    )
    [
        [
            "patient",
            "file",
        ]
    ]
    .reset_index(drop=True)
)

print(
    "\nFiles to process:",
    len(usable_files),
)

print(
    usable_files[
        "patient"
    ].value_counts()
)

expected_counts = {
    "chb09": 19,
    "chb10": 25,
}

observed_counts = (
    usable_files[
        "patient"
    ]
    .value_counts()
    .to_dict()
)

assert (
    observed_counts
    == expected_counts
), observed_counts

assert len(usable_files) == 44

print(
    "\nFile-list verification: PASS"
)

# ------------------------------------------------------------
# 12. Run
# ------------------------------------------------------------

results = []
start_time = time.time()

for i, row in usable_files.iterrows():
    patient = str(
        row["patient"]
    )
    fname = str(
        row["file"]
    )

    print(
        f"[{i+1:03d}/"
        f"{len(usable_files):03d}] "
        f"{patient}/{fname}"
    )

    result = process_one_edf(
        patient,
        fname,
    )

    results.append(result)

    if (
        result["status"]
        == "processed"
    ):
        print(
            "    "
            f"windows="
            f"{result['n_windows']:,}"
            " | "
            f"qc_fail="
            f"{result['n_qc_fail']}"
            " | "
            f"p99|uV|="
            f"{result['p99_max_abs_uv']:.1f}"
        )
    else:
        print(
            "    ",
            result["status"],
        )

    pd.DataFrame(
        results
    ).to_csv(
        EXTENSION_ROOT
        / "cache_progress.csv",
        index=False,
    )

# ------------------------------------------------------------
# 13. Final audit
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

results_df.to_csv(
    EXTENSION_ROOT
    / "cache_run_manifest.csv",
    index=False,
)

duplicate_df = pd.DataFrame(
    duplicate_audit_rows
)

duplicate_df.to_csv(
    EXTENSION_ROOT
    / "duplicate_channel_audit.csv",
    index=False,
)

runtime_minutes = (
    time.time()
    - start_time
) / 60.0

print(
    "\n"
    + "=" * 90
)
print(
    "GRAPH CACHE EXTENSION COMPLETE"
)
print(
    "=" * 90
)
print(
    f"Runtime: "
    f"{runtime_minutes:.1f} min"
)

print(
    "\nSTATUS"
)

display(
    results_df[
        "status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis("status")
    .reset_index(name="count")
)

bad_status = results_df[
    ~results_df[
        "status"
    ].isin(
        [
            "processed",
            "cached",
        ]
    )
]

if not bad_status.empty:
    raise RuntimeError(
        "At least one EDF did not "
        "process successfully."
    )

if (
    results_df[
        "n_qc_fail"
    ]
    .fillna(0)
    .sum()
    != 0
):
    raise RuntimeError(
        "Hard-QC failures detected."
    )

# ------------------------------------------------------------
# 14. Direct cache verification
# ------------------------------------------------------------

verification_rows = []

for _, row in usable_files.iterrows():
    patient = str(
        row["patient"]
    )
    fname = str(
        row["file"]
    )

    cache_path = (
        CACHE_ROOT
        / patient
        / f"{Path(fname).stem}.npz"
    )

    expected_windows = len(
        window_manifest[
            (
                window_manifest["patient"]
                == patient
            )
            & (
                window_manifest["file"]
                == fname
            )
        ]
    )

    if not cache_path.exists():
        raise RuntimeError(
            f"Missing cache: "
            f"{cache_path}"
        )

    with np.load(
        cache_path,
        allow_pickle=False,
    ) as z:
        actual_preprocess_hash = str(
            z["preprocess_hash"].item()
        )

        actual_manifest_hash = str(
            z[
                "extension_manifest_hash"
            ].item()
        )

        n_node = int(
            z[
                "node_features"
            ].shape[0]
        )

        n_conn = int(
            z[
                "connectivity"
            ].shape[0]
        )

        node_shape = tuple(
            z[
                "node_features"
            ].shape[1:]
        )

        conn_shape = tuple(
            z[
                "connectivity"
            ].shape[1:]
        )

        qc_fail = int(
            (
                ~z[
                    "hard_qc_pass"
                ]
            ).sum()
        )

        finite_nodes = bool(
            np.isfinite(
                z[
                    "node_features"
                ]
            ).all()
        )

        finite_conn = bool(
            np.isfinite(
                z[
                    "connectivity"
                ]
            ).all()
        )

    verification_rows.append(
        {
            "patient":
                patient,
            "file":
                fname,
            "expected_windows":
                expected_windows,
            "cached_windows":
                n_node,
            "connectivity_windows":
                n_conn,
            "node_shape":
                str(node_shape),
            "connectivity_shape":
                str(conn_shape),
            "qc_fail":
                qc_fail,
            "finite_node_features":
                finite_nodes,
            "finite_connectivity":
                finite_conn,
            "preprocess_hash_matches":
                (
                    actual_preprocess_hash
                    == EXISTING_PREPROCESS_HASH
                ),
            "manifest_hash_matches":
                (
                    actual_manifest_hash
                    == manifest_hash
                ),
        }
    )

verification_df = pd.DataFrame(
    verification_rows
)

verification_df.to_csv(
    EXTENSION_ROOT
    / "cache_verification.csv",
    index=False,
)

assert (
    verification_df[
        "expected_windows"
    ]
    == verification_df[
        "cached_windows"
    ]
).all()

assert (
    verification_df[
        "cached_windows"
    ]
    == verification_df[
        "connectivity_windows"
    ]
).all()

assert (
    verification_df[
        "node_shape"
    ]
    == "(22, 5)"
).all()

assert (
    verification_df[
        "connectivity_shape"
    ]
    == "(231,)"
).all()

assert (
    verification_df["qc_fail"]
    == 0
).all()

assert verification_df[
    "finite_node_features"
].all()

assert verification_df[
    "finite_connectivity"
].all()

assert verification_df[
    "preprocess_hash_matches"
].all()

assert verification_df[
    "manifest_hash_matches"
].all()

# ------------------------------------------------------------
# 15. Save extension config
# ------------------------------------------------------------

extension_config = {
    "patients":
        list(TARGET_PATIENTS),

    "existing_preprocess_hash":
        EXISTING_PREPROCESS_HASH,

    "extension_manifest_hash":
        manifest_hash,

    "sfreq":
        SFREQ,

    "window_sec":
        WINDOW_SEC,

    "stride_sec":
        STRIDE_SEC,

    "bandpass_hz":
        [0.5, 45.0],

    "filter":
        (
            "Butterworth order 4, "
            "zero-phase sosfiltfilt"
        ),

    "node_features":
        (
            "log10 absolute Welch band powers: "
            "delta/theta/alpha/beta/gamma"
        ),

    "connectivity":
        (
            "signed Pearson correlation "
            "upper triangle"
        ),

    "n_nodes":
        22,

    "n_node_features":
        5,

    "n_connectivity_features":
        231,

    "normalization":
        "none at cache stage",
}

(
    EXTENSION_ROOT
    / "extension_config.json"
).write_text(
    json.dumps(
        extension_config,
        indent=2,
    )
)

# ------------------------------------------------------------
# 16. Final summary
# ------------------------------------------------------------

print(
    "\nVERIFICATION BY PATIENT"
)

summary = (
    verification_df
    .groupby(
        "patient",
        as_index=False,
    )
    .agg(
        files=(
            "file",
            "count",
        ),
        windows=(
            "cached_windows",
            "sum",
        ),
        hard_qc_failures=(
            "qc_fail",
            "sum",
        ),
    )
)

display(summary)

print(
    "\nWINDOW ZONES"
)

zone_summary = (
    window_manifest
    .groupby(
        [
            "patient",
            "zone",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "windows"
        }
    )
)

display(zone_summary)

print(
    "\nDuplicate channel audit rows:",
    len(duplicate_df),
)

print(
    "\nSaved extension audit to:"
)
print(
    EXTENSION_ROOT
)

print(
    "\n=========================================="
)
print(
    "STAGE 5 COMPLETE"
)
print(
    "=========================================="
)

print(
    "\nNext: build FINAL chronological split for "
    "chb06 + chb09 + chb10, then train models."
)


In [ ]:
# ============================================================
# STAGE 6 — FINAL CHRONOLOGICAL SPLIT
# Final cohort: chb06 + chb09 + chb10
# ============================================================

from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd

ROOT = Path("/content/drive/MyDrive/EEG_Research/continual_graph_forecasting")
PROTOCOL_ROOT = ROOT / "protocol_v2"
EXT_ROOT = ROOT / "protocol_extension_chb09_chb10_v1"
SCREEN2_ROOT = ROOT / "cohort_screen_v2"
CACHE_ROOT = ROOT / "graph_cache_v1"
OUT_ROOT = ROOT / "final_split_v3"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

PATIENTS = ("chb06", "chb09", "chb10")
WARMUP_EXPECTED = {"chb06": 5, "chb09": 2, "chb10": 2}
FUTURE_EXPECTED = {"chb06": 4, "chb09": 2, "chb10": 4}
SEED = 2026

RET_POS_PER_EARLY_EVENT = 30
RET_NEG = 60
VAL_POS = 60
VAL_NEG = 60
MIN_TRAIN_NEG = 60


# -------------------- load --------------------

screen = pd.read_csv(SCREEN2_ROOT / "patient_screen_v2.csv")

# Use the chronology source that matches each patient's window manifest:
# chb06 uses protocol_v2; chb09 and chb10 use the extension chronology.
events06 = pd.read_csv(PROTOCOL_ROOT / "seizure_manifest_v2.csv")
events_ext = pd.read_csv(EXT_ROOT / "events.csv")

w06 = pd.read_pickle(PROTOCOL_ROOT / "window_manifest_v2.pkl")
wext = pd.read_pickle(EXT_ROOT / "window_manifest.pkl")

def as_bool(s):
    if s.dtype == bool:
        return s
    return s.astype(str).str.strip().str.lower().eq("true")

screen["qualifies"] = as_bool(screen["qualifies"])

events06["eligible"] = as_bool(events06["eligible_v2"])
events_ext["eligible"] = as_bool(events_ext["eligible"])

events06 = events06[events06["patient"] == "chb06"].copy()
events_ext = events_ext[
    events_ext["patient"].isin(("chb09", "chb10"))
].copy()

event_columns = [
    "patient",
    "file",
    "event_id",
    "onset_abs_sec",
    "offset_abs_sec",
    "preictal_start_abs_sec",
    "preictal_end_abs_sec",
    "eligible",
]

missing06 = [c for c in event_columns if c not in events06.columns]
missing_ext = [c for c in event_columns if c not in events_ext.columns]

if missing06:
    raise RuntimeError(f"protocol_v2 chb06 events missing columns: {missing06}")
if missing_ext:
    raise RuntimeError(f"extension events missing columns: {missing_ext}")

events = pd.concat(
    [
        events06[event_columns],
        events_ext[event_columns],
    ],
    ignore_index=True,
)

screen = screen[screen["patient"].isin(PATIENTS)].copy()
if set(screen["patient"]) != set(PATIENTS) or not screen["qualifies"].all():
    raise RuntimeError("Final cohort does not match cohort_screen_v2.")

warmup_map = {r.patient: int(r.warmup_events) for _, r in screen.iterrows()}
future_map = {r.patient: int(r.future_events) for _, r in screen.iterrows()}

# cohort_screen_v2 already established the available causal clean data.
# Stage 6 must be chronologically consistent with those audited counts.
screen_initial_clean_after_retention = {
    r.patient: int(round(r.initial_clean_after_retention))
    for _, r in screen.iterrows()
}

if warmup_map != WARMUP_EXPECTED:
    raise RuntimeError(f"Warm-up counts changed: {warmup_map}")
if future_map != FUTURE_EXPECTED:
    raise RuntimeError(f"Future counts changed: {future_map}")

screen_meta = json.loads((SCREEN2_ROOT / "screen_config_v2.json").read_text())
SCREEN_HASH = screen_meta["screen_hash"]

needed = [
    "patient", "file", "window_index_in_file",
    "start_rel_sec", "end_rel_sec",
    "start_abs_sec", "end_abs_sec",
    "zone", "event_id",
]

w06 = w06[w06["patient"] == "chb06"].copy()
wext = wext[wext["patient"].isin(("chb09", "chb10"))].copy()

for name, df in [("chb06", w06), ("extension", wext)]:
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise RuntimeError(f"{name} manifest missing: {missing}")

windows = pd.concat([w06[needed], wext[needed]], ignore_index=True)
windows["window_index_in_file"] = windows["window_index_in_file"].astype(np.int32)
windows["zone"] = windows["zone"].astype(str)
windows["event_id"] = windows["event_id"].fillna("").astype(str)
windows["window_key"] = (
    windows["patient"].astype(str) + "|" +
    windows["file"].astype(str) + "|" +
    windows["window_index_in_file"].astype(str)
)

if windows["window_key"].duplicated().any():
    raise RuntimeError("Duplicate physical windows after combining manifests.")

chb06_cache = sorted((CACHE_ROOT / "chb06").glob("*.npz"))
if not chb06_cache:
    raise RuntimeError("No chb06 graph cache found.")

with np.load(chb06_cache[0], allow_pickle=False) as z:
    PREPROCESS_HASH = str(z["preprocess_hash"].item())


# -------------------- helpers --------------------

def label(df):
    df = df.copy()
    df["label"] = (df["zone"] == "preictal").astype(np.int8)
    return df

def intervals(df):
    return list(zip(
        df["start_abs_sec"].astype(float),
        df["end_abs_sec"].astype(float),
    ))

def remove_overlap(df, blocked):
    if df.empty or not blocked:
        return df.copy()
    s = df["start_abs_sec"].to_numpy(float)
    e = df["end_abs_sec"].to_numpy(float)
    bad = np.zeros(len(df), dtype=bool)
    for a, b in blocked:
        bad |= (s < float(b)) & (e > float(a))
    return df.loc[~bad].copy()

def select_nonoverlap_random(df, n, seed):
    x = df.sort_values(
        ["start_abs_sec", "file", "window_index_in_file"]
    ).reset_index(drop=True)

    if len(x) < n:
        raise RuntimeError(f"Need {n} rows; only {len(x)} available.")

    rng = np.random.default_rng(seed)
    chosen, blocked = [], []

    for idx in rng.permutation(len(x)):
        row = x.iloc[idx]
        a, b = float(row.start_abs_sec), float(row.end_abs_sec)
        if any(a < d and b > c for c, d in blocked):
            continue
        chosen.append(idx)
        blocked.append((a, b))
        if len(chosen) == n:
            break

    if len(chosen) != n:
        raise RuntimeError(f"Could select only {len(chosen)}/{n} non-overlapping rows.")

    return x.iloc[chosen].sort_values("start_abs_sec").copy()

def select_latest_rows(df, n):
    """
    Chronological validation holdout.

    Validation windows are allowed to overlap one another because the
    10 s / 5 s-stride forecasting stream is itself overlapping. Leakage
    is prevented separately by removing from training every window that
    overlaps any validation interval.
    """
    x = (
        df.sort_values(
            ["start_abs_sec", "file", "window_index_in_file"]
        )
        .reset_index(drop=True)
    )

    if len(x) < n:
        raise RuntimeError(
            f"Need {n} validation rows but only {len(x)} are available."
        )

    return x.iloc[-n:].copy()

def sample_rows(df, n, seed):
    if n <= 0:
        return df.iloc[0:0].copy()
    if len(df) < n:
        raise RuntimeError(f"Need {n} rows; only {len(df)} available.")
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(len(df), n, replace=False))
    return df.iloc[idx].copy()

def exposure_hours(df):
    if df.empty:
        return 0.0

    total = 0.0
    for _, g in df.groupby(["patient", "file"], sort=False):
        ints = sorted(intervals(g))
        a0 = b0 = None

        for a, b in ints:
            if a0 is None:
                a0, b0 = a, b
            elif a <= b0:
                b0 = max(b0, b)
            else:
                total += b0 - a0
                a0, b0 = a, b

        if a0 is not None:
            total += b0 - a0

    return total / 3600.0

def tag(df, role, episode_id, episode_index,
        train=False, val=False, eval_=False, update=False):
    df = label(df)
    df["split_role"] = role
    df["episode_id"] = episode_id
    df["episode_index"] = episode_index
    df["use_for_training"] = bool(train)
    df["use_for_validation"] = bool(val)
    df["use_for_evaluation"] = bool(eval_)
    df["use_for_update"] = bool(update)
    return df


# -------------------- build split --------------------

parts = []
patient_rows = []
future_rows = []
event_rows = []

for p_idx, patient in enumerate(PATIENTS):

    print("\n" + "=" * 80)
    print("PATIENT:", patient)
    print("=" * 80)

    pe = (
        events[(events["patient"] == patient) & events["eligible"]]
        .sort_values("onset_abs_sec")
        .reset_index(drop=True)
    )

    pw = (
        windows[windows["patient"] == patient]
        .sort_values(["start_abs_sec", "file", "window_index_in_file"])
        .reset_index(drop=True)
    )

    k = warmup_map[patient]

    if len(pe) - k != future_map[patient]:
        raise RuntimeError(f"{patient}: future event count mismatch.")

    warmup = pe.iloc[:k].copy()
    future = pe.iloc[k:].reset_index(drop=True).copy()
    warmup_ids = set(warmup["event_id"].astype(str))

    for i, r in pe.iterrows():
        event_rows.append({
            "patient": patient,
            "eligible_order": i + 1,
            "event_id": r["event_id"],
            "file": r["file"],
            "onset_abs_sec": r["onset_abs_sec"],
            "offset_abs_sec": r["offset_abs_sec"],
            "role": "warmup" if i < k else "future",
            "future_episode": np.nan if i < k else i - k + 1,
        })

    # retention positives from first two eligible events
    ret_pos_list = []
    for j in range(2):
        eid = str(pe.iloc[j]["event_id"])
        pool = pw[(pw["zone"] == "preictal") & (pw["event_id"] == eid)].copy()
        ret_pos_list.append(
            select_nonoverlap_random(
                pool,
                RET_POS_PER_EARLY_EVENT,
                SEED + p_idx * 10000 + 100 + j,
            )
        )

    ret_pos = pd.concat(ret_pos_list, ignore_index=True)
    ret_pos_int = intervals(ret_pos)

    # causal clean pool
    cutoff = float(warmup.iloc[-1]["offset_abs_sec"])
    causal_clean = pw[
        (pw["zone"] == "clean_interictal") &
        (pw["end_abs_sec"] <= cutoff)
    ].copy()

    if causal_clean.empty:
        raise RuntimeError(f"{patient}: no causal clean-interictal data.")

    # --------------------------------------------------------
    # CRITICAL TIMELINE CONSISTENCY CHECK
    #
    # The cohort screen already found at least this many clean
    # rows AFTER reserving retention negatives. Therefore the
    # raw causal pool here must be at least that large.
    # If this fails, event/window time origins are inconsistent.
    # --------------------------------------------------------
    audited_after_retention = screen_initial_clean_after_retention[patient]

    if len(causal_clean) < audited_after_retention:
        raise RuntimeError(
            f"{patient}: timeline mismatch. Stage 6 found only "
            f"{len(causal_clean)} causal clean windows, while "
            f"cohort_screen_v2 audited {audited_after_retention} "
            f"remaining even after retention exclusion."
        )

    # Check that every warm-up event actually has labeled preictal
    # windows in the matching window manifest.
    for warmup_event_id in warmup["event_id"].astype(str):
        n_event_pre = int(
            (
                (pw["zone"] == "preictal")
                & (pw["event_id"] == warmup_event_id)
            ).sum()
        )
        if n_event_pre == 0:
            raise RuntimeError(
                f"{patient}/{warmup_event_id}: no matching preictal "
                "windows in the patient window manifest."
            )

    print(
        f"  timeline check PASS | causal clean={len(causal_clean):,} "
        f"| audited-after-retention>={audited_after_retention:,}"
    )

    # retention negatives
    ret_neg = select_nonoverlap_random(
        causal_clean,
        RET_NEG,
        SEED + p_idx * 10000 + 200,
    )
    ret_neg_int = intervals(ret_neg)

    # validation positives: latest warm-up event
    latest_eid = str(warmup.iloc[-1]["event_id"])
    val_pos_pool = pw[
        (pw["zone"] == "preictal") &
        (pw["event_id"] == latest_eid)
    ].copy()
    val_pos_pool = remove_overlap(val_pos_pool, ret_pos_int)
    val_pos = select_latest_rows(val_pos_pool, VAL_POS)
    val_pos_int = intervals(val_pos)

    # validation negatives: latest causal clean
    val_neg_pool = remove_overlap(causal_clean, ret_neg_int)
    val_neg = select_latest_rows(val_neg_pool, VAL_NEG)
    val_neg_int = intervals(val_neg)

    # initial training
    train_pos_all = pw[
        (pw["zone"] == "preictal") &
        (pw["event_id"].isin(warmup_ids))
    ].copy()

    train_pos = remove_overlap(
        train_pos_all,
        ret_pos_int + val_pos_int,
    )

    train_neg = remove_overlap(
        causal_clean,
        ret_neg_int + val_neg_int,
    )

    if train_pos.empty:
        raise RuntimeError(f"{patient}: no initial positive training rows.")

    print(
        f"  holdout audit | train_pos={len(train_pos):,} "
        f"| train_neg={len(train_neg):,} "
        f"| val_pos={len(val_pos):,} | val_neg={len(val_neg):,} "
        f"| ret_pos={len(ret_pos):,} | ret_neg={len(ret_neg):,}"
    )

    if len(train_neg) < MIN_TRAIN_NEG:
        raise RuntimeError(
            f"{patient}: only {len(train_neg)} initial negative training rows "
            f"after valid holdouts; expected at least {MIN_TRAIN_NEG}."
        )

    initial_train = tag(
        pd.concat([train_pos, train_neg], ignore_index=True),
        "initial_train",
        f"{patient}_INITIAL",
        0,
        train=True,
    )

    initial_val = tag(
        pd.concat([val_pos, val_neg], ignore_index=True),
        "initial_validation",
        f"{patient}_VALIDATION",
        -2,
        val=True,
    )

    retention = tag(
        pd.concat([ret_pos, ret_neg], ignore_index=True),
        "retention_eval",
        f"{patient}_RETENTION",
        -1,
        eval_=True,
    )

    parts.extend([initial_train, initial_val, retention])

    # strict future test-before-update
    previous = warmup.iloc[-1]
    patient_clean = []

    for ep_idx, (_, current) in enumerate(future.iterrows(), start=1):

        ep_id = f"{patient}_F{ep_idx:02d}"
        event_id = str(current["event_id"])
        block_start = float(previous["offset_abs_sec"])
        block_end = float(current["preictal_end_abs_sec"])

        episode = pw[
            (pw["start_abs_sec"] >= block_start) &
            (pw["end_abs_sec"] <= block_end) &
            (
                (pw["zone"] == "clean_interictal") |
                (
                    (pw["zone"] == "preictal") &
                    (pw["event_id"] == event_id)
                )
            )
        ].copy()

        episode = tag(
            episode,
            "future_episode",
            ep_id,
            ep_idx,
            eval_=True,
        )

        n_pos = int((episode["label"] == 1).sum())
        n_neg = int((episode["label"] == 0).sum())

        if n_pos == 0:
            raise RuntimeError(f"{ep_id}: no preictal evaluation rows.")

        positives = episode[episode["label"] == 1].copy()
        negatives = episode[episode["label"] == 0].copy()

        if len(negatives):
            n_take = min(len(positives), len(negatives))
            neg_update = sample_rows(
                negatives,
                n_take,
                SEED + p_idx * 10000 + 1000 + ep_idx,
            )
            update_rows = pd.concat([positives, neg_update], ignore_index=True)
        else:
            update_rows = positives.copy()

        update_keys = set(update_rows["window_key"])
        episode["use_for_update"] = episode["window_key"].isin(update_keys)

        clean = episode[episode["label"] == 0].copy()
        if not clean.empty:
            patient_clean.append(clean)

        future_rows.append({
            "patient": patient,
            "episode_id": ep_id,
            "episode_index": ep_idx,
            "event_id": event_id,
            "n_eval_windows": len(episode),
            "n_eval_preictal": n_pos,
            "n_eval_clean_interictal": n_neg,
            "clean_exposure_hours": exposure_hours(clean),
            "n_update_windows": int(episode["use_for_update"].sum()),
            "n_update_preictal": int(
                (episode["use_for_update"] & (episode["label"] == 1)).sum()
            ),
            "n_update_clean_interictal": int(
                (episode["use_for_update"] & (episode["label"] == 0)).sum()
            ),
            "positive_only_update": bool(n_neg == 0),
        })

        parts.append(episode)
        previous = current

    future_clean_hours = (
        exposure_hours(pd.concat(patient_clean, ignore_index=True))
        if patient_clean else 0.0
    )

    patient_rows.append({
        "patient": patient,
        "eligible_events": len(pe),
        "warmup_events": k,
        "future_events": len(future),
        "initial_train_positive": int((initial_train["label"] == 1).sum()),
        "initial_train_negative": int((initial_train["label"] == 0).sum()),
        "validation_positive": int((initial_val["label"] == 1).sum()),
        "validation_negative": int((initial_val["label"] == 0).sum()),
        "validation_positive_exposure_min": (
            exposure_hours(initial_val[initial_val["label"] == 1]) * 60.0
        ),
        "validation_negative_exposure_min": (
            exposure_hours(initial_val[initial_val["label"] == 0]) * 60.0
        ),
        "retention_positive": int((retention["label"] == 1).sum()),
        "retention_negative": int((retention["label"] == 0).sum()),
        "future_clean_exposure_hours": future_clean_hours,
    })

    print(
        f"warm-up={k} | future={len(future)} | "
        f"train +/−={int((initial_train.label==1).sum())}/"
        f"{int((initial_train.label==0).sum())} | "
        f"future clean={future_clean_hours:.3f} h"
    )


# -------------------- combine + integrity --------------------

split_manifest = pd.concat(parts, ignore_index=True)
patient_summary = pd.DataFrame(patient_rows)
future_summary = pd.DataFrame(future_rows)
event_roles = pd.DataFrame(event_rows)

for patient in PATIENTS:
    for role in ("initial_train", "initial_validation", "retention_eval"):
        g = split_manifest[
            (split_manifest["patient"] == patient) &
            (split_manifest["split_role"] == role)
        ]
        if set(g["label"].unique()) != {0, 1}:
            raise RuntimeError(f"{patient}/{role}: both classes not present.")

    val = split_manifest[
        (split_manifest["patient"] == patient) &
        (split_manifest["split_role"] == "initial_validation")
    ]
    ret = split_manifest[
        (split_manifest["patient"] == patient) &
        (split_manifest["split_role"] == "retention_eval")
    ]

    if val["use_for_training"].any() or val["use_for_update"].any():
        raise RuntimeError(f"{patient}: validation leakage.")
    if ret["use_for_training"].any() or ret["use_for_update"].any():
        raise RuntimeError(f"{patient}: retention leakage.")

role_counts = split_manifest.groupby("window_key")["split_role"].nunique()
if (role_counts > 1).any():
    raise RuntimeError(
        f"Leakage: {(role_counts > 1).sum()} physical windows "
        "occur in multiple split roles."
    )

future_counts = (
    future_summary.groupby("patient")["episode_id"].nunique().to_dict()
)

if future_counts != FUTURE_EXPECTED:
    raise RuntimeError(f"Future counts changed: {future_counts}")


# -------------------- verify every split window in cache --------------------

cache_rows = []

for (patient, fname), g in split_manifest.groupby(["patient", "file"]):

    path = CACHE_ROOT / patient / f"{Path(fname).stem}.npz"

    if not path.exists():
        raise RuntimeError(f"Missing cache: {path}")

    idx = g["window_index_in_file"].astype(int).to_numpy()

    with np.load(path, allow_pickle=False) as z:
        cache_hash = str(z["preprocess_hash"].item())
        n_cache = int(z["node_features"].shape[0])

        if cache_hash != PREPROCESS_HASH:
            raise RuntimeError(f"{patient}/{fname}: preprocessing hash mismatch.")

        if idx.min() < 0 or idx.max() >= n_cache:
            raise RuntimeError(f"{patient}/{fname}: invalid cache index.")

        qc = z["hard_qc_pass"][idx]

        if not qc.all():
            raise RuntimeError(f"{patient}/{fname}: hard-QC failure in final split.")

    cache_rows.append({
        "patient": patient,
        "file": fname,
        "split_windows": len(g),
        "cache_windows": n_cache,
        "qc_failures": int((~qc).sum()),
    })

cache_audit = pd.DataFrame(cache_rows)


# -------------------- freeze hash + save --------------------

hash_cols = [
    "window_key",
    "split_role",
    "episode_id",
    "episode_index",
    "label",
    "use_for_training",
    "use_for_validation",
    "use_for_evaluation",
    "use_for_update",
]

FINAL_SPLIT_HASH = hashlib.sha256(
    split_manifest[hash_cols]
    .sort_values(["window_key", "episode_id"])
    .to_csv(index=False)
    .encode("utf-8")
).hexdigest()

split_manifest.to_pickle(OUT_ROOT / "final_split_manifest.pkl")
split_manifest.to_csv(OUT_ROOT / "final_split_manifest.csv", index=False)
patient_summary.to_csv(OUT_ROOT / "patient_summary.csv", index=False)
future_summary.to_csv(OUT_ROOT / "future_episode_summary.csv", index=False)
event_roles.to_csv(OUT_ROOT / "event_roles.csv", index=False)
cache_audit.to_csv(OUT_ROOT / "cache_audit.csv", index=False)

config = {
    "patients": list(PATIENTS),
    "cohort_screen_hash": SCREEN_HASH,
    "preprocess_hash": PREPROCESS_HASH,
    "event_timeline_sources": {
        "chb06": "protocol_v2/seizure_manifest_v2.csv",
        "chb09": "protocol_extension_chb09_chb10_v1/events.csv",
        "chb10": "protocol_extension_chb09_chb10_v1/events.csv",
    },
    "warmup_events": WARMUP_EXPECTED,
    "future_events": FUTURE_EXPECTED,
    "retention": (
        "30 non-overlapping preictal windows from each of first "
        "2 eligible seizures + 60 non-overlapping causal clean negatives"
    ),
    "validation": (
        "60 latest preictal windows from the latest warm-up seizure + "
        "60 latest causal clean-interictal windows; every training "
        "window overlapping validation is excluded"
    ),
    "normalization": "fit on initial_train only",
    "future_protocol": "strict test-before-update",
    "future_update": (
        "all newly revealed preictal windows plus up to equal number of "
        "new clean negatives; positive-only update allowed when no new "
        "clean negative exposure exists"
    ),
    "seed": SEED,
    "final_split_hash": FINAL_SPLIT_HASH,
}

(OUT_ROOT / "final_split_config.json").write_text(
    json.dumps(config, indent=2)
)


# -------------------- report --------------------

print("\n" + "=" * 90)
print("FINAL CHRONOLOGICAL SPLIT V3 FROZEN")
print("=" * 90)
print("Split hash:", FINAL_SPLIT_HASH)

print("\nPATIENT SUMMARY")
display(patient_summary)

print("\nFUTURE EPISODES")
display(future_summary)

print("\nTOTAL FUTURE SEIZURES:", len(future_summary))
print(
    "TOTAL FUTURE CLEAN EXPOSURE:",
    f"{patient_summary['future_clean_exposure_hours'].sum():.3f} h"
)

print("\nPOSITIVE-ONLY UPDATE EPISODES")
display(
    future_summary[
        future_summary["positive_only_update"]
    ]
)

print("\nCACHE AUDIT")
display(
    cache_audit.groupby("patient", as_index=False).agg(
        files=("file", "count"),
        split_windows=("split_windows", "sum"),
        qc_failures=("qc_failures", "sum"),
    )
)

print("\nSaved to:", OUT_ROOT)
print("\nSTAGE 6 COMPLETE")
print(
    "Next: load cached tensors, fit initial-train-only normalization, "
    "define the fixed graph model, and run the four methods."
)


In [ ]:
# ============================================================
# Load final tensors, fit train-only normalization, and define GNN
# ============================================================

from pathlib import Path
import json
import random
import hashlib

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# 1. PATHS + REPRODUCIBILITY
# ============================================================

ROOT = Path(
    "/content/drive/MyDrive/EEG_Research/"
    "continual_graph_forecasting"
)

SPLIT_ROOT = ROOT / "final_split_v3"
CACHE_ROOT = ROOT / "graph_cache_v1"

PATIENTS = (
    "chb06",
    "chb09",
    "chb10",
)

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)


# ============================================================
# 2. LOAD FROZEN SPLIT
# ============================================================

split_path = (
    SPLIT_ROOT
    / "final_split_manifest.pkl"
)

config_path = (
    SPLIT_ROOT
    / "final_split_config.json"
)

if not split_path.exists():
    raise FileNotFoundError(
        f"Missing final split: {split_path}"
    )

if not config_path.exists():
    raise FileNotFoundError(
        f"Missing split config: {config_path}"
    )


FINAL_SPLIT = pd.read_pickle(
    split_path
)

FINAL_CONFIG = json.loads(
    config_path.read_text()
)


required_split_columns = {
    "patient",
    "file",
    "window_index_in_file",
    "window_key",
    "label",
    "split_role",
    "episode_id",
    "episode_index",
    "event_id",
    "start_abs_sec",
    "use_for_training",
    "use_for_validation",
    "use_for_evaluation",
    "use_for_update",
}

missing_columns = (
    required_split_columns
    -
    set(FINAL_SPLIT.columns)
)

if missing_columns:
    raise RuntimeError(
        "Final split is missing columns: "
        f"{sorted(missing_columns)}"
    )


if set(
    FINAL_SPLIT["patient"].unique()
) != set(PATIENTS):
    raise RuntimeError(
        "Unexpected patient set in final split: "
        f"{sorted(FINAL_SPLIT['patient'].unique())}"
    )


if FINAL_SPLIT[
    "window_key"
].duplicated().any():
    raise RuntimeError(
        "Duplicate physical windows found "
        "in final split."
    )


FINAL_SPLIT = (
    FINAL_SPLIT
    .reset_index(
        drop=True
    )
    .copy()
)

FINAL_SPLIT[
    "row_id"
] = np.arange(
    len(FINAL_SPLIT),
    dtype=np.int64
)


EXPECTED_SPLIT_HASH = (
    FINAL_CONFIG[
        "final_split_hash"
    ]
)

EXPECTED_PREPROCESS_HASH = (
    FINAL_CONFIG[
        "preprocess_hash"
    ]
)


# Recompute the exact frozen split hash before loading tensors.
hash_columns = [
    "window_key",
    "split_role",
    "episode_id",
    "episode_index",
    "label",
    "use_for_training",
    "use_for_validation",
    "use_for_evaluation",
    "use_for_update",
]

actual_split_hash = hashlib.sha256(
    FINAL_SPLIT[
        hash_columns
    ]
    .sort_values(
        [
            "window_key",
            "episode_id"
        ]
    )
    .to_csv(
        index=False
    )
    .encode(
        "utf-8"
    )
).hexdigest()


if actual_split_hash != EXPECTED_SPLIT_HASH:
    raise RuntimeError(
        "Final split hash mismatch. "
        f"Expected {EXPECTED_SPLIT_HASH}, "
        f"found {actual_split_hash}."
    )


print(
    "Final split hash:",
    EXPECTED_SPLIT_HASH
)

print(
    "Preprocess hash:",
    EXPECTED_PREPROCESS_HASH
)

print(
    "Split rows:",
    f"{len(FINAL_SPLIT):,}"
)


# ============================================================
# 3. LOAD ONLY SPLIT WINDOWS FROM CACHE
# ============================================================

N = len(
    FINAL_SPLIT
)

NODE_FEATURES = np.empty(
    (
        N,
        22,
        5
    ),
    dtype=np.float32
)

CONNECTIVITY = np.empty(
    (
        N,
        231
    ),
    dtype=np.float32
)

loaded = np.zeros(
    N,
    dtype=bool
)


for (
    patient,
    fname
), group in FINAL_SPLIT.groupby(
    [
        "patient",
        "file"
    ],
    sort=False
):

    cache_path = (
        CACHE_ROOT
        / str(patient)
        / f"{Path(str(fname)).stem}.npz"
    )


    if not cache_path.exists():
        raise FileNotFoundError(
            f"Missing cache: {cache_path}"
        )


    split_rows = (
        group[
            "row_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    cache_indices = (
        group[
            "window_index_in_file"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    with np.load(
        cache_path,
        allow_pickle=False
    ) as z:

        cache_hash = str(
            z[
                "preprocess_hash"
            ].item()
        )


        if (
            cache_hash
            != EXPECTED_PREPROCESS_HASH
        ):
            raise RuntimeError(
                f"{patient}/{fname}: "
                "preprocessing hash mismatch."
            )


        n_cache = int(
            z[
                "node_features"
            ].shape[0]
        )


        if (
            cache_indices.min()
            < 0
            or
            cache_indices.max()
            >= n_cache
        ):
            raise RuntimeError(
                f"{patient}/{fname}: "
                "cache index outside range."
            )


        qc = z[
            "hard_qc_pass"
        ][
            cache_indices
        ]


        if not bool(
            np.all(qc)
        ):
            raise RuntimeError(
                f"{patient}/{fname}: "
                "hard-QC failure in selected rows."
            )


        node = z[
            "node_features"
        ][
            cache_indices
        ].astype(
            np.float32,
            copy=False
        )


        conn = z[
            "connectivity"
        ][
            cache_indices
        ].astype(
            np.float32,
            copy=False
        )


    if node.shape != (
        len(group),
        22,
        5
    ):
        raise RuntimeError(
            f"{patient}/{fname}: "
            f"node shape={node.shape}"
        )


    if conn.shape != (
        len(group),
        231
    ):
        raise RuntimeError(
            f"{patient}/{fname}: "
            f"connectivity shape={conn.shape}"
        )


    NODE_FEATURES[
        split_rows
    ] = node

    CONNECTIVITY[
        split_rows
    ] = conn

    loaded[
        split_rows
    ] = True


if not loaded.all():
    raise RuntimeError(
        f"{int((~loaded).sum())} "
        "split rows were not loaded."
    )


if not np.isfinite(
    NODE_FEATURES
).all():
    raise RuntimeError(
        "Non-finite node features."
    )


if not np.isfinite(
    CONNECTIVITY
).all():
    raise RuntimeError(
        "Non-finite connectivity values."
    )


if (
    np.max(
        np.abs(
            CONNECTIVITY
        )
    )
    > 1.00001
):
    raise RuntimeError(
        "Connectivity outside Pearson "
        "correlation range [-1, 1]."
    )


print(
    "\nCache loading: PASS"
)

print(
    "Node feature tensor:",
    NODE_FEATURES.shape
)

print(
    "Connectivity tensor:",
    CONNECTIVITY.shape
)


# ============================================================
# 4. FIT NORMALIZATION — INITIAL TRAIN ONLY, PER PATIENT
#
# Mean/std are computed per frequency-band feature using
# all nodes and windows in that patient's initial training set.
#
# Connectivity remains in its natural [-1, 1] scale.
# ============================================================

NORMALIZATION = {}

NODE_FEATURES_Z = np.empty_like(
    NODE_FEATURES,
    dtype=np.float32
)

POS_WEIGHT = {}


for patient in PATIENTS:

    patient_mask = (
        FINAL_SPLIT[
            "patient"
        ].to_numpy()
        == patient
    )

    train_mask = (
        patient_mask
        &
        FINAL_SPLIT[
            "use_for_training"
        ].to_numpy(
            dtype=bool
        )
    )


    train_idx = np.flatnonzero(
        train_mask
    )


    if len(
        train_idx
    ) == 0:
        raise RuntimeError(
            f"{patient}: no initial "
            "training rows."
        )


    train_x = NODE_FEATURES[
        train_idx
    ]


    mean = train_x.mean(
        axis=(0, 1),
        dtype=np.float64
    ).astype(
        np.float32
    )


    std = train_x.std(
        axis=(0, 1),
        dtype=np.float64
    ).astype(
        np.float32
    )


    std = np.maximum(
        std,
        np.float32(
            1e-6
        )
    )


    NORMALIZATION[
        patient
    ] = {
        "mean":
            mean.copy(),

        "std":
            std.copy(),
    }


    patient_idx = np.flatnonzero(
        patient_mask
    )


    NODE_FEATURES_Z[
        patient_idx
    ] = (
        (
            NODE_FEATURES[
                patient_idx
            ]
            -
            mean[
                None,
                None,
                :
            ]
        )
        /
        std[
            None,
            None,
            :
        ]
    ).astype(
        np.float32
    )


    y_train = (
        FINAL_SPLIT.iloc[
            train_idx
        ][
            "label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    n_pos = int(
        (
            y_train
            == 1
        ).sum()
    )

    n_neg = int(
        (
            y_train
            == 0
        ).sum()
    )


    if (
        n_pos == 0
        or
        n_neg == 0
    ):
        raise RuntimeError(
            f"{patient}: initial training "
            "does not contain both classes."
        )


    POS_WEIGHT[
        patient
    ] = float(
        n_neg
        /
        n_pos
    )


    print(
        f"\n{patient}: "
        f"train={len(train_idx):,} | "
        f"positive={n_pos:,} | "
        f"negative={n_neg:,} | "
        f"pos_weight={POS_WEIGHT[patient]:.3f}"
    )

    print(
        "  feature mean:",
        np.round(
            mean,
            4
        )
    )

    print(
        "  feature std :",
        np.round(
            std,
            4
        )
    )


if not np.isfinite(
    NODE_FEATURES_Z
).all():
    raise RuntimeError(
        "Normalization created "
        "non-finite values."
    )


# ============================================================
# 5. BUILD PATIENT-SPECIFIC INDEX MAPS
# ============================================================

PATIENT_DATA = {}


for patient in PATIENTS:

    p = FINAL_SPLIT[
        FINAL_SPLIT[
            "patient"
        ]
        == patient
    ]


    train_idx = (
        p[
            p[
                "use_for_training"
            ]
        ][
            "row_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    val_idx = (
        p[
            p[
                "use_for_validation"
            ]
        ][
            "row_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    retention_idx = (
        p[
            p[
                "split_role"
            ]
            == "retention_eval"
        ][
            "row_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    future_df = (
        p[
            p[
                "split_role"
            ]
            == "future_episode"
        ]
        .sort_values(
            [
                "episode_index",
                "start_abs_sec",
                "file",
                "window_index_in_file"
            ]
        )
    )


    episodes = []


    for (
        episode_id,
        episode_index
    ), ep in future_df.groupby(
        [
            "episode_id",
            "episode_index"
        ],
        sort=True
    ):

        eval_idx = (
            ep[
                ep[
                    "use_for_evaluation"
                ]
            ][
                "row_id"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )


        update_idx = (
            ep[
                ep[
                    "use_for_update"
                ]
            ][
                "row_id"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )


        episodes.append(
            {
                "episode_id":
                    str(
                        episode_id
                    ),

                "episode_index":
                    int(
                        episode_index
                    ),

                "event_id":
                    str(
                        ep[
                            "event_id"
                        ].iloc[0]
                    ),

                "eval_idx":
                    eval_idx,

                "update_idx":
                    update_idx,
            }
        )


    PATIENT_DATA[
        patient
    ] = {
        "train_idx":
            train_idx,

        "val_idx":
            val_idx,

        "retention_idx":
            retention_idx,

        "episodes":
            episodes,
    }


    if len(
        episodes
    ) == 0:
        raise RuntimeError(
            f"{patient}: no future episodes."
        )


    print(
        f"\n{patient}: "
        f"train={len(train_idx):,} | "
        f"validation={len(val_idx):,} | "
        f"retention={len(retention_idx):,} | "
        f"future episodes={len(episodes)}"
    )


# ============================================================
# 6. FIXED MODEL
#
# Signed Pearson edges are preserved.
#
# We use absolute edge magnitude only for degree
# normalization, while the signed edge value itself is
# retained in message passing.
# ============================================================

MODEL_CONFIG = {
    "n_nodes": 22,
    "node_features": 5,
    "hidden_dim": 32,
    "graph_layers": 2,
    "dropout": 0.20,
    "connectivity": (
        "signed Pearson correlation"
    ),
    "degree_normalization": (
        "absolute weighted degree"
    ),
    "graph_pooling": "mean",
}


class SignedGraphConv(
    nn.Module
):

    def __init__(
        self,
        in_dim,
        out_dim,
        dropout=0.0
    ):

        super().__init__()

        self.self_linear = nn.Linear(
            in_dim,
            out_dim
        )

        self.neigh_linear = nn.Linear(
            in_dim,
            out_dim,
            bias=False
        )

        self.norm = nn.LayerNorm(
            out_dim
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x,
        adjacency
    ):

        message = torch.bmm(
            adjacency,
            x
        )

        h = (
            self.self_linear(
                x
            )
            +
            self.neigh_linear(
                message
            )
        )

        h = self.norm(
            h
        )

        h = F.gelu(
            h
        )

        h = self.dropout(
            h
        )

        return h


class SignedConnectivityGNN(
    nn.Module
):

    def __init__(
        self,
        n_nodes=22,
        node_features=5,
        hidden_dim=32,
        dropout=0.20
    ):

        super().__init__()

        self.n_nodes = int(
            n_nodes
        )

        tri_i, tri_j = np.triu_indices(
            self.n_nodes,
            k=1
        )

        self.register_buffer(
            "tri_i",
            torch.tensor(
                tri_i,
                dtype=torch.long
            )
        )

        self.register_buffer(
            "tri_j",
            torch.tensor(
                tri_j,
                dtype=torch.long
            )
        )


        self.gconv1 = SignedGraphConv(
            node_features,
            hidden_dim,
            dropout=dropout
        )

        self.gconv2 = SignedGraphConv(
            hidden_dim,
            hidden_dim,
            dropout=dropout
        )


        self.head = nn.Sequential(

            nn.Linear(
                hidden_dim,
                hidden_dim
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_dim,
                1
            )
        )


    def build_adjacency(
        self,
        edge_values
    ):

        batch_size = int(
            edge_values.shape[0]
        )

        adjacency = torch.zeros(
            (
                batch_size,
                self.n_nodes,
                self.n_nodes
            ),
            dtype=edge_values.dtype,
            device=edge_values.device
        )


        adjacency[
            :,
            self.tri_i,
            self.tri_j
        ] = edge_values


        adjacency[
            :,
            self.tri_j,
            self.tri_i
        ] = edge_values


        # Self loops
        eye = torch.eye(
            self.n_nodes,
            dtype=edge_values.dtype,
            device=edge_values.device
        ).unsqueeze(0)


        adjacency = (
            adjacency
            +
            eye
        )


        # Stable normalization for signed graphs:
        # degree uses absolute edge mass,
        # while signed edges remain signed.
        degree = adjacency.abs().sum(
            dim=-1
        )


        degree_inv_sqrt = torch.rsqrt(
            degree.clamp_min(
                1e-6
            )
        )


        adjacency = (
            degree_inv_sqrt.unsqueeze(
                -1
            )
            *
            adjacency
            *
            degree_inv_sqrt.unsqueeze(
                -2
            )
        )


        return adjacency


    def forward(
        self,
        node_features,
        connectivity
    ):

        adjacency = self.build_adjacency(
            connectivity
        )


        h = self.gconv1(
            node_features,
            adjacency
        )


        h = self.gconv2(
            h,
            adjacency
        )


        graph_embedding = h.mean(
            dim=1
        )


        logits = self.head(
            graph_embedding
        ).squeeze(
            -1
        )


        return logits


# ============================================================
# 7. SMOKE TEST
# ============================================================

model = SignedConnectivityGNN(
    n_nodes=MODEL_CONFIG[
        "n_nodes"
    ],
    node_features=MODEL_CONFIG[
        "node_features"
    ],
    hidden_dim=MODEL_CONFIG[
        "hidden_dim"
    ],
    dropout=MODEL_CONFIG[
        "dropout"
    ],
).to(
    DEVICE
)


smoke_idx = np.concatenate(
    [
        PATIENT_DATA[
            patient
        ][
            "train_idx"
        ][
            :2
        ]

        for patient in PATIENTS
    ]
)


x_smoke = torch.from_numpy(
    NODE_FEATURES_Z[
        smoke_idx
    ]
).to(
    DEVICE
)


c_smoke = torch.from_numpy(
    CONNECTIVITY[
        smoke_idx
    ]
).to(
    DEVICE
)


with torch.no_grad():

    logits = model(
        x_smoke,
        c_smoke
    )


if logits.shape != (
    len(
        smoke_idx
    ),
):
    raise RuntimeError(
        f"Unexpected model output shape: "
        f"{tuple(logits.shape)}"
    )


if not torch.isfinite(
    logits
).all():
    raise RuntimeError(
        "Model smoke test produced "
        "non-finite logits."
    )


n_parameters = sum(
    p.numel()
    for p in model.parameters()
)


print(
    "\n"
    + "=" * 80
)

print(
    "STAGE 7 COMPLETE"
)

print(
    "=" * 80
)

print(
    "Model:",
    model.__class__.__name__
)

print(
    "Trainable parameters:",
    f"{n_parameters:,}"
)

print(
    "Smoke-test output shape:",
    tuple(
        logits.shape
    )
)

print(
    "Smoke-test logits:",
    np.round(
        logits.detach()
        .cpu()
        .numpy(),
        4
    )
)

print(
    "\nNormalization leakage check: PASS"
)

print(
    "Cache/hash/QC checks: PASS"
)

print(
    "\nNext: train the initial patient-specific models "
    "and run Static / Fine-tune / Random Replay / "
    "Connectivity-Diverse Replay with identical budgets."
)


In [ ]:
# ============================================================
# Initial training, evaluation, and replay-selection helpers
# ============================================================

import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
)

INITIAL_MAX_EPOCHS = 20
INITIAL_MIN_EPOCHS = 5
EARLY_STOP_PATIENCE = 4
INITIAL_BATCH_SIZE = 512
INITIAL_LR = 1e-3

WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 5.0
PREDICT_BATCH_SIZE = 1024

REPLAY_MEMORY_TOTAL = 128
REPLAY_PER_CLASS = 64

def set_all_seeds(seed):

    random.seed(
        int(seed)
    )

    np.random.seed(
        int(seed)
    )

    torch.manual_seed(
        int(seed)
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            int(seed)
        )


def make_model():

    return SignedConnectivityGNN(
        n_nodes=MODEL_CONFIG[
            "n_nodes"
        ],
        node_features=MODEL_CONFIG[
            "node_features"
        ],
        hidden_dim=MODEL_CONFIG[
            "hidden_dim"
        ],
        dropout=MODEL_CONFIG[
            "dropout"
        ],
    ).to(
        DEVICE
    )


def criterion_for_patient(
    patient
):

    return nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            float(
                POS_WEIGHT[
                    patient
                ]
            ),
            dtype=torch.float32,
            device=DEVICE,
        )
    )


def batch_arrays(
    indices
):

    indices = np.asarray(
        indices,
        dtype=np.int64
    )

    x = torch.from_numpy(
        NODE_FEATURES_Z[
            indices
        ]
    ).to(
        DEVICE,
        non_blocking=True
    )

    c = torch.from_numpy(
        CONNECTIVITY[
            indices
        ]
    ).to(
        DEVICE,
        non_blocking=True
    )

    y = torch.from_numpy(
        FINAL_SPLIT.iloc[
            indices
        ][
            "label"
        ]
        .to_numpy(
            dtype=np.float32
        )
    ).to(
        DEVICE,
        non_blocking=True
    )

    return (
        x,
        c,
        y
    )


def predict_indices(
    model,
    indices,
    batch_size=PREDICT_BATCH_SIZE
):

    indices = np.asarray(
        indices,
        dtype=np.int64
    )

    model.eval()

    probs = np.empty(
        len(indices),
        dtype=np.float32
    )

    cursor = 0

    for start in range(
        0,
        len(indices),
        batch_size
    ):

        stop = min(
            start + batch_size,
            len(indices)
        )

        batch_idx = indices[
            start:stop
        ]

        x = torch.from_numpy(
            NODE_FEATURES_Z[
                batch_idx
            ]
        ).to(
            DEVICE,
            non_blocking=True
        )

        c = torch.from_numpy(
            CONNECTIVITY[
                batch_idx
            ]
        ).to(
            DEVICE,
            non_blocking=True
        )

        logits = model(
            x,
            c
        )

        batch_probs = torch.sigmoid(
            logits
        ).detach().cpu().numpy().astype(
            np.float32
        )

        probs[
            cursor:
            cursor + len(batch_idx)
        ] = batch_probs

        cursor += len(
            batch_idx
        )

    return probs


def safe_average_precision(
    y_true,
    probs
):

    y_true = np.asarray(
        y_true,
        dtype=np.int64
    )

    probs = np.asarray(
        probs,
        dtype=np.float64
    )

    if np.unique(
        y_true
    ).size < 2:
        return np.nan

    return float(
        average_precision_score(
            y_true,
            probs
        )
    )


def safe_brier(
    y_true,
    probs
):

    y_true = np.asarray(
        y_true,
        dtype=np.int64
    )

    probs = np.asarray(
        probs,
        dtype=np.float64
    )

    return float(
        brier_score_loss(
            y_true,
            probs
        )
    )


def choose_f1_threshold(
    y_true,
    probs
):

    y_true = np.asarray(
        y_true,
        dtype=np.int64
    )

    probs = np.asarray(
        probs,
        dtype=np.float64
    )

    if set(
        np.unique(
            y_true
        )
    ) != {
        0,
        1
    }:
        raise RuntimeError(
            "Validation threshold selection "
            "requires both classes."
        )

    precision, recall, thresholds = (
        precision_recall_curve(
            y_true,
            probs
        )
    )

    if len(
        thresholds
    ) == 0:
        return 0.5

    precision_t = precision[
        :-1
    ]

    recall_t = recall[
        :-1
    ]

    denom = (
        precision_t
        +
        recall_t
    )

    f1 = np.divide(
        2.0
        * precision_t
        * recall_t,
        denom,
        out=np.zeros_like(
            denom,
            dtype=np.float64
        ),
        where=denom > 0,
    )

    best = int(
        np.nanargmax(
            f1
        )
    )

    return float(
        thresholds[
            best
        ]
    )


def evaluate_set(
    model,
    indices,
    threshold
):

    indices = np.asarray(
        indices,
        dtype=np.int64
    )

    probs = predict_indices(
        model,
        indices
    )

    y = (
        FINAL_SPLIT.iloc[
            indices
        ][
            "label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    pred = (
        probs
        >= float(
            threshold
        )
    ).astype(
        np.int8
    )

    positive_mask = (
        y == 1
    )

    negative_mask = (
        y == 0
    )

    positive_recall = (
        float(
            pred[
                positive_mask
            ].mean()
        )
        if positive_mask.any()
        else np.nan
    )

    negative_fp_rate = (
        float(
            pred[
                negative_mask
            ].mean()
        )
        if negative_mask.any()
        else np.nan
    )

    event_detected = (
        bool(
            pred[
                positive_mask
            ].any()
        )
        if positive_mask.any()
        else False
    )

    return {
        "probabilities":
            probs,

        "labels":
            y,

        "average_precision":
            safe_average_precision(
                y,
                probs
            ),

        "brier":
            safe_brier(
                y,
                probs
            ),

        "positive_window_recall":
            positive_recall,

        "clean_window_fp_rate":
            negative_fp_rate,

        "event_detected":
            event_detected,
    }


def train_initial_model(
    patient,
    seed
):

    set_all_seeds(
        seed
    )

    model = make_model()

    criterion = criterion_for_patient(
        patient
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=INITIAL_LR,
        weight_decay=WEIGHT_DECAY,
    )

    train_idx = (
        PATIENT_DATA[
            patient
        ][
            "train_idx"
        ]
    )

    val_idx = (
        PATIENT_DATA[
            patient
        ][
            "val_idx"
        ]
    )

    rng = np.random.default_rng(
        seed
    )

    best_ap = -np.inf

    best_epoch = None

    best_state = None

    epochs_without_improvement = 0

    epoch_rows = []


    for epoch in range(
        1,
        INITIAL_MAX_EPOCHS + 1
    ):

        model.train()

        permutation = rng.permutation(
            train_idx
        )

        running_loss = 0.0

        n_seen = 0


        for start in range(
            0,
            len(permutation),
            INITIAL_BATCH_SIZE
        ):

            batch_idx = permutation[
                start:
                start + INITIAL_BATCH_SIZE
            ]

            x, c, y = batch_arrays(
                batch_idx
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(
                x,
                c
            )

            loss = criterion(
                logits,
                y
            )

            if not torch.isfinite(
                loss
            ):
                raise RuntimeError(
                    f"{patient}: non-finite "
                    "initial-training loss."
                )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP_NORM
            )

            optimizer.step()

            running_loss += (
                float(
                    loss.detach().cpu()
                )
                * len(
                    batch_idx
                )
            )

            n_seen += len(
                batch_idx
            )


        train_loss = (
            running_loss
            /
            max(
                n_seen,
                1
            )
        )

        val_probs = predict_indices(
            model,
            val_idx
        )

        val_y = (
            FINAL_SPLIT.iloc[
                val_idx
            ][
                "label"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        val_ap = safe_average_precision(
            val_y,
            val_probs
        )

        epoch_rows.append(
            {
                "patient":
                    patient,

                "epoch":
                    epoch,

                "train_loss":
                    train_loss,

                "validation_ap":
                    val_ap,
            }
        )


        improved = (
            np.isfinite(
                val_ap
            )
            and
            val_ap
            >
            best_ap
            +
            1e-6
        )


        if improved:

            best_ap = float(
                val_ap
            )

            best_epoch = int(
                epoch
            )

            best_state = {
                key:
                    value.detach()
                    .cpu()
                    .clone()

                for key, value
                in model.state_dict().items()
            }

            epochs_without_improvement = 0

        else:

            epochs_without_improvement += 1


        print(
            f"    epoch {epoch:02d} | "
            f"loss={train_loss:.4f} | "
            f"val_AP={val_ap:.4f}"
        )


        if (
            epoch >= INITIAL_MIN_EPOCHS
            and
            epochs_without_improvement
            >= EARLY_STOP_PATIENCE
        ):
            break


    if best_state is None:
        raise RuntimeError(
            f"{patient}: initial model "
            "never produced a valid checkpoint."
        )


    model.load_state_dict(
        best_state
    )


    val_probs = predict_indices(
        model,
        val_idx
    )

    val_y = (
        FINAL_SPLIT.iloc[
            val_idx
        ][
            "label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    threshold = choose_f1_threshold(
        val_y,
        val_probs
    )


    return (
        model,
        float(
            threshold
        ),
        float(
            best_ap
        ),
        int(
            best_epoch
        ),
        epoch_rows,
    )


def classwise_random_memory(
    candidate_indices,
    seed
):

    candidate_indices = np.unique(
        np.asarray(
            candidate_indices,
            dtype=np.int64
        )
    )

    y = (
        FINAL_SPLIT.iloc[
            candidate_indices
        ][
            "label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    rng = np.random.default_rng(
        seed
    )

    selected = []


    for cls in (
        0,
        1
    ):

        cls_idx = candidate_indices[
            y == cls
        ]

        if len(
            cls_idx
        ) == 0:
            continue

        n_take = min(
            REPLAY_PER_CLASS,
            len(
                cls_idx
            )
        )

        chosen = rng.choice(
            cls_idx,
            size=n_take,
            replace=False,
        )

        selected.extend(
            chosen.tolist()
        )


    return np.asarray(
        sorted(
            selected
        ),
        dtype=np.int64
    )


def farthest_point_indices(
    candidate_indices,
    k
):

    candidate_indices = np.unique(
        np.asarray(
            candidate_indices,
            dtype=np.int64
        )
    )

    if len(
        candidate_indices
    ) <= k:
        return candidate_indices.copy()


    x = CONNECTIVITY[
        candidate_indices
    ].astype(
        np.float32,
        copy=False
    )


    centroid = x.mean(
        axis=0,
        dtype=np.float64
    ).astype(
        np.float32
    )


    dist_to_centroid = np.sum(
        (
            x
            -
            centroid[
                None,
                :
            ]
        )
        ** 2,
        axis=1,
        dtype=np.float64,
    )


    first = int(
        np.argmin(
            dist_to_centroid
        )
    )

    selected_local = [
        first
    ]


    min_dist = np.sum(
        (
            x
            -
            x[
                first:
                first + 1
            ]
        )
        ** 2,
        axis=1,
        dtype=np.float64,
    )


    min_dist[
        first
    ] = -np.inf


    while len(
        selected_local
    ) < k:

        next_local = int(
            np.argmax(
                min_dist
            )
        )

        selected_local.append(
            next_local
        )

        new_dist = np.sum(
            (
                x
                -
                x[
                    next_local:
                    next_local + 1
                ]
            )
            ** 2,
            axis=1,
            dtype=np.float64,
        )

        min_dist = np.minimum(
            min_dist,
            new_dist
        )

        min_dist[
            selected_local
        ] = -np.inf


    return candidate_indices[
        np.asarray(
            selected_local,
            dtype=np.int64
        )
    ]


def classwise_diverse_memory(
    candidate_indices
):

    candidate_indices = np.unique(
        np.asarray(
            candidate_indices,
            dtype=np.int64
        )
    )

    y = (
        FINAL_SPLIT.iloc[
            candidate_indices
        ][
            "label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    selected = []


    for cls in (
        0,
        1
    ):

        cls_idx = candidate_indices[
            y == cls
        ]

        if len(
            cls_idx
        ) == 0:
            continue

        chosen = farthest_point_indices(
            cls_idx,
            min(
                REPLAY_PER_CLASS,
                len(
                    cls_idx
                )
            )
        )

        selected.extend(
            chosen.tolist()
        )


    return np.asarray(
        sorted(
            selected
        ),
        dtype=np.int64
    )


def memory_class_counts(
    memory_idx
):

    memory_idx = np.asarray(
        memory_idx,
        dtype=np.int64
    )

    if len(
        memory_idx
    ) == 0:
        return (
            0,
            0
        )

    y = (
        FINAL_SPLIT.iloc[
            memory_idx
        ][
            "label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    return (
        int(
            (
                y == 0
            ).sum()
        ),
        int(
            (
                y == 1
            ).sum()
        ),
    )


def mean_pairwise_connectivity_distance(
    memory_idx
):

    memory_idx = np.asarray(
        memory_idx,
        dtype=np.int64
    )

    if len(
        memory_idx
    ) < 2:
        return np.nan


    x = CONNECTIVITY[
        memory_idx
    ].astype(
        np.float64,
        copy=False
    )


    sq_norm = np.sum(
        x * x,
        axis=1
    )

    d2 = (
        sq_norm[
            :,
            None
        ]
        +
        sq_norm[
            None,
            :
        ]
        -
        2.0
        *
        (
            x
            @
            x.T
        )
    )

    d2 = np.maximum(
        d2,
        0.0
    )

    tri = np.triu_indices(
        len(
            memory_idx
        ),
        k=1
    )

    return float(
        np.sqrt(
            d2[
                tri
            ]
        ).mean()
    )


In [ ]:
# ============================================================
# Continual-update rule used in the reported experiments
# ============================================================
# Initial training uses weighted BCE on the naturally imbalanced
# training set. Continual updates use unweighted BCE and balanced
# sampling whenever both classes are available.

UPDATE_STEPS_V2 = 40
UPDATE_BATCH_SIZE_V2 = 128
UPDATE_LR_V2 = 3e-4
WEIGHT_DECAY_V2 = 1e-4
GRAD_CLIP_NORM_V2 = 5.0
REPLAY_MEMORY_TOTAL_V2 = 128
REPLAY_PER_CLASS_V2 = 64

def set_seed_v2(seed):

    seed = int(
        seed
    )

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


def balanced_sample_from_pool(
    pool_idx,
    n,
    rng
):

    pool_idx = np.asarray(
        pool_idx,
        dtype=np.int64
    )

    if len(
        pool_idx
    ) == 0:

        return np.empty(
            0,
            dtype=np.int64
        )


    labels = (
        FINAL_SPLIT.iloc[
            pool_idx
        ][
            "label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    negative_idx = pool_idx[
        labels == 0
    ]

    positive_idx = pool_idx[
        labels == 1
    ]


    # --------------------------------------------------------
    # Both classes present
    # --------------------------------------------------------

    if (
        len(
            negative_idx
        ) > 0
        and
        len(
            positive_idx
        ) > 0
    ):

        n_positive = (
            n // 2
        )

        n_negative = (
            n
            -
            n_positive
        )


        sampled_negative = rng.choice(
            negative_idx,
            size=n_negative,
            replace=(
                len(
                    negative_idx
                )
                <
                n_negative
            ),
        )


        sampled_positive = rng.choice(
            positive_idx,
            size=n_positive,
            replace=(
                len(
                    positive_idx
                )
                <
                n_positive
            ),
        )


        sampled = np.concatenate(
            [
                sampled_negative,
                sampled_positive,
            ]
        )


        rng.shuffle(
            sampled
        )


        return sampled.astype(
            np.int64,
            copy=False
        )


    # --------------------------------------------------------
    # Only one class present
    # --------------------------------------------------------

    return rng.choice(
        pool_idx,
        size=n,
        replace=(
            len(
                pool_idx
            )
            <
            n
        ),
    ).astype(
        np.int64,
        copy=False
    )


def train_fixed_steps_v2(
    model,
    current_idx,
    memory_idx,
    seed
):

    current_idx = np.asarray(
        current_idx,
        dtype=np.int64
    )

    memory_idx = np.asarray(
        memory_idx,
        dtype=np.int64
    )


    if len(
        current_idx
    ) == 0:

        return


    set_seed_v2(
        seed
    )


    rng = np.random.default_rng(
        int(
            seed
        )
    )


    # --------------------------------------------------------
    # Update loss is intentionally UNWEIGHTED.
    # --------------------------------------------------------

    criterion = nn.BCEWithLogitsLoss()


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=UPDATE_LR_V2,
        weight_decay=WEIGHT_DECAY_V2,
    )


    for _ in range(
        UPDATE_STEPS_V2
    ):

        # ----------------------------------------------------
        # Replay methods:
        # 64 samples from current revealed data
        # +
        # 64 samples from replay memory.
        #
        # Both sides are class-balanced when both classes exist.
        # ----------------------------------------------------

        if len(
            memory_idx
        ) > 0:

            n_current = (
                UPDATE_BATCH_SIZE_V2
                // 2
            )

            n_memory = (
                UPDATE_BATCH_SIZE_V2
                -
                n_current
            )


            current_batch = (
                balanced_sample_from_pool(
                    current_idx,
                    n_current,
                    rng
                )
            )


            memory_batch = (
                balanced_sample_from_pool(
                    memory_idx,
                    n_memory,
                    rng
                )
            )


            batch_idx = np.concatenate(
                [
                    current_batch,
                    memory_batch,
                ]
            )


            rng.shuffle(
                batch_idx
            )


        # ----------------------------------------------------
        # Fine-tuning:
        # whole update mini-batch comes from current data.
        # Balanced if both classes exist.
        # Positive-only if the episode genuinely has no
        # available clean negative data.
        # ----------------------------------------------------

        else:

            batch_idx = (
                balanced_sample_from_pool(
                    current_idx,
                    UPDATE_BATCH_SIZE_V2,
                    rng
                )
            )


        x, c, y = batch_arrays(
            batch_idx
        )


        model.train()


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = model(
            x,
            c
        )


        loss = criterion(
            logits,
            y
        )


        if not torch.isfinite(
            loss
        ):

            raise RuntimeError(
                "Non-finite continual-update loss."
            )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP_NORM_V2
        )


        optimizer.step()


In [ ]:
# ============================================================
# Five-seed continual-learning and alarm-level evaluation
#
# Frozen protocol:
#   Seeds: 2026, 2027, 2028, 2029, 2030
#   Methods:
#     - static
#     - finetune
#     - random_replay
#     - connectivity_diverse_replay
#
# Continual-update rule:
#   - weighted BCE only for initial imbalanced training
#   - unweighted BCE for continual updates
#   - class-balanced update sampling when both classes exist
#
# Alarm conversion:
#   - threshold remains validation-only and frozen per seed/patient
#   - an above-threshold prediction may raise an alarm
#   - after an accepted alarm, suppress new alarms for
#       SPH + SOP = 5 + 30 = 35 minutes
#   - event is predicted if an accepted alarm occurs in its
#       frozen preictal interval
#   - false alarms/hour uses pooled clean-interictal exposure
#
# Hyperparameters are fixed before future evaluation.
# ============================================================

from pathlib import Path
import json
import random
import time

import numpy as np
import pandas as pd

import torch


# ============================================================
# 0. VERIFY REQUIRED OBJECTS
# ============================================================

required = [
    "FINAL_SPLIT",
    "NODE_FEATURES_Z",
    "CONNECTIVITY",
    "PATIENT_DATA",
    "PATIENTS",
    "DEVICE",
    "FINAL_CONFIG",
    "train_initial_model",
    "make_model",
    "evaluate_set",
    "train_fixed_steps_v2",
    "classwise_random_memory",
    "classwise_diverse_memory",
    "memory_class_counts",
    "mean_pairwise_connectivity_distance",
    "safe_average_precision",
    "safe_brier",
]

missing = [
    x
    for x in required
    if x not in globals()
]

if missing:

    raise RuntimeError(
        "Run the model and update-helper cells first. "
        f"Missing: {missing}"
    )


# ============================================================
# 1. FROZEN CONFIGURATION
# ============================================================

SEEDS = (
    2026,
    2027,
    2028,
    2029,
    2030,
)

METHODS_FINAL = (
    "static",
    "finetune",
    "random_replay",
    "connectivity_diverse_replay",
)

SPH_SEC = (
    5 * 60
)

SOP_SEC = (
    30 * 60
)

ALARM_REFRACTORY_SEC = (
    SPH_SEC
    +
    SOP_SEC
)

ROOT = Path(
    "/content/drive/MyDrive/"
    "EEG_Research/"
    "continual_graph_forecasting"
)

STAGE9_ROOT = (
    ROOT
    / "experiments"
    / "continual_multiseed_final_v1"
)

STAGE9_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

def seed_all(
    seed
):

    seed = int(
        seed
    )

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


# ============================================================
# 3. ALARM / EXPOSURE HELPERS
# ============================================================

def merged_intervals(
    df
):

    out = []

    if df.empty:

        return out


    for patient, g in df.groupby(
        "patient",
        sort=False
    ):

        intervals = sorted(
            zip(
                g[
                    "start_abs_sec"
                ].astype(
                    float
                ),

                g[
                    "end_abs_sec"
                ].astype(
                    float
                )
            )
        )


        start_current = None

        end_current = None


        for start, end in intervals:

            if start_current is None:

                start_current = start

                end_current = end


            elif (
                start
                <=
                end_current
            ):

                end_current = max(
                    end_current,
                    end
                )


            else:

                out.append(
                    (
                        str(
                            patient
                        ),

                        float(
                            start_current
                        ),

                        float(
                            end_current
                        ),
                    )
                )

                start_current = start

                end_current = end


        if start_current is not None:

            out.append(
                (
                    str(
                        patient
                    ),

                    float(
                        start_current
                    ),

                    float(
                        end_current
                    ),
                )
            )


    return out


def interval_hours(
    intervals
):

    total_seconds = sum(

        end - start

        for _,
        start,
        end
        in intervals
    )


    return float(
        total_seconds
        /
        3600.0
    )


def overlap_seconds(
    start,
    end,
    intervals,
    patient
):

    return float(

        sum(

            max(
                0.0,

                min(
                    end,
                    b
                )

                -

                max(
                    start,
                    a
                )
            )

            for p,
            a,
            b
            in intervals

            if p == patient
        )
    )


def alarm_metrics(
    prediction_df
):

    prediction_df = (
        prediction_df
        .sort_values(
            [
                "patient",
                "start_abs_sec",
                "file",
                "window_index_in_file",
            ]
        )
        .reset_index(
            drop=True
        )
        .copy()
    )


    clean_intervals = merged_intervals(

        prediction_df[
            prediction_df[
                "label"
            ]
            == 0
        ]
    )


    clean_hours = interval_hours(
        clean_intervals
    )


    all_events = (

        set(

            prediction_df.loc[
                prediction_df[
                    "label"
                ]
                == 1,

                "event_id"
            ]
            .astype(
                str
            )
        )

        -

        {""}
    )


    detected_events = set()

    false_alarms = 0

    accepted_alarms = 0

    clean_warning_seconds = 0.0

    alarm_rows = []


    for patient, g in (
        prediction_df
        .groupby(
            "patient",
            sort=False
        )
    ):

        g = (
            g
            .sort_values(
                [
                    "start_abs_sec",
                    "file",
                    "window_index_in_file",
                ]
            )
        )


        refractory_until = (
            -np.inf
        )


        for _, row in g.iterrows():

            if int(
                row[
                    "predicted_positive"
                ]
            ) != 1:

                continue


            alarm_time = float(
                row[
                    "start_abs_sec"
                ]
            )


            if (
                alarm_time
                <
                refractory_until
            ):

                continue


            accepted_alarms += 1


            event_id = str(
                row[
                    "event_id"
                ]
            )


            if (
                int(
                    row[
                        "label"
                    ]
                )
                == 1

                and

                event_id != ""
            ):

                alarm_type = (
                    "true"
                )

                detected_events.add(
                    event_id
                )


            else:

                alarm_type = (
                    "false"
                )

                false_alarms += 1


                warning_start = (
                    alarm_time
                    +
                    SPH_SEC
                )


                warning_end = (
                    warning_start
                    +
                    SOP_SEC
                )


                clean_warning_seconds += (
                    overlap_seconds(
                        warning_start,
                        warning_end,
                        clean_intervals,
                        str(
                            patient
                        )
                    )
                )


            alarm_rows.append(
                {
                    "patient":
                        str(
                            patient
                        ),

                    "alarm_time_abs_sec":
                        alarm_time,

                    "alarm_type":
                        alarm_type,

                    "event_id":
                        event_id,
                }
            )


            refractory_until = (
                alarm_time
                +
                ALARM_REFRACTORY_SEC
            )


    n_events = len(
        all_events
    )


    sensitivity = (

        len(
            detected_events
        )

        /

        n_events

        if n_events > 0

        else np.nan
    )


    false_alarms_per_hour = (

        false_alarms
        /
        clean_hours

        if clean_hours > 0

        else np.nan
    )


    clean_time_in_warning = (

        clean_warning_seconds
        /
        (
            clean_hours
            *
            3600.0
        )

        if clean_hours > 0

        else np.nan
    )


    return {

        "future_seizures":
            n_events,

        "detected_seizures":
            len(
                detected_events
            ),

        "seizure_sensitivity":
            float(
                sensitivity
            ),

        "false_alarms":
            int(
                false_alarms
            ),

        "clean_exposure_hours":
            float(
                clean_hours
            ),

        "false_alarms_per_hour":
            float(
                false_alarms_per_hour
            ),

        "clean_time_in_warning":
            float(
                clean_time_in_warning
            ),

        "accepted_alarms":
            int(
                accepted_alarms
            ),

        "alarms":
            pd.DataFrame(
                alarm_rows
            ),
    }


# ============================================================
# 4. RUN ONE COMPLETE SEED
# ============================================================

def run_one_seed(
    seed
):

    seed_all(
        seed
    )


    print(
        "\n"
        + "#" * 90
    )

    print(
        "SEED",
        seed
    )

    print(
        "#" * 90
    )


    pred_parts = []

    episode_rows = []

    retention_rows = []

    memory_rows = []

    adaptation_rows = []

    threshold_rows = []


    for patient_number, patient in enumerate(
        PATIENTS,
        start=1
    ):

        print(
            "\n"
            + "=" * 70
        )

        print(
            patient,
            "| seed",
            seed
        )

        print(
            "=" * 70
        )


        patient_seed = (
            int(
                seed
            )
            +
            patient_number
            * 1000
        )


        # ----------------------------------------------------
        # INITIAL MODEL
        # ----------------------------------------------------

        (
            initial_model,
            threshold,
            best_val_ap,
            best_epoch,
            _
        ) = train_initial_model(
            patient,
            patient_seed
        )


        threshold_rows.append(
            {
                "seed":
                    seed,

                "patient":
                    patient,

                "threshold":
                    threshold,

                "best_validation_ap":
                    best_val_ap,

                "best_epoch":
                    best_epoch,
            }
        )


        initial_retention = evaluate_set(
            initial_model,
            PATIENT_DATA[
                patient
            ][
                "retention_idx"
            ],
            threshold
        )


        initial_state = {

            key:
                value.detach()
                .cpu()
                .clone()

            for key,
            value
            in initial_model
            .state_dict()
            .items()
        }


        models = {}


        for method in METHODS_FINAL:

            model = make_model()

            model.load_state_dict(
                initial_state
            )

            models[
                method
            ] = model


        seen_initial = np.asarray(
            PATIENT_DATA[
                patient
            ][
                "train_idx"
            ],
            dtype=np.int64
        )


        random_seen = (
            seen_initial.copy()
        )

        diverse_seen = (
            seen_initial.copy()
        )


        random_memory = (
            classwise_random_memory(
                random_seen,
                seed=(
                    patient_seed
                    +
                    101
                )
            )
        )


        diverse_memory = (
            classwise_diverse_memory(
                diverse_seen
            )
        )


        # ----------------------------------------------------
        # MEMORY AUDIT
        # ----------------------------------------------------

        for method, memory in (

            (
                "random_replay",
                random_memory
            ),

            (
                "connectivity_diverse_replay",
                diverse_memory
            ),
        ):

            n_negative, n_positive = (
                memory_class_counts(
                    memory
                )
            )


            if (
                len(
                    memory
                )
                != 128

                or

                n_negative
                != 64

                or

                n_positive
                != 64
            ):

                raise RuntimeError(
                    f"{patient}/{method}: "
                    "replay-budget invariant failed."
                )


            memory_rows.append(
                {
                    "seed":
                        seed,

                    "patient":
                        patient,

                    "method":
                        method,

                    "after_episode":
                        0,

                    "memory_size":
                        len(
                            memory
                        ),

                    "memory_negative":
                        n_negative,

                    "memory_positive":
                        n_positive,

                    "connectivity_diversity":
                        mean_pairwise_connectivity_distance(
                            memory
                        ),
                }
            )


        # ----------------------------------------------------
        # INITIAL RETENTION
        # ----------------------------------------------------

        for method in METHODS_FINAL:

            result = evaluate_set(
                models[
                    method
                ],
                PATIENT_DATA[
                    patient
                ][
                    "retention_idx"
                ],
                threshold
            )


            retention_rows.append(
                {
                    "seed":
                        seed,

                    "patient":
                        patient,

                    "method":
                        method,

                    "after_episode":
                        0,

                    "retention_ap":
                        result[
                            "average_precision"
                        ],

                    "initial_retention_ap":
                        initial_retention[
                            "average_precision"
                        ],
                }
            )


        # ====================================================
        # FUTURE EPISODES
        # ====================================================

        for episode in PATIENT_DATA[
            patient
        ][
            "episodes"
        ]:

            episode_id = str(
                episode[
                    "episode_id"
                ]
            )


            episode_index = int(
                episode[
                    "episode_index"
                ]
            )


            eval_idx = np.asarray(
                episode[
                    "eval_idx"
                ],
                dtype=np.int64
            )


            update_idx = np.asarray(
                episode[
                    "update_idx"
                ],
                dtype=np.int64
            )


            # ================================================
            # TEST FIRST
            # ================================================

            for method in METHODS_FINAL:

                result = evaluate_set(
                    models[
                        method
                    ],
                    eval_idx,
                    threshold
                )


                y = result[
                    "labels"
                ]


                probabilities = result[
                    "probabilities"
                ]


                episode_rows.append(
                    {
                        "seed":
                            seed,

                        "patient":
                            patient,

                        "method":
                            method,

                        "episode_id":
                            episode_id,

                        "episode_index":
                            episode_index,

                        "event_id":
                            str(
                                episode[
                                    "event_id"
                                ]
                            ),

                        "n_eval":
                            len(
                                eval_idx
                            ),

                        "n_positive":
                            int(
                                (
                                    y == 1
                                ).sum()
                            ),

                        "n_negative":
                            int(
                                (
                                    y == 0
                                ).sum()
                            ),

                        "average_precision":
                            result[
                                "average_precision"
                            ],

                        "brier":
                            result[
                                "brier"
                            ],

                        "threshold":
                            threshold,
                    }
                )


                metadata = (
                    FINAL_SPLIT
                    .iloc[
                        eval_idx
                    ][
                        [
                            "row_id",
                            "window_key",
                            "patient",
                            "file",
                            "window_index_in_file",
                            "start_abs_sec",
                            "end_abs_sec",
                            "event_id",
                            "label",
                        ]
                    ]
                    .copy()
                )


                metadata[
                    "seed"
                ] = seed


                metadata[
                    "method"
                ] = method


                metadata[
                    "episode_id"
                ] = episode_id


                metadata[
                    "episode_index"
                ] = episode_index


                metadata[
                    "probability"
                ] = probabilities


                metadata[
                    "threshold"
                ] = threshold


                metadata[
                    "predicted_positive"
                ] = (
                    probabilities
                    >=
                    threshold
                ).astype(
                    np.int8
                )


                pred_parts.append(
                    metadata
                )


            # ================================================
            # REVEAL LABELS → UPDATE
            # ================================================

            start_clock = (
                time.perf_counter()
            )


            train_fixed_steps_v2(
                models[
                    "finetune"
                ],
                update_idx,
                np.empty(
                    0,
                    dtype=np.int64
                ),
                patient_seed
                +
                episode_index
                * 100
                +
                1
            )


            finetune_seconds = (
                time.perf_counter()
                -
                start_clock
            )


            start_clock = (
                time.perf_counter()
            )


            train_fixed_steps_v2(
                models[
                    "random_replay"
                ],
                update_idx,
                random_memory,
                patient_seed
                +
                episode_index
                * 100
                +
                2
            )


            random_seconds = (
                time.perf_counter()
                -
                start_clock
            )


            start_clock = (
                time.perf_counter()
            )


            train_fixed_steps_v2(
                models[
                    "connectivity_diverse_replay"
                ],
                update_idx,
                diverse_memory,
                patient_seed
                +
                episode_index
                * 100
                +
                3
            )


            diverse_seconds = (
                time.perf_counter()
                -
                start_clock
            )


            adaptation_rows.extend(
                [
                    {
                        "seed":
                            seed,

                        "patient":
                            patient,

                        "episode_id":
                            episode_id,

                        "method":
                            "static",

                        "seconds":
                            0.0,
                    },

                    {
                        "seed":
                            seed,

                        "patient":
                            patient,

                        "episode_id":
                            episode_id,

                        "method":
                            "finetune",

                        "seconds":
                            finetune_seconds,
                    },

                    {
                        "seed":
                            seed,

                        "patient":
                            patient,

                        "episode_id":
                            episode_id,

                        "method":
                            "random_replay",

                        "seconds":
                            random_seconds,
                    },

                    {
                        "seed":
                            seed,

                        "patient":
                            patient,

                        "episode_id":
                            episode_id,

                        "method":
                            "connectivity_diverse_replay",

                        "seconds":
                            diverse_seconds,
                    },
                ]
            )


            # ================================================
            # UPDATE MEMORIES
            # ================================================

            random_seen = np.unique(
                np.concatenate(
                    [
                        random_seen,
                        update_idx
                    ]
                )
            )


            diverse_seen = np.unique(
                np.concatenate(
                    [
                        diverse_seen,
                        update_idx
                    ]
                )
            )


            random_memory = (
                classwise_random_memory(
                    random_seen,
                    seed=(
                        patient_seed
                        +
                        episode_index
                        * 1000
                        +
                        11
                    )
                )
            )


            diverse_memory = (
                classwise_diverse_memory(
                    diverse_seen
                )
            )


            for method, memory in (

                (
                    "random_replay",
                    random_memory
                ),

                (
                    "connectivity_diverse_replay",
                    diverse_memory
                ),
            ):

                n_negative, n_positive = (
                    memory_class_counts(
                        memory
                    )
                )


                if (
                    len(
                        memory
                    )
                    != 128

                    or

                    n_negative
                    != 64

                    or

                    n_positive
                    != 64
                ):

                    raise RuntimeError(
                        f"{patient}/"
                        f"{episode_id}/"
                        f"{method}: "
                        "replay-budget invariant failed."
                    )


                memory_rows.append(
                    {
                        "seed":
                            seed,

                        "patient":
                            patient,

                        "method":
                            method,

                        "after_episode":
                            episode_index,

                        "memory_size":
                            len(
                                memory
                            ),

                        "memory_negative":
                            n_negative,

                        "memory_positive":
                            n_positive,

                        "connectivity_diversity":
                            mean_pairwise_connectivity_distance(
                                memory
                            ),
                    }
                )


            # ================================================
            # RETENTION AFTER UPDATE
            # ================================================

            for method in METHODS_FINAL:

                result = evaluate_set(
                    models[
                        method
                    ],
                    PATIENT_DATA[
                        patient
                    ][
                        "retention_idx"
                    ],
                    threshold
                )


                retention_rows.append(
                    {
                        "seed":
                            seed,

                        "patient":
                            patient,

                        "method":
                            method,

                        "after_episode":
                            episode_index,

                        "retention_ap":
                            result[
                                "average_precision"
                            ],

                        "initial_retention_ap":
                            initial_retention[
                                "average_precision"
                            ],
                    }
                )


    return {

        "predictions":
            pd.concat(
                pred_parts,
                ignore_index=True
            ),

        "episodes":
            pd.DataFrame(
                episode_rows
            ),

        "retention":
            pd.DataFrame(
                retention_rows
            ),

        "memory":
            pd.DataFrame(
                memory_rows
            ),

        "adaptation":
            pd.DataFrame(
                adaptation_rows
            ),

        "thresholds":
            pd.DataFrame(
                threshold_rows
            ),
    }


# ============================================================
# 5. RUN ALL FIVE SEEDS
# ============================================================

stage9_start = (
    time.time()
)

outputs = []


for seed in SEEDS:

    outputs.append(
        run_one_seed(
            seed
        )
    )


PRED = pd.concat(
    [
        x[
            "predictions"
        ]
        for x in outputs
    ],
    ignore_index=True
)


EPISODES = pd.concat(
    [
        x[
            "episodes"
        ]
        for x in outputs
    ],
    ignore_index=True
)


RETENTION = pd.concat(
    [
        x[
            "retention"
        ]
        for x in outputs
    ],
    ignore_index=True
)


MEMORY = pd.concat(
    [
        x[
            "memory"
        ]
        for x in outputs
    ],
    ignore_index=True
)


ADAPT = pd.concat(
    [
        x[
            "adaptation"
        ]
        for x in outputs
    ],
    ignore_index=True
)


THRESHOLDS = pd.concat(
    [
        x[
            "thresholds"
        ]
        for x in outputs
    ],
    ignore_index=True
)


# ============================================================
# 6. ALARM METRICS
# ============================================================

alarm_rows = []

alarm_parts = []


for seed in SEEDS:

    for method in METHODS_FINAL:

        subset = PRED[
            (
                PRED[
                    "seed"
                ]
                == seed
            )
            &
            (
                PRED[
                    "method"
                ]
                == method
            )
        ].copy()


        alarm = alarm_metrics(
            subset
        )


        y = subset[
            "label"
        ].to_numpy(
            dtype=np.int64
        )


        probability = subset[
            "probability"
        ].to_numpy(
            dtype=float
        )


        alarm_rows.append(
            {
                "seed":
                    seed,

                "method":
                    method,

                "pooled_future_ap":
                    safe_average_precision(
                        y,
                        probability
                    ),

                "pooled_future_brier":
                    safe_brier(
                        y,
                        probability
                    ),

                "future_seizures":
                    alarm[
                        "future_seizures"
                    ],

                "detected_seizures":
                    alarm[
                        "detected_seizures"
                    ],

                "seizure_sensitivity":
                    alarm[
                        "seizure_sensitivity"
                    ],

                "false_alarms":
                    alarm[
                        "false_alarms"
                    ],

                "clean_exposure_hours":
                    alarm[
                        "clean_exposure_hours"
                    ],

                "false_alarms_per_hour":
                    alarm[
                        "false_alarms_per_hour"
                    ],

                "clean_time_in_warning":
                    alarm[
                        "clean_time_in_warning"
                    ],

                "accepted_alarms":
                    alarm[
                        "accepted_alarms"
                    ],
            }
        )


        if not alarm[
            "alarms"
        ].empty:

            alarms = (
                alarm[
                    "alarms"
                ]
                .copy()
            )

            alarms[
                "seed"
            ] = seed

            alarms[
                "method"
            ] = method

            alarm_parts.append(
                alarms
            )


ALARM_SEED = pd.DataFrame(
    alarm_rows
)


ALARMS = (

    pd.concat(
        alarm_parts,
        ignore_index=True
    )

    if alarm_parts

    else pd.DataFrame()
)


# ============================================================
# 7. RETENTION / FORGETTING
# ============================================================

retention_summary = []


for (
    seed,
    patient,
    method
), group in RETENTION.groupby(
    [
        "seed",
        "patient",
        "method"
    ],
    sort=False
):

    group = group.sort_values(
        "after_episode"
    )


    initial_ap = float(
        group.iloc[
            0
        ][
            "retention_ap"
        ]
    )


    final_ap = float(
        group.iloc[
            -1
        ][
            "retention_ap"
        ]
    )


    retention_summary.append(
        {
            "seed":
                seed,

            "patient":
                patient,

            "method":
                method,

            "initial_retention_ap":
                initial_ap,

            "final_retention_ap":
                final_ap,

            "backward_transfer":
                final_ap
                -
                initial_ap,

            "forgetting":
                max(
                    0.0,
                    initial_ap
                    -
                    final_ap
                ),
        }
    )


RET_SUM = pd.DataFrame(
    retention_summary
)


BWT = (
    RET_SUM
    .groupby(
        [
            "seed",
            "method"
        ],
        as_index=False
    )
    .agg(
        mean_backward_transfer=(
            "backward_transfer",
            "mean"
        ),

        mean_forgetting=(
            "forgetting",
            "mean"
        ),

        mean_final_retention_ap=(
            "final_retention_ap",
            "mean"
        ),
    )
)


# ============================================================
# 8. COMPUTATIONAL / MEMORY SUMMARIES
# ============================================================

ADAPT_SUM = (
    ADAPT
    .groupby(
        [
            "seed",
            "method"
        ],
        as_index=False
    )
    .agg(
        mean_adaptation_seconds=(
            "seconds",
            "mean"
        ),

        total_adaptation_seconds=(
            "seconds",
            "sum"
        ),
    )
)


MEM_SUM = (
    MEMORY
    .groupby(
        [
            "seed",
            "method"
        ],
        as_index=False
    )
    .agg(
        mean_memory_size=(
            "memory_size",
            "mean"
        ),

        mean_connectivity_diversity=(
            "connectivity_diversity",
            "mean"
        ),
    )
)


SEED_METHOD = (
    ALARM_SEED
    .merge(
        BWT,
        on=[
            "seed",
            "method"
        ],
        how="left"
    )
    .merge(
        ADAPT_SUM,
        on=[
            "seed",
            "method"
        ],
        how="left"
    )
    .merge(
        MEM_SUM,
        on=[
            "seed",
            "method"
        ],
        how="left"
    )
)


# ============================================================
# 9. FIVE-SEED ROBUSTNESS
# ============================================================

metrics = [
    "pooled_future_ap",
    "seizure_sensitivity",
    "false_alarms_per_hour",
    "clean_time_in_warning",
    "mean_backward_transfer",
    "mean_forgetting",
    "mean_final_retention_ap",
    "mean_adaptation_seconds",
]


robustness_rows = []


for method in METHODS_FINAL:

    subset = SEED_METHOD[
        SEED_METHOD[
            "method"
        ]
        == method
    ]


    row = {
        "method":
            method
    }


    for metric in metrics:

        values = pd.to_numeric(
            subset[
                metric
            ],
            errors="coerce"
        ).dropna()


        row[
            metric
            +
            "_mean"
        ] = float(
            values.mean()
        )


        row[
            metric
            +
            "_sd"
        ] = float(
            values.std(
                ddof=1
            )
        )


    robustness_rows.append(
        row
    )


ROBUSTNESS = pd.DataFrame(
    robustness_rows
)


# ============================================================
# 10. RANDOM VS DIVERSE PAIRED SEED DIFFERENCES
#
# Descriptive optimization robustness only.
# Seeds are NOT independent clinical samples.
# ============================================================

random_seed = (
    SEED_METHOD[
        SEED_METHOD[
            "method"
        ]
        == "random_replay"
    ]
    .set_index(
        "seed"
    )
)


diverse_seed = (
    SEED_METHOD[
        SEED_METHOD[
            "method"
        ]
        ==
        "connectivity_diverse_replay"
    ]
    .set_index(
        "seed"
    )
)


paired_rows = []


for metric in metrics:

    difference = (

        diverse_seed[
            metric
        ]

        -

        random_seed[
            metric
        ]
    ).dropna()


    paired_rows.append(
        {
            "metric":
                metric,

            "direction":
                "diverse_minus_random",

            "mean_difference":
                float(
                    difference.mean()
                ),

            "sd_difference":
                float(
                    difference.std(
                        ddof=1
                    )
                ),

            "min_difference":
                float(
                    difference.min()
                ),

            "max_difference":
                float(
                    difference.max()
                ),

            "n_seeds":
                len(
                    difference
                ),
        }
    )


PAIRED = pd.DataFrame(
    paired_rows
)


# ============================================================
# 11. MEMORY ROBUSTNESS
# ============================================================

MEM_ROB = (
    MEMORY
    .groupby(
        "method",
        as_index=False
    )
    .agg(
        mean_memory_size=(
            "memory_size",
            "mean"
        ),

        mean_pairwise_connectivity_distance=(
            "connectivity_diversity",
            "mean"
        ),

        sd_pairwise_connectivity_distance=(
            "connectivity_diversity",
            "std"
        ),
    )
)


# ============================================================
# 12. SAVE
# ============================================================

PRED.to_pickle(
    STAGE9_ROOT
    / "multiseed_future_predictions.pkl"
)


PRED.to_csv(
    STAGE9_ROOT
    / "multiseed_future_predictions.csv",
    index=False
)


EPISODES.to_csv(
    STAGE9_ROOT
    / "multiseed_episode_metrics.csv",
    index=False
)


RETENTION.to_csv(
    STAGE9_ROOT
    / "multiseed_retention_metrics.csv",
    index=False
)


MEMORY.to_csv(
    STAGE9_ROOT
    / "multiseed_memory_audit.csv",
    index=False
)


ADAPT.to_csv(
    STAGE9_ROOT
    / "multiseed_adaptation_time.csv",
    index=False
)


THRESHOLDS.to_csv(
    STAGE9_ROOT
    / "multiseed_thresholds.csv",
    index=False
)


ALARM_SEED.to_csv(
    STAGE9_ROOT
    / "alarm_seed_summary.csv",
    index=False
)


ALARMS.to_csv(
    STAGE9_ROOT
    / "accepted_alarm_events.csv",
    index=False
)


RET_SUM.to_csv(
    STAGE9_ROOT
    / "retention_seed_patient.csv",
    index=False
)


SEED_METHOD.to_csv(
    STAGE9_ROOT
    / "seed_method_summary.csv",
    index=False
)


ROBUSTNESS.to_csv(
    STAGE9_ROOT
    / "robustness_summary.csv",
    index=False
)


PAIRED.to_csv(
    STAGE9_ROOT
    / "random_vs_diverse_seed_differences.csv",
    index=False
)


MEM_ROB.to_csv(
    STAGE9_ROOT
    / "memory_robustness.csv",
    index=False
)


config = {

    "seeds":
        list(
            SEEDS
        ),

    "methods":
        list(
            METHODS_FINAL
        ),

    "sph_seconds":
        SPH_SEC,

    "sop_seconds":
        SOP_SEC,

    "alarm_refractory_seconds":
        ALARM_REFRACTORY_SEC,

    "alarm_rule":
        (
            "first above-threshold prediction outside "
            "refractory raises an alarm; new alarms "
            "suppressed for SPH+SOP"
        ),

    "threshold_rule":
        (
            "selected on initial validation only "
            "and frozen"
        ),

    "future_protocol":
        "strict test-before-update",

    "continual_update":
        (
            "Stage 8B corrected unweighted BCE "
            "with balanced sampling when both "
            "classes exist"
        ),

    "final_split_hash":
        FINAL_CONFIG[
            "final_split_hash"
        ],

    "preprocess_hash":
        FINAL_CONFIG[
            "preprocess_hash"
        ],
}


(
    STAGE9_ROOT
    / "stage9_config.json"
).write_text(
    json.dumps(
        config,
        indent=2
    )
)


# ============================================================
# 13. REPORT
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "STAGE 9 FIVE-SEED + ALARM EVALUATION COMPLETE"
)

print(
    "=" * 100
)


print(
    f"Runtime: "
    f"{(time.time() - stage9_start) / 60.0:.1f} minutes"
)


print(
    "\nSEED × METHOD SUMMARY"
)

display(
    SEED_METHOD
)


print(
    "\nFIVE-SEED ROBUSTNESS SUMMARY"
)

display(
    ROBUSTNESS
)


print(
    "\nRANDOM VS CONNECTIVITY-DIVERSE"
)

display(
    PAIRED
)


print(
    "\nMEMORY ROBUSTNESS"
)

display(
    MEM_ROB
)


print(
    "\nTHRESHOLDS"
)

display(
    THRESHOLDS
)


print(
    "\nSaved to:",
    STAGE9_ROOT
)


In [ ]:
# ============================================================
# Paper tables, descriptive analyses, and figures
# ============================================================
# This cell reads saved experiment outputs and does not retrain models.

from pathlib import Path
import itertools
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
)


# ============================================================
# 1. PATHS
# ============================================================

ROOT = Path(
    "/content/drive/MyDrive/EEG_Research/"
    "continual_graph_forecasting"
)

STAGE9_ROOT = (
    ROOT
    / "experiments"
    / "continual_multiseed_final_v1"
)

FINAL_ROOT = (
    ROOT
    / "experiments"
    / "final_paper_results_v1"
)

FINAL_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. LOAD FROZEN STAGE 9 RESULTS
# ============================================================

required_files = [
    "multiseed_future_predictions.csv",
    "multiseed_retention_metrics.csv",
    "multiseed_memory_audit.csv",
    "multiseed_thresholds.csv",
    "seed_method_summary.csv",
    "robustness_summary.csv",
    "random_vs_diverse_seed_differences.csv",
    "memory_robustness.csv",
]

for fname in required_files:
    path = STAGE9_ROOT / fname

    if not path.exists():
        raise FileNotFoundError(
            f"Missing Stage 9 output: {path}"
        )


PRED = pd.read_csv(
    STAGE9_ROOT
    / "multiseed_future_predictions.csv"
)

RET = pd.read_csv(
    STAGE9_ROOT
    / "multiseed_retention_metrics.csv"
)

MEM = pd.read_csv(
    STAGE9_ROOT
    / "multiseed_memory_audit.csv"
)

THRESHOLDS = pd.read_csv(
    STAGE9_ROOT
    / "multiseed_thresholds.csv"
)

SEED_METHOD = pd.read_csv(
    STAGE9_ROOT
    / "seed_method_summary.csv"
)

ROBUSTNESS = pd.read_csv(
    STAGE9_ROOT
    / "robustness_summary.csv"
)

SEED_DIFF = pd.read_csv(
    STAGE9_ROOT
    / "random_vs_diverse_seed_differences.csv"
)

MEM_ROB = pd.read_csv(
    STAGE9_ROOT
    / "memory_robustness.csv"
)


METHODS = [
    "static",
    "finetune",
    "random_replay",
    "connectivity_diverse_replay",
]

PATIENTS = sorted(
    PRED[
        "patient"
    ].unique()
)

SEEDS = sorted(
    PRED[
        "seed"
    ].unique()
)


if PATIENTS != [
    "chb06",
    "chb09",
    "chb10",
]:
    raise RuntimeError(
        f"Unexpected patient cohort: {PATIENTS}"
    )


if SEEDS != [
    2026,
    2027,
    2028,
    2029,
    2030,
]:
    raise RuntimeError(
        f"Unexpected seeds: {SEEDS}"
    )


# ============================================================
# 3. HELPERS
# ============================================================

SPH_SEC = 5 * 60
SOP_SEC = 30 * 60
REFRACTORY_SEC = SPH_SEC + SOP_SEC


def safe_ap(
    y,
    p
):

    y = np.asarray(
        y,
        dtype=np.int64
    )

    p = np.asarray(
        p,
        dtype=float
    )

    if np.unique(
        y
    ).size < 2:

        return np.nan

    return float(
        average_precision_score(
            y,
            p
        )
    )


def union_hours(
    df
):

    if df.empty:
        return 0.0

    total_sec = 0.0

    for _, file_df in df.groupby(
        "file",
        sort=False
    ):

        intervals = sorted(
            zip(
                file_df[
                    "start_abs_sec"
                ].astype(float),

                file_df[
                    "end_abs_sec"
                ].astype(float)
            )
        )

        if not intervals:
            continue

        cur_start, cur_end = (
            intervals[0]
        )

        for start, end in intervals[
            1:
        ]:

            if start <= cur_end:

                cur_end = max(
                    cur_end,
                    end
                )

            else:

                total_sec += (
                    cur_end
                    -
                    cur_start
                )

                cur_start = start
                cur_end = end

        total_sec += (
            cur_end
            -
            cur_start
        )

    return float(
        total_sec
        /
        3600.0
    )


def patient_alarm_metrics(
    df
):

    df = (
        df.sort_values(
            [
                "start_abs_sec",
                "file",
                "window_index_in_file"
            ]
        )
        .reset_index(
            drop=True
        )
    )

    clean_df = df[
        df[
            "label"
        ]
        == 0
    ]

    clean_hours = union_hours(
        clean_df
    )

    event_ids = sorted(
        set(
            df.loc[
                df[
                    "label"
                ]
                == 1,

                "event_id"
            ]
            .astype(str)
        )
        -
        {""}
    )

    detected = set()

    false_alarms = 0

    refractory_until = (
        -np.inf
    )

    for _, row in df.iterrows():

        if int(
            row[
                "predicted_positive"
            ]
        ) != 1:
            continue

        alarm_time = float(
            row[
                "start_abs_sec"
            ]
        )

        if (
            alarm_time
            <
            refractory_until
        ):
            continue

        if int(
            row[
                "label"
            ]
        ) == 1:

            event_id = str(
                row[
                    "event_id"
                ]
            )

            if event_id != "":

                detected.add(
                    event_id
                )

        else:

            false_alarms += 1


        refractory_until = (
            alarm_time
            +
            REFRACTORY_SEC
        )


    sensitivity = (

        len(
            detected
        )
        /
        len(
            event_ids
        )

        if event_ids

        else np.nan
    )


    far_h = (

        false_alarms
        /
        clean_hours

        if clean_hours > 0

        else np.nan
    )


    return {

        "future_events":
            len(
                event_ids
            ),

        "detected_events":
            len(
                detected
            ),

        "seizure_sensitivity":
            float(
                sensitivity
            ),

        "false_alarms":
            int(
                false_alarms
            ),

        "clean_exposure_hours":
            float(
                clean_hours
            ),

        "false_alarms_per_hour":
            float(
                far_h
            ),
    }


def exact_signflip_test(
    differences
):

    differences = np.asarray(
        differences,
        dtype=float
    )

    differences = differences[
        np.isfinite(
            differences
        )
    ]

    n = len(
        differences
    )

    if n == 0:
        return np.nan


    observed = abs(
        differences.mean()
    )


    permuted = []


    for signs in itertools.product(
        [-1.0, 1.0],
        repeat=n
    ):

        signs = np.asarray(
            signs,
            dtype=float
        )


        permuted.append(
            abs(
                np.mean(
                    differences
                    *
                    signs
                )
            )
        )


    permuted = np.asarray(
        permuted
    )


    return float(
        np.mean(
            permuted
            >=
            observed
            -
            1e-12
        )
    )


# ============================================================
# 4. PATIENT × SEED × METHOD METRICS
# ============================================================

patient_seed_rows = []


for seed in SEEDS:

    for patient in PATIENTS:

        for method in METHODS:

            subset = PRED[
                (
                    PRED[
                        "seed"
                    ]
                    == seed
                )
                &
                (
                    PRED[
                        "patient"
                    ]
                    == patient
                )
                &
                (
                    PRED[
                        "method"
                    ]
                    == method
                )
            ].copy()


            if subset.empty:

                raise RuntimeError(
                    f"Missing predictions for "
                    f"{seed}/{patient}/{method}"
                )


            y = subset[
                "label"
            ].to_numpy(
                dtype=np.int64
            )


            probability = subset[
                "probability"
            ].to_numpy(
                dtype=float
            )


            alarm = (
                patient_alarm_metrics(
                    subset
                )
            )


            retention = (
                RET[
                    (
                        RET[
                            "seed"
                        ]
                        == seed
                    )
                    &
                    (
                        RET[
                            "patient"
                        ]
                        == patient
                    )
                    &
                    (
                        RET[
                            "method"
                        ]
                        == method
                    )
                ]
                .sort_values(
                    "after_episode"
                )
            )


            if retention.empty:

                raise RuntimeError(
                    f"Missing retention rows for "
                    f"{seed}/{patient}/{method}"
                )


            initial_ret_ap = float(
                retention.iloc[
                    0
                ][
                    "retention_ap"
                ]
            )


            final_ret_ap = float(
                retention.iloc[
                    -1
                ][
                    "retention_ap"
                ]
            )


            patient_seed_rows.append(
                {
                    "seed":
                        seed,

                    "patient":
                        patient,

                    "method":
                        method,

                    "pooled_future_ap":
                        safe_ap(
                            y,
                            probability
                        ),

                    "pooled_future_brier":
                        float(
                            brier_score_loss(
                                y,
                                probability
                            )
                        ),

                    "future_events":
                        alarm[
                            "future_events"
                        ],

                    "detected_events":
                        alarm[
                            "detected_events"
                        ],

                    "seizure_sensitivity":
                        alarm[
                            "seizure_sensitivity"
                        ],

                    "false_alarms":
                        alarm[
                            "false_alarms"
                        ],

                    "clean_exposure_hours":
                        alarm[
                            "clean_exposure_hours"
                        ],

                    "false_alarms_per_hour":
                        alarm[
                            "false_alarms_per_hour"
                        ],

                    "initial_retention_ap":
                        initial_ret_ap,

                    "final_retention_ap":
                        final_ret_ap,

                    "backward_transfer":
                        final_ret_ap
                        -
                        initial_ret_ap,

                    "forgetting":
                        max(
                            0.0,
                            initial_ret_ap
                            -
                            final_ret_ap
                        ),
                }
            )


PATIENT_SEED = pd.DataFrame(
    patient_seed_rows
)


# ============================================================
# 5. PATIENT-LEVEL RESULTS
# ============================================================

PATIENT_METHOD = (
    PATIENT_SEED
    .groupby(
        [
            "patient",
            "method"
        ],
        as_index=False
    )
    .agg(
        pooled_future_ap_mean=(
            "pooled_future_ap",
            "mean"
        ),

        pooled_future_ap_sd=(
            "pooled_future_ap",
            "std"
        ),

        seizure_sensitivity_mean=(
            "seizure_sensitivity",
            "mean"
        ),

        false_alarms_per_hour_mean=(
            "false_alarms_per_hour",
            "mean"
        ),

        false_alarms_per_hour_sd=(
            "false_alarms_per_hour",
            "std"
        ),

        backward_transfer_mean=(
            "backward_transfer",
            "mean"
        ),

        backward_transfer_sd=(
            "backward_transfer",
            "std"
        ),

        forgetting_mean=(
            "forgetting",
            "mean"
        ),

        final_retention_ap_mean=(
            "final_retention_ap",
            "mean"
        ),
    )
)


# ============================================================
# 6. FINAL METHOD TABLE
# ============================================================

FINAL_METHOD_TABLE = (
    ROBUSTNESS[
        [
            "method",
            "pooled_future_ap_mean",
            "pooled_future_ap_sd",
            "seizure_sensitivity_mean",
            "false_alarms_per_hour_mean",
            "false_alarms_per_hour_sd",
            "clean_time_in_warning_mean",
            "clean_time_in_warning_sd",
            "mean_backward_transfer_mean",
            "mean_backward_transfer_sd",
            "mean_forgetting_mean",
            "mean_forgetting_sd",
            "mean_final_retention_ap_mean",
            "mean_final_retention_ap_sd",
            "mean_adaptation_seconds_mean",
            "mean_adaptation_seconds_sd",
        ]
    ]
    .copy()
)


# ============================================================
# 7. RANDOM VS DIVERSE — PATIENT-LEVEL PAIRED ANALYSIS
# ============================================================

random_patient = (
    PATIENT_METHOD[
        PATIENT_METHOD[
            "method"
        ]
        ==
        "random_replay"
    ]
    .set_index(
        "patient"
    )
)


diverse_patient = (
    PATIENT_METHOD[
        PATIENT_METHOD[
            "method"
        ]
        ==
        "connectivity_diverse_replay"
    ]
    .set_index(
        "patient"
    )
)


comparison_specs = [
    (
        "pooled_future_ap_mean",
        "higher_better"
    ),

    (
        "false_alarms_per_hour_mean",
        "lower_better"
    ),

    (
        "backward_transfer_mean",
        "higher_better"
    ),

    (
        "forgetting_mean",
        "lower_better"
    ),

    (
        "final_retention_ap_mean",
        "higher_better"
    ),
]


paired_rows = []


for metric, interpretation in (
    comparison_specs
):

    difference = (

        diverse_patient[
            metric
        ]

        -

        random_patient[
            metric
        ]
    )


    paired_rows.append(
        {
            "metric":
                metric,

            "interpretation":
                interpretation,

            "chb06_difference":
                float(
                    difference.loc[
                        "chb06"
                    ]
                ),

            "chb09_difference":
                float(
                    difference.loc[
                        "chb09"
                    ]
                ),

            "chb10_difference":
                float(
                    difference.loc[
                        "chb10"
                    ]
                ),

            "mean_patient_difference":
                float(
                    difference.mean()
                ),

            "exact_signflip_p":
                exact_signflip_test(
                    difference.values
                ),

            "n_patients":
                len(
                    difference
                ),
        }
    )


RANDOM_VS_DIVERSE_PATIENT = (
    pd.DataFrame(
        paired_rows
    )
)


# ============================================================
# 8. MEMORY-DIVERSITY EFFECT
# ============================================================

random_memory_distance = float(

    MEM_ROB.loc[
        MEM_ROB[
            "method"
        ]
        ==
        "random_replay",

        "mean_pairwise_connectivity_distance"
    ].iloc[
        0
    ]
)


diverse_memory_distance = float(

    MEM_ROB.loc[
        MEM_ROB[
            "method"
        ]
        ==
        "connectivity_diverse_replay",

        "mean_pairwise_connectivity_distance"
    ].iloc[
        0
    ]
)


memory_diversity_gain_percent = (

    (
        diverse_memory_distance
        /
        random_memory_distance
    )

    -
    1.0

) * 100.0


MEMORY_EFFECT = pd.DataFrame(
    [
        {
            "random_replay_mean_distance":
                random_memory_distance,

            "connectivity_diverse_mean_distance":
                diverse_memory_distance,

            "relative_increase_percent":
                memory_diversity_gain_percent,
        }
    ]
)


# ============================================================
# 9. KEY RESULT CHECK
# ============================================================

random_seed_row = FINAL_METHOD_TABLE[
    FINAL_METHOD_TABLE[
        "method"
    ]
    ==
    "random_replay"
].iloc[
    0
]


diverse_seed_row = FINAL_METHOD_TABLE[
    FINAL_METHOD_TABLE[
        "method"
    ]
    ==
    "connectivity_diverse_replay"
].iloc[
    0
]


AP_DIFF = float(
    diverse_seed_row[
        "pooled_future_ap_mean"
    ]
    -
    random_seed_row[
        "pooled_future_ap_mean"
    ]
)


FAR_DIFF = float(
    diverse_seed_row[
        "false_alarms_per_hour_mean"
    ]
    -
    random_seed_row[
        "false_alarms_per_hour_mean"
    ]
)


BWT_DIFF = float(
    diverse_seed_row[
        "mean_backward_transfer_mean"
    ]
    -
    random_seed_row[
        "mean_backward_transfer_mean"
    ]
)


# ============================================================
# 10. FIGURE 1 — PERFORMANCE / FALSE-ALARM TRADEOFF
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        7.2,
        5.2
    )
)


for _, row in FINAL_METHOD_TABLE.iterrows():

    ax.scatter(
        row[
            "false_alarms_per_hour_mean"
        ],
        row[
            "pooled_future_ap_mean"
        ],
        s=90
    )


    label = str(
        row[
            "method"
        ]
    ).replace(
        "_",
        " "
    )


    ax.annotate(
        label,
        (
            row[
                "false_alarms_per_hour_mean"
            ],
            row[
                "pooled_future_ap_mean"
            ]
        ),
        xytext=(
            5,
            5
        ),
        textcoords="offset points",
        fontsize=9
    )


ax.set_xlabel(
    "False alarms per hour"
)

ax.set_ylabel(
    "Pooled future AUPRC"
)

ax.set_title(
    "Future forecasting performance across five seeds"
)

ax.grid(
    alpha=0.25
)

fig.tight_layout()


fig.savefig(
    FINAL_ROOT
    / "fig_performance_vs_false_alarms.png",
    dpi=300,
    bbox_inches="tight"
)


fig.savefig(
    FINAL_ROOT
    / "fig_performance_vs_false_alarms.pdf",
    bbox_inches="tight"
)


plt.show()


# ============================================================
# 11. FIGURE 2 — MEMORY CONNECTIVITY DIVERSITY
# ============================================================

memory_plot = MEM_ROB[
    MEM_ROB[
        "method"
    ].isin(
        [
            "random_replay",
            "connectivity_diverse_replay"
        ]
    )
].copy()


memory_plot[
    "label"
] = (
    memory_plot[
        "method"
    ]
    .str.replace(
        "_",
        " ",
        regex=False
    )
)


fig, ax = plt.subplots(
    figsize=(
        6.5,
        4.8
    )
)


ax.bar(
    memory_plot[
        "label"
    ],
    memory_plot[
        "mean_pairwise_connectivity_distance"
    ],
    yerr=memory_plot[
        "sd_pairwise_connectivity_distance"
    ],
    capsize=5
)


ax.set_ylabel(
    "Mean pairwise connectivity distance"
)

ax.set_title(
    "Replay-memory connectivity diversity"
)

ax.tick_params(
    axis="x",
    rotation=10
)

fig.tight_layout()


fig.savefig(
    FINAL_ROOT
    / "fig_memory_diversity.png",
    dpi=300,
    bbox_inches="tight"
)


fig.savefig(
    FINAL_ROOT
    / "fig_memory_diversity.pdf",
    bbox_inches="tight"
)


plt.show()


# ============================================================
# 12. FIGURE 3 — RETENTION / BACKWARD TRANSFER
# ============================================================

plot_df = FINAL_METHOD_TABLE.copy()


plot_df[
    "label"
] = (
    plot_df[
        "method"
    ]
    .str.replace(
        "_",
        " ",
        regex=False
    )
)


fig, ax = plt.subplots(
    figsize=(
        7.3,
        4.8
    )
)


ax.bar(
    plot_df[
        "label"
    ],
    plot_df[
        "mean_backward_transfer_mean"
    ],
    yerr=plot_df[
        "mean_backward_transfer_sd"
    ],
    capsize=5
)


ax.axhline(
    0.0,
    linewidth=1
)


ax.set_ylabel(
    "Mean backward transfer"
)

ax.set_title(
    "Retention change after continual adaptation"
)

ax.tick_params(
    axis="x",
    rotation=12
)

fig.tight_layout()


fig.savefig(
    FINAL_ROOT
    / "fig_backward_transfer.png",
    dpi=300,
    bbox_inches="tight"
)


fig.savefig(
    FINAL_ROOT
    / "fig_backward_transfer.pdf",
    bbox_inches="tight"
)


plt.show()


# ============================================================
# 13. SAVE FINAL TABLES
# ============================================================

PATIENT_SEED.to_csv(
    FINAL_ROOT
    / "table_patient_seed_metrics.csv",
    index=False
)


PATIENT_METHOD.to_csv(
    FINAL_ROOT
    / "table_patient_method_metrics.csv",
    index=False
)


FINAL_METHOD_TABLE.to_csv(
    FINAL_ROOT
    / "table_final_method_summary.csv",
    index=False
)


RANDOM_VS_DIVERSE_PATIENT.to_csv(
    FINAL_ROOT
    / "table_random_vs_diverse_patient_paired.csv",
    index=False
)


MEMORY_EFFECT.to_csv(
    FINAL_ROOT
    / "table_memory_diversity_effect.csv",
    index=False
)


# ============================================================
# 14. AUTOMATIC RESULTS TEXT
# ============================================================

static_row = FINAL_METHOD_TABLE[
    FINAL_METHOD_TABLE[
        "method"
    ]
    ==
    "static"
].iloc[
    0
]


finetune_row = FINAL_METHOD_TABLE[
    FINAL_METHOD_TABLE[
        "method"
    ]
    ==
    "finetune"
].iloc[
    0
]


random_row = FINAL_METHOD_TABLE[
    FINAL_METHOD_TABLE[
        "method"
    ]
    ==
    "random_replay"
].iloc[
    0
]


diverse_row = FINAL_METHOD_TABLE[
    FINAL_METHOD_TABLE[
        "method"
    ]
    ==
    "connectivity_diverse_replay"
].iloc[
    0
]


results_text = f"""
FINAL EXPERIMENTAL SUMMARY

Cohort:
3 patients (chb06, chb09, chb10), 10 future seizure events,
five optimization seeds (2026–2030), and a strict chronological
test-before-update protocol.

Across seeds, seizure-level sensitivity was
{static_row['seizure_sensitivity_mean']:.3f} for the static model,
{finetune_row['seizure_sensitivity_mean']:.3f} for fine-tuning,
{random_row['seizure_sensitivity_mean']:.3f} for random replay, and
{diverse_row['seizure_sensitivity_mean']:.3f} for connectivity-diverse replay.

Mean pooled future AUPRC was
{static_row['pooled_future_ap_mean']:.4f} ± {static_row['pooled_future_ap_sd']:.4f}
for static,
{finetune_row['pooled_future_ap_mean']:.4f} ± {finetune_row['pooled_future_ap_sd']:.4f}
for fine-tuning,
{random_row['pooled_future_ap_mean']:.4f} ± {random_row['pooled_future_ap_sd']:.4f}
for random replay, and
{diverse_row['pooled_future_ap_mean']:.4f} ± {diverse_row['pooled_future_ap_sd']:.4f}
for connectivity-diverse replay.

Mean false alarms/hour were
{static_row['false_alarms_per_hour_mean']:.3f} for static,
{finetune_row['false_alarms_per_hour_mean']:.3f} for fine-tuning,
{random_row['false_alarms_per_hour_mean']:.3f} for random replay, and
{diverse_row['false_alarms_per_hour_mean']:.3f} for connectivity-diverse replay.

Connectivity-diverse replay increased the mean pairwise
connectivity distance of the replay memory by
{memory_diversity_gain_percent:.1f}% relative to random replay
({diverse_memory_distance:.3f} versus {random_memory_distance:.3f}).

However, this increased representation-space diversity did not
translate into a consistent forecasting advantage over random replay:
the five-seed mean AUPRC difference (diverse minus random) was
{AP_DIFF:+.4f}, the false-alarm/hour difference was {FAR_DIFF:+.4f},
and the mean backward-transfer difference was {BWT_DIFF:+.4f}.

Therefore, under this cohort, architecture, replay budget, and
chronological protocol, the experiments support the claim that
connectivity-aware exemplar selection produces a substantially more
diverse replay memory, but they do NOT support a claim that it
consistently improves seizure-forecasting performance over random replay.

Because only three patients satisfied the frozen cohort criteria,
patient-level inferential comparisons have low statistical power and
should be interpreted cautiously.
""".strip()


(
    FINAL_ROOT
    / "RESULTS_SUMMARY.txt"
).write_text(
    results_text
)


# ============================================================
# 15. FINAL REPORT
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "STAGE 10 FINAL ANALYSIS COMPLETE"
)

print(
    "=" * 100
)


print(
    "\nFINAL METHOD TABLE"
)

display(
    FINAL_METHOD_TABLE
)


print(
    "\nPATIENT × METHOD TABLE"
)

display(
    PATIENT_METHOD
)


print(
    "\nRANDOM VS DIVERSE — PATIENT-LEVEL PAIRED ANALYSIS"
)

display(
    RANDOM_VS_DIVERSE_PATIENT
)


print(
    "\nMEMORY DIVERSITY EFFECT"
)

display(
    MEMORY_EFFECT
)


print(
    "\nRESULTS TEXT"
)

print(
    results_text
)


print(
    "\nSaved all final tables and figures to:"
)

print(
    FINAL_ROOT
)


print(
    "\n=============================================="
)

print(
    "CORE EXPERIMENTS FINISHED"
)

print(
    "=============================================="
)


In [ ]:
# ============================================================
# Patient-specific temporal EEG state knowledge graphs
# ============================================================
# State centroids are learned from the initial training history only.
# Transitions are counted only between consecutive 5-s windows in
# the same EDF file.

import networkx as nx
from sklearn.cluster import MiniBatchKMeans

N_STATES = 12
KG_RANDOM_STATE = 2026

KG_BY_PATIENT = {}
kg_summary_rows = []

for patient_number, patient in enumerate(PATIENTS, start=1):
    train_idx = np.asarray(
        PATIENT_DATA[patient]["train_idx"],
        dtype=np.int64,
    )

    train_meta = (
        FINAL_SPLIT.iloc[train_idx]
        .copy()
        .sort_values(
            ["start_abs_sec", "file", "window_index_in_file"]
        )
    )

    ordered_idx = train_meta["row_id"].to_numpy(dtype=np.int64)
    X = CONNECTIVITY[ordered_idx].astype(np.float32)
    y = train_meta["label"].to_numpy(dtype=np.int64)

    kmeans = MiniBatchKMeans(
        n_clusters=N_STATES,
        random_state=KG_RANDOM_STATE + patient_number,
        batch_size=1024,
        n_init=10,
    )
    state_labels = kmeans.fit_predict(X)

    state_stats = {}
    for state in range(N_STATES):
        mask = state_labels == state
        state_y = y[mask]
        n_total = int(mask.sum())
        n_preictal = int((state_y == 1).sum())
        n_interictal = int((state_y == 0).sum())

        state_stats[state] = {
            "count": n_total,
            "preictal": n_preictal,
            "interictal": n_interictal,
            "preictal_rate": (
                float(n_preictal / n_total)
                if n_total > 0
                else 0.0
            ),
        }

    transition_matrix = np.zeros(
        (N_STATES, N_STATES),
        dtype=np.int64,
    )

    previous_row = None
    previous_state = None

    for (_, row), state in zip(
        train_meta.iterrows(),
        state_labels,
    ):
        if previous_row is not None:
            same_file = str(row["file"]) == str(previous_row["file"])
            expected_stride = np.isclose(
                float(row["start_abs_sec"])
                - float(previous_row["start_abs_sec"]),
                5.0,
                atol=0.01,
            )

            if same_file and expected_stride:
                transition_matrix[
                    int(previous_state),
                    int(state),
                ] += 1

        previous_row = row
        previous_state = state

    graph = nx.MultiDiGraph()
    patient_node = f"patient:{patient}"
    preictal_node = "condition:preictal"
    interictal_node = "condition:interictal"

    graph.add_node(
        patient_node,
        entity_type="patient",
        patient=patient,
    )
    graph.add_node(preictal_node, entity_type="condition")
    graph.add_node(interictal_node, entity_type="condition")

    for state in range(N_STATES):
        state_node = f"{patient}:state:{state}"
        stats = state_stats[state]

        graph.add_node(
            state_node,
            entity_type="connectivity_state",
            state_id=int(state),
            count=int(stats["count"]),
            preictal_count=int(stats["preictal"]),
            interictal_count=int(stats["interictal"]),
            preictal_rate=float(stats["preictal_rate"]),
        )

        graph.add_edge(
            patient_node,
            state_node,
            relation="exhibits",
            weight=int(stats["count"]),
        )

        if stats["preictal"] > 0:
            graph.add_edge(
                state_node,
                preictal_node,
                relation="associated_with",
                weight=int(stats["preictal"]),
            )

        if stats["interictal"] > 0:
            graph.add_edge(
                state_node,
                interictal_node,
                relation="associated_with",
                weight=int(stats["interictal"]),
            )

    n_transition_relations = 0
    for source in range(N_STATES):
        for target in range(N_STATES):
            count = int(transition_matrix[source, target])
            if count > 0:
                graph.add_edge(
                    f"{patient}:state:{source}",
                    f"{patient}:state:{target}",
                    relation="transitions_to",
                    weight=count,
                )
                n_transition_relations += 1

    KG_BY_PATIENT[patient] = {
        "graph": graph,
        "kmeans": kmeans,
        "state_stats": state_stats,
        "transition_matrix": transition_matrix,
    }

    kg_summary_rows.append(
        {
            "patient": patient,
            "knowledge_states": N_STATES,
            "kg_nodes": graph.number_of_nodes(),
            "kg_edges": graph.number_of_edges(),
            "transition_relations": n_transition_relations,
        }
    )

KG_SUMMARY = pd.DataFrame(kg_summary_rows)
display(KG_SUMMARY)


In [ ]:
# ============================================================
# Knowledge-graph-guided replay experiment
# ============================================================

from pathlib import Path
import random, time
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score

REQ = [
    "FINAL_SPLIT", "CONNECTIVITY", "PATIENT_DATA", "PATIENTS",
    "train_initial_model", "make_model", "evaluate_set",
    "train_fixed_steps_v2", "KG_BY_PATIENT"
]
missing = [x for x in REQ if x not in globals()]
if missing:
    raise RuntimeError(f"Run the previous cells first. Missing: {missing}")

SEEDS_KG = (2026, 2027, 2028, 2029, 2030)
KG_METHODS = ("state_kg_replay", "temporal_kg_replay")
N_STATES = 12
MEMORY_PER_CLASS = 64
MEMORY_TOTAL = 128
REFRACTORY_SEC = 35 * 60

ROOT = Path(
    "/content/drive/MyDrive/EEG_Research/"
    "continual_graph_forecasting"
)

BASELINE_ROOT = (
    ROOT
    / "experiments"
    / "continual_multiseed_final_v1"
)

OUT_ROOT = (
    ROOT
    / "experiments"
    / "kg_guided_replay_final_v1"
)

OUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    random.seed(
        int(seed)
    )

    np.random.seed(
        int(seed)
    )

    torch.manual_seed(
        int(seed)
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            int(seed)
        )


# ============================================================
# STATE ASSIGNMENT
# ============================================================

def assign_states(
    patient,
    idx
):

    idx = np.asarray(
        idx,
        dtype=np.int64
    )

    if len(idx) == 0:
        return np.empty(
            0,
            dtype=np.int64
        )

    return (
        KG_BY_PATIENT[
            patient
        ][
            "kmeans"
        ]
        .predict(
            CONNECTIVITY[
                idx
            ].astype(
                np.float32,
                copy=False
            )
        )
        .astype(
            np.int64
        )
    )


# ============================================================
# BUILD CURRENT KNOWLEDGE FROM REVEALED HISTORY ONLY
# ============================================================

def build_kg_stats(
    patient,
    seen_idx
):

    seen_idx = np.unique(
        np.asarray(
            seen_idx,
            dtype=np.int64
        )
    )

    meta = (
        FINAL_SPLIT.iloc[
            seen_idx
        ]
        .copy()
        .sort_values(
            [
                "start_abs_sec",
                "file",
                "window_index_in_file"
            ]
        )
    )

    ordered_idx = (
        meta[
            "row_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    states = assign_states(
        patient,
        ordered_idx
    )

    y = (
        meta[
            "label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    # --------------------------------------------------------
    # State -> preictal/interictal associations
    # --------------------------------------------------------

    pos = np.zeros(
        N_STATES,
        dtype=np.float64
    )

    total = np.zeros(
        N_STATES,
        dtype=np.float64
    )


    for s in range(
        N_STATES
    ):

        mask = (
            states
            ==
            s
        )

        total[
            s
        ] = mask.sum()

        pos[
            s
        ] = (
            y[
                mask
            ]
            ==
            1
        ).sum()


    # Laplace-smoothed historical P(preictal | state)
    p_preictal = (
        pos
        +
        1.0
    ) / (
        total
        +
        2.0
    )


    # --------------------------------------------------------
    # State -> State temporal relations
    # --------------------------------------------------------

    transitions = np.zeros(
        (
            N_STATES,
            N_STATES
        ),
        dtype=np.float64
    )

    previous_row = None

    previous_state = None


    for (
        (_, row),
        state
    ) in zip(
        meta.iterrows(),
        states
    ):

        if previous_row is not None:

            same_file = (
                str(
                    row[
                        "file"
                    ]
                )
                ==
                str(
                    previous_row[
                        "file"
                    ]
                )
            )

            consecutive = np.isclose(
                float(
                    row[
                        "start_abs_sec"
                    ]
                )
                -
                float(
                    previous_row[
                        "start_abs_sec"
                    ]
                ),
                5.0,
                atol=0.01
            )


            if (
                same_file
                and
                consecutive
            ):

                transitions[
                    int(
                        previous_state
                    ),
                    int(
                        state
                    )
                ] += 1.0


        previous_row = row

        previous_state = int(
            state
        )


    row_sum = transitions.sum(
        axis=1,
        keepdims=True
    )


    transition_prob = np.divide(
        transitions,
        row_sum,
        out=np.zeros_like(
            transitions
        ),
        where=row_sum > 0
    )


    state_by_row = dict(
        zip(
            ordered_idx.tolist(),
            states.tolist()
        )
    )


    return {
        "p_preictal":
            p_preictal,

        "transition_prob":
            transition_prob,

        "state_by_row":
            state_by_row
    }


# ============================================================
# MEMORY SLOT ALLOCATION
# ============================================================

def allocate_slots(
    weights,
    capacities,
    total_slots
):

    weights = np.asarray(
        weights,
        dtype=np.float64
    )

    capacities = np.asarray(
        capacities,
        dtype=np.int64
    )


    if capacities.sum() < total_slots:

        raise RuntimeError(
            "Not enough samples to fill KG memory."
        )


    allocation = np.zeros_like(
        capacities
    )


    active = (
        capacities
        >
        0
    )


    safe_weights = np.where(
        active,
        np.maximum(
            weights,
            1e-12
        ),
        0.0
    )


    while (
        allocation.sum()
        <
        total_slots
    ):

        remaining = (
            total_slots
            -
            allocation.sum()
        )


        available = (
            allocation
            <
            capacities
        )


        w = np.where(
            available,
            safe_weights,
            0.0
        )


        if w.sum() <= 0:

            candidates = np.flatnonzero(
                available
            )

            for s in candidates[
                :remaining
            ]:

                allocation[
                    s
                ] += 1

            continue


        raw = (
            remaining
            *
            (
                w
                /
                w.sum()
            )
        )


        add = np.floor(
            raw
        ).astype(
            np.int64
        )


        add = np.minimum(
            add,
            capacities
            -
            allocation
        )


        if add.sum() > 0:

            allocation += add

            continue


        raw[
            ~available
        ] = -np.inf


        s = int(
            np.argmax(
                raw
            )
        )


        allocation[
            s
        ] += 1


    return allocation


# ============================================================
# KG MEMORY SELECTION
# ============================================================

def make_kg_memory(
    patient,
    seen_idx,
    temporal
):

    seen_idx = np.unique(
        np.asarray(
            seen_idx,
            dtype=np.int64
        )
    )


    labels = (
        FINAL_SPLIT.iloc[
            seen_idx
        ][
            "label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    kg = build_kg_stats(
        patient,
        seen_idx
    )


    states = np.asarray(
        [
            kg[
                "state_by_row"
            ][
                int(i)
            ]

            for i in seen_idx
        ],
        dtype=np.int64
    )


    centers = (
        KG_BY_PATIENT[
            patient
        ][
            "kmeans"
        ]
        .cluster_centers_
        .astype(
            np.float32,
            copy=False
        )
    )


    selected = []


    # --------------------------------------------------------
    # Select exactly 64 samples from each class
    # --------------------------------------------------------

    for cls in (
        0,
        1
    ):

        if cls == 1:

            current_score = (
                kg[
                    "p_preictal"
                ].copy()
            )

        else:

            current_score = (
                1.0
                -
                kg[
                    "p_preictal"
                ]
            )


        # ----------------------------------------------------
        # State-only KG:
        # only current state's class association
        #
        # Temporal KG:
        # current-state association
        # +
        # expected class association of next states
        # ----------------------------------------------------

        if temporal:

            next_score = (
                kg[
                    "transition_prob"
                ]
                @
                current_score
            )


            state_score = (
                current_score
                +
                next_score
            )

        else:

            state_score = (
                current_score
            )


        capacities = np.zeros(
            N_STATES,
            dtype=np.int64
        )


        pools = {}


        for s in range(
            N_STATES
        ):

            mask = (
                (
                    labels
                    ==
                    cls
                )
                &
                (
                    states
                    ==
                    s
                )
            )


            pool = seen_idx[
                mask
            ]


            capacities[
                s
            ] = len(
                pool
            )


            if len(
                pool
            ) == 0:

                pools[
                    s
                ] = []

                continue


            # -----------------------------------------------
            # Prefer representative examples near the
            # connectivity-state prototype.
            # -----------------------------------------------

            x = CONNECTIVITY[
                pool
            ].astype(
                np.float32,
                copy=False
            )


            d2 = np.sum(
                (
                    x
                    -
                    centers[
                        s
                    ][
                        None,
                        :
                    ]
                )
                ** 2,
                axis=1,
                dtype=np.float64
            )


            order = np.lexsort(
                (
                    pool,
                    d2
                )
            )


            pools[
                s
            ] = (
                pool[
                    order
                ]
                .astype(
                    np.int64
                )
                .tolist()
            )


        allocation = allocate_slots(
            weights=state_score,
            capacities=capacities,
            total_slots=MEMORY_PER_CLASS
        )


        for s in range(
            N_STATES
        ):

            selected.extend(
                pools[
                    s
                ][
                    :
                    int(
                        allocation[
                            s
                        ]
                    )
                ]
            )


    memory = np.asarray(
        selected,
        dtype=np.int64
    )


    memory_y = (
        FINAL_SPLIT.iloc[
            memory
        ][
            "label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    if (
        len(
            memory
        )
        !=
        MEMORY_TOTAL
        or
        (
            memory_y
            ==
            0
        ).sum()
        !=
        MEMORY_PER_CLASS
        or
        (
            memory_y
            ==
            1
        ).sum()
        !=
        MEMORY_PER_CLASS
    ):

        raise RuntimeError(
            f"{patient}: KG memory balance invariant failed."
        )


    return memory


# ============================================================
# EXACT FUTURE CLEAN EXPOSURE
# ============================================================

def exact_clean_hours(
    df
):

    clean = df[
        df[
            "label"
        ]
        ==
        0
    ].copy()


    total = 0.0


    for (
        _,
        _
    ), g in clean.groupby(
        [
            "patient",
            "file"
        ],
        sort=False
    ):

        intervals = sorted(
            zip(
                g[
                    "start_abs_sec"
                ].astype(
                    float
                ),
                g[
                    "end_abs_sec"
                ].astype(
                    float
                )
            )
        )


        if not intervals:
            continue


        a, b = intervals[
            0
        ]


        for x, y in intervals[
            1:
        ]:

            if x <= b:

                b = max(
                    b,
                    y
                )

            else:

                total += (
                    b
                    -
                    a
                )

                a, b = (
                    x,
                    y
                )


        total += (
            b
            -
            a
        )


    return (
        total
        /
        3600.0
    )


# ============================================================
# ALARM METRICS
# ============================================================

def alarm_metrics_local(
    df
):

    detected = set()


    events = set(
        df.loc[
            df[
                "label"
            ]
            ==
            1,
            "event_id"
        ]
        .astype(
            str
        )
    )


    events.discard(
        ""
    )


    false_alarms = 0


    for patient, g in df.groupby(
        "patient",
        sort=False
    ):

        g = g.sort_values(
            [
                "start_abs_sec",
                "file",
                "window_index_in_file"
            ]
        )


        refractory_until = (
            -np.inf
        )


        for _, row in g.iterrows():

            if int(
                row[
                    "predicted_positive"
                ]
            ) != 1:

                continue


            t = float(
                row[
                    "start_abs_sec"
                ]
            )


            if (
                t
                <
                refractory_until
            ):

                continue


            if int(
                row[
                    "label"
                ]
            ) == 1:

                event_id = str(
                    row[
                        "event_id"
                    ]
                )


                if event_id != "":

                    detected.add(
                        event_id
                    )


            else:

                false_alarms += 1


            refractory_until = (
                t
                +
                REFRACTORY_SEC
            )


    clean_hours = exact_clean_hours(
        df
    )


    return {
        "future_seizures":
            len(
                events
            ),

        "detected_seizures":
            len(
                detected
            ),

        "seizure_sensitivity":
            (
                len(
                    detected
                )
                /
                len(
                    events
                )

                if events

                else np.nan
            ),

        "false_alarms":
            false_alarms,

        "clean_exposure_hours":
            clean_hours,

        "false_alarms_per_hour":
            (
                false_alarms
                /
                clean_hours

                if clean_hours > 0

                else np.nan
            )
    }


# ============================================================
# RUN FIVE SEEDS
# ============================================================

start = time.time()

prediction_parts = []

retention_rows = []

memory_rows = []


for seed in SEEDS_KG:

    print(
        "\n"
        +
        "#" * 80
    )

    print(
        "SEED",
        seed
    )

    print(
        "#" * 80
    )


    for patient_number, patient in enumerate(
        PATIENTS,
        start=1
    ):

        patient_seed = (
            int(
                seed
            )
            +
            patient_number
            *
            1000
        )


        print(
            "\n",
            patient
        )


        # ----------------------------------------------------
        # Same initial model training as previous experiments
        # ----------------------------------------------------

        (
            initial_model,
            threshold,
            _,
            _,
            _
        ) = train_initial_model(
            patient,
            patient_seed
        )


        initial_state = {
            k:
                v.detach()
                .cpu()
                .clone()

            for k, v
            in initial_model
            .state_dict()
            .items()
        }


        models = {}


        for method in KG_METHODS:

            model = make_model()

            model.load_state_dict(
                initial_state
            )

            models[
                method
            ] = model


        seen_idx = np.asarray(
            PATIENT_DATA[
                patient
            ][
                "train_idx"
            ],
            dtype=np.int64
        )


        memories = {

            "state_kg_replay":
                make_kg_memory(
                    patient,
                    seen_idx,
                    temporal=False
                ),

            "temporal_kg_replay":
                make_kg_memory(
                    patient,
                    seen_idx,
                    temporal=True
                )
        }


        # ----------------------------------------------------
        # Initial retention
        # ----------------------------------------------------

        for method in KG_METHODS:

            ret = evaluate_set(
                models[
                    method
                ],
                PATIENT_DATA[
                    patient
                ][
                    "retention_idx"
                ],
                threshold
            )


            retention_rows.append(
                {
                    "seed":
                        seed,

                    "patient":
                        patient,

                    "method":
                        method,

                    "after_episode":
                        0,

                    "retention_ap":
                        ret[
                            "average_precision"
                        ]
                }
            )


        # ====================================================
        # FUTURE STREAM
        # ====================================================

        for episode in PATIENT_DATA[
            patient
        ][
            "episodes"
        ]:

            episode_index = int(
                episode[
                    "episode_index"
                ]
            )


            episode_id = str(
                episode[
                    "episode_id"
                ]
            )


            eval_idx = np.asarray(
                episode[
                    "eval_idx"
                ],
                dtype=np.int64
            )


            update_idx = np.asarray(
                episode[
                    "update_idx"
                ],
                dtype=np.int64
            )


            print(
                f"  {episode_id} | "
                f"eval={len(eval_idx):,} | "
                f"update={len(update_idx):,}"
            )


            # =================================================
            # A. TEST FIRST
            # =================================================

            for method in KG_METHODS:

                result = evaluate_set(
                    models[
                        method
                    ],
                    eval_idx,
                    threshold
                )


                meta = (
                    FINAL_SPLIT.iloc[
                        eval_idx
                    ][
                        [
                            "row_id",
                            "window_key",
                            "patient",
                            "file",
                            "window_index_in_file",
                            "start_abs_sec",
                            "end_abs_sec",
                            "event_id",
                            "label"
                        ]
                    ]
                    .copy()
                )


                meta[
                    "seed"
                ] = seed


                meta[
                    "method"
                ] = method


                meta[
                    "episode_id"
                ] = episode_id


                meta[
                    "episode_index"
                ] = episode_index


                meta[
                    "probability"
                ] = result[
                    "probabilities"
                ]


                meta[
                    "threshold"
                ] = threshold


                meta[
                    "predicted_positive"
                ] = (
                    result[
                        "probabilities"
                    ]
                    >=
                    threshold
                ).astype(
                    np.int8
                )


                prediction_parts.append(
                    meta
                )


            # =================================================
            # B. UPDATE AFTER LABELS ARE REVEALED
            # =================================================

            train_fixed_steps_v2(
                models[
                    "state_kg_replay"
                ],
                current_idx=update_idx,
                memory_idx=memories[
                    "state_kg_replay"
                ],
                seed=(
                    patient_seed
                    +
                    episode_index
                    *
                    100
                    +
                    31
                )
            )


            train_fixed_steps_v2(
                models[
                    "temporal_kg_replay"
                ],
                current_idx=update_idx,
                memory_idx=memories[
                    "temporal_kg_replay"
                ],
                seed=(
                    patient_seed
                    +
                    episode_index
                    *
                    100
                    +
                    32
                )
            )


            # -------------------------------------------------
            # Episode becomes historical only after evaluation.
            # -------------------------------------------------

            seen_idx = np.unique(
                np.concatenate(
                    [
                        seen_idx,
                        update_idx
                    ]
                )
            )


            memories[
                "state_kg_replay"
            ] = make_kg_memory(
                patient,
                seen_idx,
                temporal=False
            )


            memories[
                "temporal_kg_replay"
            ] = make_kg_memory(
                patient,
                seen_idx,
                temporal=True
            )


            # -------------------------------------------------
            # Audit memory and retention
            # -------------------------------------------------

            for method in KG_METHODS:

                memory = memories[
                    method
                ]


                memory_states = assign_states(
                    patient,
                    memory
                )


                memory_rows.append(
                    {
                        "seed":
                            seed,

                        "patient":
                            patient,

                        "method":
                            method,

                        "after_episode":
                            episode_index,

                        "memory_size":
                            len(
                                memory
                            ),

                        "state_coverage":
                            int(
                                np.unique(
                                    memory_states
                                ).size
                            )
                    }
                )


                ret = evaluate_set(
                    models[
                        method
                    ],
                    PATIENT_DATA[
                        patient
                    ][
                        "retention_idx"
                    ],
                    threshold
                )


                retention_rows.append(
                    {
                        "seed":
                            seed,

                        "patient":
                            patient,

                        "method":
                            method,

                        "after_episode":
                            episode_index,

                        "retention_ap":
                            ret[
                                "average_precision"
                            ]
                    }
                )


# ============================================================
# COMBINE OUTPUTS
# ============================================================

KG_PRED = pd.concat(
    prediction_parts,
    ignore_index=True
)

KG_RET = pd.DataFrame(
    retention_rows
)

KG_MEMORY = pd.DataFrame(
    memory_rows
)


# ============================================================
# KG METHOD SUMMARY
# ============================================================

kg_rows = []


for seed in SEEDS_KG:

    for method in KG_METHODS:

        subset = KG_PRED[
            (
                KG_PRED[
                    "seed"
                ]
                ==
                seed
            )
            &
            (
                KG_PRED[
                    "method"
                ]
                ==
                method
            )
        ].copy()


        y = subset[
            "label"
        ].to_numpy(
            dtype=np.int64
        )


        probability = subset[
            "probability"
        ].to_numpy(
            dtype=float
        )


        alarm = alarm_metrics_local(
            subset
        )


        patient_bwt = []


        for patient in PATIENTS:

            r = (
                KG_RET[
                    (
                        KG_RET[
                            "seed"
                        ]
                        ==
                        seed
                    )
                    &
                    (
                        KG_RET[
                            "patient"
                        ]
                        ==
                        patient
                    )
                    &
                    (
                        KG_RET[
                            "method"
                        ]
                        ==
                        method
                    )
                ]
                .sort_values(
                    "after_episode"
                )
            )


            initial_ap = float(
                r.iloc[
                    0
                ][
                    "retention_ap"
                ]
            )


            final_ap = float(
                r.iloc[
                    -1
                ][
                    "retention_ap"
                ]
            )


            patient_bwt.append(
                final_ap
                -
                initial_ap
            )


        kg_rows.append(
            {
                "seed":
                    seed,

                "method":
                    method,

                "pooled_future_ap":
                    float(
                        average_precision_score(
                            y,
                            probability
                        )
                    ),

                "future_seizures":
                    alarm[
                        "future_seizures"
                    ],

                "detected_seizures":
                    alarm[
                        "detected_seizures"
                    ],

                "seizure_sensitivity":
                    alarm[
                        "seizure_sensitivity"
                    ],

                "false_alarms":
                    alarm[
                        "false_alarms"
                    ],

                "clean_exposure_hours":
                    alarm[
                        "clean_exposure_hours"
                    ],

                "false_alarms_per_hour":
                    alarm[
                        "false_alarms_per_hour"
                    ],

                "mean_backward_transfer":
                    float(
                        np.mean(
                            patient_bwt
                        )
                    )
            }
        )


KG_SEED_METHOD = pd.DataFrame(
    kg_rows
)


# ============================================================
# MERGE WITH EXISTING BASELINES
# ============================================================

baseline_file = (
    BASELINE_ROOT
    / "seed_method_summary.csv"
)


if not baseline_file.exists():

    raise FileNotFoundError(
        f"Missing Stage 9 baseline file: "
        f"{baseline_file}"
    )


BASELINE = pd.read_csv(
    baseline_file
)


columns = [
    "seed",
    "method",
    "pooled_future_ap",
    "future_seizures",
    "detected_seizures",
    "seizure_sensitivity",
    "false_alarms",
    "clean_exposure_hours",
    "false_alarms_per_hour",
    "mean_backward_transfer"
]


ALL_METHODS = pd.concat(
    [
        BASELINE[
            columns
        ].copy(),

        KG_SEED_METHOD[
            columns
        ].copy()
    ],
    ignore_index=True
)


# ============================================================
# FINAL FIVE-SEED COMPARISON
# ============================================================

FINAL_KG_COMPARISON = (
    ALL_METHODS
    .groupby(
        "method",
        as_index=False
    )
    .agg(
        future_ap_mean=(
            "pooled_future_ap",
            "mean"
        ),

        future_ap_sd=(
            "pooled_future_ap",
            "std"
        ),

        sensitivity_mean=(
            "seizure_sensitivity",
            "mean"
        ),

        false_alarms_per_hour_mean=(
            "false_alarms_per_hour",
            "mean"
        ),

        false_alarms_per_hour_sd=(
            "false_alarms_per_hour",
            "std"
        ),

        backward_transfer_mean=(
            "mean_backward_transfer",
            "mean"
        ),

        backward_transfer_sd=(
            "mean_backward_transfer",
            "std"
        )
    )
)


# ============================================================
# TEMPORAL RELATION ABLATION
# ============================================================

state_seed = (
    KG_SEED_METHOD[
        KG_SEED_METHOD[
            "method"
        ]
        ==
        "state_kg_replay"
    ]
    .set_index(
        "seed"
    )
)


temporal_seed = (
    KG_SEED_METHOD[
        KG_SEED_METHOD[
            "method"
        ]
        ==
        "temporal_kg_replay"
    ]
    .set_index(
        "seed"
    )
)


ablation_rows = []


for metric in [
    "pooled_future_ap",
    "seizure_sensitivity",
    "false_alarms_per_hour",
    "mean_backward_transfer"
]:

    difference = (
        temporal_seed[
            metric
        ]
        -
        state_seed[
            metric
        ]
    )


    ablation_rows.append(
        {
            "metric":
                metric,

            "temporal_minus_state_mean":
                float(
                    difference.mean()
                ),

            "temporal_minus_state_sd":
                float(
                    difference.std(
                        ddof=1
                    )
                ),

            "min_difference":
                float(
                    difference.min()
                ),

            "max_difference":
                float(
                    difference.max()
                )
        }
    )


KG_ABLATION = pd.DataFrame(
    ablation_rows
)


# ============================================================
# MEMORY SUMMARY
# ============================================================

KG_MEMORY_SUMMARY = (
    KG_MEMORY
    .groupby(
        "method",
        as_index=False
    )
    .agg(
        mean_memory_size=(
            "memory_size",
            "mean"
        ),

        mean_state_coverage=(
            "state_coverage",
            "mean"
        ),

        state_coverage_sd=(
            "state_coverage",
            "std"
        )
    )
)


# ============================================================
# SAVE
# ============================================================

KG_PRED.to_pickle(
    OUT_ROOT
    / "kg_predictions.pkl"
)


KG_RET.to_csv(
    OUT_ROOT
    / "kg_retention.csv",
    index=False
)


KG_MEMORY.to_csv(
    OUT_ROOT
    / "kg_memory.csv",
    index=False
)


KG_SEED_METHOD.to_csv(
    OUT_ROOT
    / "kg_seed_method.csv",
    index=False
)


FINAL_KG_COMPARISON.to_csv(
    OUT_ROOT
    / "all_methods_comparison.csv",
    index=False
)


KG_ABLATION.to_csv(
    OUT_ROOT
    / "temporal_kg_ablation.csv",
    index=False
)


KG_MEMORY_SUMMARY.to_csv(
    OUT_ROOT
    / "kg_memory_summary.csv",
    index=False
)


# ============================================================
# REPORT
# ============================================================

print(
    "\n"
    +
    "=" * 90
)

print(
    "KG-GUIDED REPLAY EXPERIMENT COMPLETE"
)

print(
    "=" * 90
)


print(
    f"Runtime: "
    f"{(time.time() - start) / 60:.1f} minutes"
)


print(
    "\nALL METHODS"
)

display(
    FINAL_KG_COMPARISON
)


print(
    "\nTEMPORAL KG ABLATION"
)

display(
    KG_ABLATION
)


print(
    "\nKG MEMORY"
)

display(
    KG_MEMORY_SUMMARY
)


print(
    "\nSaved to:"
)

print(
    OUT_ROOT
)


In [ ]:
# ============================================================
# Direct knowledge-graph guidance of seizure prediction
#
# Compares:
#   1. GNN only
#   2. GNN + KG state knowledge
#   3. GNN + KG state + transition knowledge
#
# Design:
#   - same frozen chronological split
#   - same initial GNN architecture
#   - five optimization seeds
#   - KG state vocabulary fixed from initial historical training
#   - KG statistics updated only AFTER each future episode is evaluated
#   - no future labels used before prediction
#   - fusion models trained only on the historical validation data
#   - one validation half fits the small fusion model
#   - the other validation half selects the decision threshold
# ============================================================

from pathlib import Path
import random
import time

import numpy as np
import pandas as pd

import torch

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
)


# ============================================================
# Required objects
# ============================================================

REQUIRED = [
    "FINAL_SPLIT",
    "CONNECTIVITY",
    "PATIENT_DATA",
    "PATIENTS",
    "KG_BY_PATIENT",
    "train_initial_model",
    "predict_indices",
]

missing = [
    name
    for name in REQUIRED
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run the earlier GNN and KG cells first. "
        f"Missing: {missing}"
    )


# ============================================================
# Settings
# ============================================================

SEEDS_DIRECT_KG = (
    2026,
    2027,
    2028,
    2029,
    2030,
)

METHODS_DIRECT_KG = (
    "gnn_only",
    "kg_state_guided",
    "kg_temporal_guided",
)

N_STATES_DIRECT_KG = 12

SPH_SEC_DIRECT_KG = 5 * 60
SOP_SEC_DIRECT_KG = 30 * 60
REFRACTORY_SEC_DIRECT_KG = (
    SPH_SEC_DIRECT_KG
    +
    SOP_SEC_DIRECT_KG
)

ROOT_DIRECT_KG = Path(
    "/content/drive/MyDrive/EEG_Research/"
    "continual_graph_forecasting"
)

OUT_DIRECT_KG = (
    ROOT_DIRECT_KG
    / "experiments"
    / "kg_direct_prediction_final_v1"
)

OUT_DIRECT_KG.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

def set_seed_direct_kg(seed):

    seed = int(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# 3. STATE ASSIGNMENT
# ============================================================

def assign_states_direct_kg(
    patient,
    indices
):

    indices = np.asarray(
        indices,
        dtype=np.int64
    )

    if len(indices) == 0:
        return np.empty(
            0,
            dtype=np.int64
        )

    return (
        KG_BY_PATIENT[
            patient
        ][
            "kmeans"
        ]
        .predict(
            CONNECTIVITY[
                indices
            ].astype(
                np.float32,
                copy=False
            )
        )
        .astype(
            np.int64
        )
    )


# ============================================================
# 4. BUILD KG KNOWLEDGE FROM REVEALED HISTORY ONLY
# ============================================================

def build_history_kg_direct(
    patient,
    seen_idx
):

    seen_idx = np.unique(
        np.asarray(
            seen_idx,
            dtype=np.int64
        )
    )

    if len(seen_idx) == 0:
        raise RuntimeError(
            f"{patient}: empty historical KG."
        )

    meta = (
        FINAL_SPLIT.iloc[
            seen_idx
        ]
        .copy()
        .sort_values(
            [
                "start_abs_sec",
                "file",
                "window_index_in_file",
            ]
        )
    )

    ordered_idx = (
        meta[
            "row_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    states = assign_states_direct_kg(
        patient,
        ordered_idx
    )

    labels = (
        meta[
            "label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    state_total = np.zeros(
        N_STATES_DIRECT_KG,
        dtype=np.float64
    )

    state_positive = np.zeros(
        N_STATES_DIRECT_KG,
        dtype=np.float64
    )

    for state in range(
        N_STATES_DIRECT_KG
    ):

        mask = (
            states
            ==
            state
        )

        state_total[
            state
        ] = float(
            mask.sum()
        )

        state_positive[
            state
        ] = float(
            (
                labels[
                    mask
                ]
                ==
                1
            ).sum()
        )


    # Laplace-smoothed P(preictal | state)
    p_preictal = (
        state_positive
        +
        1.0
    ) / (
        state_total
        +
        2.0
    )


    # --------------------------------------------------------
    # Temporal state transitions
    # --------------------------------------------------------

    transition_counts = np.zeros(
        (
            N_STATES_DIRECT_KG,
            N_STATES_DIRECT_KG
        ),
        dtype=np.float64
    )

    previous_row = None
    previous_state = None

    for (
        (_, row),
        state
    ) in zip(
        meta.iterrows(),
        states
    ):

        state = int(state)

        if previous_row is not None:

            same_file = (
                str(
                    row[
                        "file"
                    ]
                )
                ==
                str(
                    previous_row[
                        "file"
                    ]
                )
            )

            consecutive = np.isclose(
                float(
                    row[
                        "start_abs_sec"
                    ]
                )
                -
                float(
                    previous_row[
                        "start_abs_sec"
                    ]
                ),
                5.0,
                atol=0.01,
            )

            if (
                same_file
                and
                consecutive
            ):

                transition_counts[
                    int(
                        previous_state
                    ),
                    state
                ] += 1.0

        previous_row = row
        previous_state = state


    row_sum = transition_counts.sum(
        axis=1,
        keepdims=True
    )

    transition_prob = np.divide(
        transition_counts,
        row_sum,
        out=np.zeros_like(
            transition_counts
        ),
        where=row_sum > 0,
    )


    # Expected preictal association of next states
    next_preictal = (
        transition_prob
        @
        p_preictal
    )


    # If no outgoing transition was observed,
    # fall back to current state knowledge.
    no_outgoing = (
        row_sum.squeeze(
            1
        )
        ==
        0
    )

    next_preictal[
        no_outgoing
    ] = p_preictal[
        no_outgoing
    ]


    return {
        "p_preictal":
            p_preictal,

        "transition_prob":
            transition_prob,

        "next_preictal":
            next_preictal,

        "state_total":
            state_total,
    }


# ============================================================
# 5. KG FEATURES
# ============================================================

def kg_features_direct(
    patient,
    indices,
    kg
):

    indices = np.asarray(
        indices,
        dtype=np.int64
    )

    states = assign_states_direct_kg(
        patient,
        indices
    )

    state_risk = (
        kg[
            "p_preictal"
        ][
            states
        ]
    )

    transition_risk = (
        kg[
            "next_preictal"
        ][
            states
        ]
    )

    return (
        states,
        state_risk.astype(
            np.float64
        ),
        transition_risk.astype(
            np.float64
        ),
    )


# ============================================================
# 6. GNN LOGIT FEATURE
# ============================================================

def gnn_logit_direct(
    model,
    indices
):

    probability = predict_indices(
        model,
        indices
    ).astype(
        np.float64
    )

    probability = np.clip(
        probability,
        1e-6,
        1.0 - 1e-6
    )

    logit = np.log(
        probability
        /
        (
            1.0
            -
            probability
        )
    )

    return (
        probability,
        logit
    )


# ============================================================
# 7. SPLIT EXISTING VALIDATION DATA
#
# Half per class:
#   fusion-fit subset
#   threshold-selection subset
# ============================================================

def split_validation_direct(
    patient
):

    val_idx = np.asarray(
        PATIENT_DATA[
            patient
        ][
            "val_idx"
        ],
        dtype=np.int64
    )

    meta = (
        FINAL_SPLIT.iloc[
            val_idx
        ][
            [
                "row_id",
                "label",
                "start_abs_sec",
                "file",
                "window_index_in_file",
            ]
        ]
        .copy()
    )

    fit_rows = []
    threshold_rows = []

    for cls in (
        0,
        1
    ):

        cls_df = (
            meta[
                meta[
                    "label"
                ]
                ==
                cls
            ]
            .sort_values(
                [
                    "start_abs_sec",
                    "file",
                    "window_index_in_file",
                ]
            )
        )

        n = len(
            cls_df
        )

        if n < 4:
            raise RuntimeError(
                f"{patient}: too few validation "
                f"rows for class {cls}."
            )

        split_point = (
            n // 2
        )

        fit_rows.extend(
            cls_df.iloc[
                :split_point
            ][
                "row_id"
            ].tolist()
        )

        threshold_rows.extend(
            cls_df.iloc[
                split_point:
            ][
                "row_id"
            ].tolist()
        )


    fit_idx = np.asarray(
        sorted(
            fit_rows
        ),
        dtype=np.int64
    )

    threshold_idx = np.asarray(
        sorted(
            threshold_rows
        ),
        dtype=np.int64
    )


    for name, idx in (
        (
            "fusion-fit",
            fit_idx
        ),
        (
            "threshold",
            threshold_idx
        ),
    ):

        y = (
            FINAL_SPLIT.iloc[
                idx
            ][
                "label"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        if set(
            np.unique(
                y
            )
        ) != {
            0,
            1
        }:
            raise RuntimeError(
                f"{patient}: {name} validation "
                "subset does not contain both classes."
            )


    return (
        fit_idx,
        threshold_idx
    )


# ============================================================
# 8. THRESHOLD SELECTION
# ============================================================

def choose_threshold_direct(
    y_true,
    probabilities
):

    y_true = np.asarray(
        y_true,
        dtype=np.int64
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    )

    precision, recall, thresholds = (
        precision_recall_curve(
            y_true,
            probabilities
        )
    )

    if len(
        thresholds
    ) == 0:
        return 0.5

    precision = precision[
        :-1
    ]

    recall = recall[
        :-1
    ]

    denominator = (
        precision
        +
        recall
    )

    f1 = np.divide(
        2.0
        *
        precision
        *
        recall,
        denominator,
        out=np.zeros_like(
            denominator,
            dtype=np.float64
        ),
        where=denominator > 0,
    )

    best_value = np.nanmax(
        f1
    )

    candidates = np.flatnonzero(
        np.isclose(
            f1,
            best_value,
            rtol=0.0,
            atol=1e-12,
        )
    )

    # If F1 ties, use the higher threshold.
    best = int(
        candidates[
            -1
        ]
    )

    return float(
        thresholds[
            best
        ]
    )


# ============================================================
# 9. FIT KG-GUIDED PREDICTION HEADS
# ============================================================

def fit_fusion_heads_direct(
    patient,
    model,
    historical_idx,
    seed
):

    fit_idx, threshold_idx = (
        split_validation_direct(
            patient
        )
    )

    kg = build_history_kg_direct(
        patient,
        historical_idx
    )


    # --------------------------------------------------------
    # Fusion fitting subset
    # --------------------------------------------------------

    _, fit_base_logit = (
        gnn_logit_direct(
            model,
            fit_idx
        )
    )

    (
        _,
        fit_state_risk,
        fit_transition_risk,
    ) = kg_features_direct(
        patient,
        fit_idx,
        kg
    )


    y_fit = (
        FINAL_SPLIT.iloc[
            fit_idx
        ][
            "label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    # --------------------------------------------------------
    # GNN + state knowledge
    # --------------------------------------------------------

    X_state_fit = np.column_stack(
        [
            fit_base_logit,
            fit_state_risk,
        ]
    )

    state_head = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            solver="lbfgs",
            max_iter=2000,
            random_state=int(
                seed
            ),
        )
    )

    state_head.fit(
        X_state_fit,
        y_fit
    )


    # --------------------------------------------------------
    # GNN + state + transition knowledge
    # --------------------------------------------------------

    X_temporal_fit = np.column_stack(
        [
            fit_base_logit,
            fit_state_risk,
            fit_transition_risk,
        ]
    )

    temporal_head = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0,
            solver="lbfgs",
            max_iter=2000,
            random_state=int(
                seed
            ),
        )
    )

    temporal_head.fit(
        X_temporal_fit,
        y_fit
    )


    # --------------------------------------------------------
    # Independent threshold subset
    # --------------------------------------------------------

    threshold_base_prob, threshold_base_logit = (
        gnn_logit_direct(
            model,
            threshold_idx
        )
    )

    (
        _,
        threshold_state_risk,
        threshold_transition_risk,
    ) = kg_features_direct(
        patient,
        threshold_idx,
        kg
    )

    y_threshold = (
        FINAL_SPLIT.iloc[
            threshold_idx
        ][
            "label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    state_threshold_prob = (
        state_head.predict_proba(
            np.column_stack(
                [
                    threshold_base_logit,
                    threshold_state_risk,
                ]
            )
        )[
            :,
            1
        ]
    )


    temporal_threshold_prob = (
        temporal_head.predict_proba(
            np.column_stack(
                [
                    threshold_base_logit,
                    threshold_state_risk,
                    threshold_transition_risk,
                ]
            )
        )[
            :,
            1
        ]
    )


    thresholds = {
        "gnn_only":
            choose_threshold_direct(
                y_threshold,
                threshold_base_prob
            ),

        "kg_state_guided":
            choose_threshold_direct(
                y_threshold,
                state_threshold_prob
            ),

        "kg_temporal_guided":
            choose_threshold_direct(
                y_threshold,
                temporal_threshold_prob
            ),
    }


    return {
        "state_head":
            state_head,

        "temporal_head":
            temporal_head,

        "thresholds":
            thresholds,

        "fit_idx":
            fit_idx,

        "threshold_idx":
            threshold_idx,
    }


# ============================================================
# 10. PREDICT ONE FUTURE EPISODE
# ============================================================

def predict_direct_methods(
    patient,
    model,
    fusion,
    kg,
    eval_idx
):

    base_prob, base_logit = (
        gnn_logit_direct(
            model,
            eval_idx
        )
    )

    (
        states,
        state_risk,
        transition_risk,
    ) = kg_features_direct(
        patient,
        eval_idx,
        kg
    )


    state_prob = (
        fusion[
            "state_head"
        ]
        .predict_proba(
            np.column_stack(
                [
                    base_logit,
                    state_risk,
                ]
            )
        )[
            :,
            1
        ]
    )


    temporal_prob = (
        fusion[
            "temporal_head"
        ]
        .predict_proba(
            np.column_stack(
                [
                    base_logit,
                    state_risk,
                    transition_risk,
                ]
            )
        )[
            :,
            1
        ]
    )


    return {
        "states":
            states,

        "state_risk":
            state_risk,

        "transition_risk":
            transition_risk,

        "probabilities": {
            "gnn_only":
                base_prob,

            "kg_state_guided":
                state_prob,

            "kg_temporal_guided":
                temporal_prob,
        }
    }


# ============================================================
# 11. EXACT CLEAN EXPOSURE
# ============================================================

def exact_clean_hours_direct(
    df
):

    clean = df[
        df[
            "label"
        ]
        ==
        0
    ].copy()

    total_seconds = 0.0

    for (
        _patient,
        _file
    ), group in clean.groupby(
        [
            "patient",
            "file"
        ],
        sort=False
    ):

        intervals = sorted(
            zip(
                group[
                    "start_abs_sec"
                ].astype(
                    float
                ),
                group[
                    "end_abs_sec"
                ].astype(
                    float
                )
            )
        )

        if not intervals:
            continue

        current_start, current_end = (
            intervals[
                0
            ]
        )

        for start, end in intervals[
            1:
        ]:

            if start <= current_end:

                current_end = max(
                    current_end,
                    end
                )

            else:

                total_seconds += (
                    current_end
                    -
                    current_start
                )

                current_start = start
                current_end = end

        total_seconds += (
            current_end
            -
            current_start
        )


    return float(
        total_seconds
        /
        3600.0
    )


# ============================================================
# 12. ALARM METRICS
# ============================================================

def alarm_metrics_direct(
    prediction_df
):

    prediction_df = (
        prediction_df
        .sort_values(
            [
                "patient",
                "start_abs_sec",
                "file",
                "window_index_in_file",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    all_events = (
        set(
            prediction_df.loc[
                prediction_df[
                    "label"
                ]
                ==
                1,
                "event_id"
            ]
            .astype(
                str
            )
        )
        -
        {""}
    )

    detected_events = set()
    false_alarms = 0
    accepted_alarms = 0

    for patient, group in (
        prediction_df.groupby(
            "patient",
            sort=False
        )
    ):

        group = group.sort_values(
            [
                "start_abs_sec",
                "file",
                "window_index_in_file",
            ]
        )

        refractory_until = (
            -np.inf
        )

        for _, row in group.iterrows():

            if int(
                row[
                    "predicted_positive"
                ]
            ) != 1:
                continue

            alarm_time = float(
                row[
                    "start_abs_sec"
                ]
            )

            if (
                alarm_time
                <
                refractory_until
            ):
                continue

            accepted_alarms += 1

            if int(
                row[
                    "label"
                ]
            ) == 1:

                event_id = str(
                    row[
                        "event_id"
                    ]
                )

                if event_id != "":
                    detected_events.add(
                        event_id
                    )

            else:

                false_alarms += 1

            refractory_until = (
                alarm_time
                +
                REFRACTORY_SEC_DIRECT_KG
            )


    clean_hours = (
        exact_clean_hours_direct(
            prediction_df
        )
    )


    sensitivity = (
        len(
            detected_events
        )
        /
        len(
            all_events
        )
        if all_events
        else np.nan
    )


    far_h = (
        false_alarms
        /
        clean_hours
        if clean_hours > 0
        else np.nan
    )


    return {
        "future_seizures":
            len(
                all_events
            ),

        "detected_seizures":
            len(
                detected_events
            ),

        "seizure_sensitivity":
            float(
                sensitivity
            ),

        "false_alarms":
            int(
                false_alarms
            ),

        "false_alarms_per_hour":
            float(
                far_h
            ),

        "clean_exposure_hours":
            float(
                clean_hours
            ),

        "accepted_alarms":
            int(
                accepted_alarms
            ),
    }


# ============================================================
# 13. RUN FIVE SEEDS
# ============================================================

experiment_start = time.time()

prediction_parts = []
threshold_rows = []
episode_rows = []


for seed in SEEDS_DIRECT_KG:

    print(
        "\n"
        +
        "#" * 90
    )

    print(
        "SEED",
        seed
    )

    print(
        "#" * 90
    )


    for patient_number, patient in enumerate(
        PATIENTS,
        start=1
    ):

        print(
            "\n"
            +
            "=" * 70
        )

        print(
            patient
        )

        print(
            "=" * 70
        )


        patient_seed = (
            int(
                seed
            )
            +
            patient_number
            *
            1000
        )


        set_seed_direct_kg(
            patient_seed
        )


        # ----------------------------------------------------
        # Same initial GNN training procedure.
        # GNN stays fixed during the future stream.
        # This isolates the contribution of KG guidance.
        # ----------------------------------------------------

        (
            model,
            _old_threshold,
            best_val_ap,
            best_epoch,
            _
        ) = train_initial_model(
            patient,
            patient_seed
        )


        seen_idx = np.asarray(
            PATIENT_DATA[
                patient
            ][
                "train_idx"
            ],
            dtype=np.int64
        )


        fusion = fit_fusion_heads_direct(
            patient,
            model,
            seen_idx,
            seed=patient_seed
        )


        for method in METHODS_DIRECT_KG:

            threshold_rows.append(
                {
                    "seed":
                        seed,

                    "patient":
                        patient,

                    "method":
                        method,

                    "threshold":
                        fusion[
                            "thresholds"
                        ][
                            method
                        ],

                    "best_gnn_validation_ap":
                        best_val_ap,

                    "best_gnn_epoch":
                        best_epoch,
                }
            )


        # ====================================================
        # FUTURE STREAM
        # ====================================================

        for episode in PATIENT_DATA[
            patient
        ][
            "episodes"
        ]:

            episode_id = str(
                episode[
                    "episode_id"
                ]
            )

            episode_index = int(
                episode[
                    "episode_index"
                ]
            )

            eval_idx = np.asarray(
                episode[
                    "eval_idx"
                ],
                dtype=np.int64
            )

            update_idx = np.asarray(
                episode[
                    "update_idx"
                ],
                dtype=np.int64
            )


            # KG uses only information already revealed
            # before the current future episode.
            kg = build_history_kg_direct(
                patient,
                seen_idx
            )


            result = predict_direct_methods(
                patient,
                model,
                fusion,
                kg,
                eval_idx
            )


            labels = (
                FINAL_SPLIT.iloc[
                    eval_idx
                ][
                    "label"
                ]
                .to_numpy(
                    dtype=np.int64
                )
            )


            print(
                f"  {episode_id} | "
                f"eval={len(eval_idx):,} | "
                f"historical={len(seen_idx):,}"
            )


            for method in METHODS_DIRECT_KG:

                probabilities = np.asarray(
                    result[
                        "probabilities"
                    ][
                        method
                    ],
                    dtype=np.float64
                )

                threshold = float(
                    fusion[
                        "thresholds"
                    ][
                        method
                    ]
                )


                if np.unique(
                    labels
                ).size == 2:

                    episode_ap = float(
                        average_precision_score(
                            labels,
                            probabilities
                        )
                    )

                else:

                    episode_ap = np.nan


                episode_brier = float(
                    brier_score_loss(
                        labels,
                        probabilities
                    )
                )


                episode_rows.append(
                    {
                        "seed":
                            seed,

                        "patient":
                            patient,

                        "method":
                            method,

                        "episode_id":
                            episode_id,

                        "episode_index":
                            episode_index,

                        "n_eval":
                            len(
                                eval_idx
                            ),

                        "average_precision":
                            episode_ap,

                        "brier":
                            episode_brier,
                    }
                )


                meta = (
                    FINAL_SPLIT.iloc[
                        eval_idx
                    ][
                        [
                            "row_id",
                            "window_key",
                            "patient",
                            "file",
                            "window_index_in_file",
                            "start_abs_sec",
                            "end_abs_sec",
                            "event_id",
                            "label",
                        ]
                    ]
                    .copy()
                )


                meta[
                    "seed"
                ] = seed

                meta[
                    "method"
                ] = method

                meta[
                    "episode_id"
                ] = episode_id

                meta[
                    "episode_index"
                ] = episode_index

                meta[
                    "probability"
                ] = probabilities

                meta[
                    "threshold"
                ] = threshold

                meta[
                    "predicted_positive"
                ] = (
                    probabilities
                    >=
                    threshold
                ).astype(
                    np.int8
                )

                meta[
                    "kg_state"
                ] = result[
                    "states"
                ]

                meta[
                    "kg_state_preictal_risk"
                ] = result[
                    "state_risk"
                ]

                meta[
                    "kg_transition_preictal_risk"
                ] = result[
                    "transition_risk"
                ]


                prediction_parts.append(
                    meta
                )


            # Strict test-before-update.
            # Only now does current episode enter KG history.
            seen_idx = np.unique(
                np.concatenate(
                    [
                        seen_idx,
                        update_idx
                    ]
                )
            )


# ============================================================
# 14. COMBINE OUTPUTS
# ============================================================

DIRECT_KG_PRED = pd.concat(
    prediction_parts,
    ignore_index=True
)

DIRECT_KG_THRESHOLDS = pd.DataFrame(
    threshold_rows
)

DIRECT_KG_EPISODES = pd.DataFrame(
    episode_rows
)


# ============================================================
# 15. SEED × METHOD RESULTS
# ============================================================

seed_method_rows = []


for seed in SEEDS_DIRECT_KG:

    for method in METHODS_DIRECT_KG:

        subset = DIRECT_KG_PRED[
            (
                DIRECT_KG_PRED[
                    "seed"
                ]
                ==
                seed
            )
            &
            (
                DIRECT_KG_PRED[
                    "method"
                ]
                ==
                method
            )
        ].copy()


        y = (
            subset[
                "label"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        probability = (
            subset[
                "probability"
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        pooled_ap = float(
            average_precision_score(
                y,
                probability
            )
        )


        pooled_brier = float(
            brier_score_loss(
                y,
                probability
            )
        )


        alarm = alarm_metrics_direct(
            subset
        )


        seed_method_rows.append(
            {
                "seed":
                    seed,

                "method":
                    method,

                "pooled_future_ap":
                    pooled_ap,

                "pooled_future_brier":
                    pooled_brier,

                "future_seizures":
                    alarm[
                        "future_seizures"
                    ],

                "detected_seizures":
                    alarm[
                        "detected_seizures"
                    ],

                "seizure_sensitivity":
                    alarm[
                        "seizure_sensitivity"
                    ],

                "false_alarms":
                    alarm[
                        "false_alarms"
                    ],

                "false_alarms_per_hour":
                    alarm[
                        "false_alarms_per_hour"
                    ],

                "clean_exposure_hours":
                    alarm[
                        "clean_exposure_hours"
                    ],

                "accepted_alarms":
                    alarm[
                        "accepted_alarms"
                    ],
            }
        )


DIRECT_KG_SEED_METHOD = pd.DataFrame(
    seed_method_rows
)


# ============================================================
# 16. FIVE-SEED SUMMARY
# ============================================================

DIRECT_KG_SUMMARY = (
    DIRECT_KG_SEED_METHOD
    .groupby(
        "method",
        as_index=False
    )
    .agg(
        future_ap_mean=(
            "pooled_future_ap",
            "mean"
        ),

        future_ap_sd=(
            "pooled_future_ap",
            "std"
        ),

        brier_mean=(
            "pooled_future_brier",
            "mean"
        ),

        brier_sd=(
            "pooled_future_brier",
            "std"
        ),

        sensitivity_mean=(
            "seizure_sensitivity",
            "mean"
        ),

        sensitivity_sd=(
            "seizure_sensitivity",
            "std"
        ),

        false_alarms_per_hour_mean=(
            "false_alarms_per_hour",
            "mean"
        ),

        false_alarms_per_hour_sd=(
            "false_alarms_per_hour",
            "std"
        ),
    )
)


# ============================================================
# 17. DESCRIPTIVE PAIRED DIFFERENCES
# ============================================================

base = (
    DIRECT_KG_SEED_METHOD[
        DIRECT_KG_SEED_METHOD[
            "method"
        ]
        ==
        "gnn_only"
    ]
    .set_index(
        "seed"
    )
)

state = (
    DIRECT_KG_SEED_METHOD[
        DIRECT_KG_SEED_METHOD[
            "method"
        ]
        ==
        "kg_state_guided"
    ]
    .set_index(
        "seed"
    )
)

temporal = (
    DIRECT_KG_SEED_METHOD[
        DIRECT_KG_SEED_METHOD[
            "method"
        ]
        ==
        "kg_temporal_guided"
    ]
    .set_index(
        "seed"
    )
)


comparison_rows = []


for metric in (
    "pooled_future_ap",
    "pooled_future_brier",
    "seizure_sensitivity",
    "false_alarms_per_hour",
):

    for comparison_name, difference in (
        (
            "state_minus_gnn",
            state[
                metric
            ]
            -
            base[
                metric
            ]
        ),
        (
            "temporal_minus_gnn",
            temporal[
                metric
            ]
            -
            base[
                metric
            ]
        ),
        (
            "temporal_minus_state",
            temporal[
                metric
            ]
            -
            state[
                metric
            ]
        ),
    ):

        comparison_rows.append(
            {
                "metric":
                    metric,

                "comparison":
                    comparison_name,

                "mean_difference":
                    float(
                        difference.mean()
                    ),

                "sd_difference":
                    float(
                        difference.std(
                            ddof=1
                        )
                    ),

                "min_difference":
                    float(
                        difference.min()
                    ),

                "max_difference":
                    float(
                        difference.max()
                    ),
            }
        )


DIRECT_KG_COMPARISONS = pd.DataFrame(
    comparison_rows
)


# ============================================================
# 18. PATIENT-LEVEL SUMMARY
# ============================================================

patient_rows = []


for patient in PATIENTS:

    for method in METHODS_DIRECT_KG:

        for seed in SEEDS_DIRECT_KG:

            subset = DIRECT_KG_PRED[
                (
                    DIRECT_KG_PRED[
                        "patient"
                    ]
                    ==
                    patient
                )
                &
                (
                    DIRECT_KG_PRED[
                        "method"
                    ]
                    ==
                    method
                )
                &
                (
                    DIRECT_KG_PRED[
                        "seed"
                    ]
                    ==
                    seed
                )
            ]


            y = (
                subset[
                    "label"
                ]
                .to_numpy(
                    dtype=np.int64
                )
            )

            probability = (
                subset[
                    "probability"
                ]
                .to_numpy(
                    dtype=np.float64
                )
            )


            if np.unique(
                y
            ).size == 2:

                ap = float(
                    average_precision_score(
                        y,
                        probability
                    )
                )

            else:

                ap = np.nan


            alarm = alarm_metrics_direct(
                subset
            )


            patient_rows.append(
                {
                    "seed":
                        seed,

                    "patient":
                        patient,

                    "method":
                        method,

                    "future_ap":
                        ap,

                    "seizure_sensitivity":
                        alarm[
                            "seizure_sensitivity"
                        ],

                    "false_alarms_per_hour":
                        alarm[
                            "false_alarms_per_hour"
                        ],
                }
            )


DIRECT_KG_PATIENT_SEED = pd.DataFrame(
    patient_rows
)


DIRECT_KG_PATIENT_SUMMARY = (
    DIRECT_KG_PATIENT_SEED
    .groupby(
        [
            "patient",
            "method"
        ],
        as_index=False
    )
    .agg(
        future_ap_mean=(
            "future_ap",
            "mean"
        ),

        future_ap_sd=(
            "future_ap",
            "std"
        ),

        sensitivity_mean=(
            "seizure_sensitivity",
            "mean"
        ),

        false_alarms_per_hour_mean=(
            "false_alarms_per_hour",
            "mean"
        ),
    )
)


# ============================================================
# 19. SAVE
# ============================================================

DIRECT_KG_PRED.to_pickle(
    OUT_DIRECT_KG
    / "future_predictions.pkl"
)

DIRECT_KG_PRED.to_csv(
    OUT_DIRECT_KG
    / "future_predictions.csv",
    index=False
)

DIRECT_KG_THRESHOLDS.to_csv(
    OUT_DIRECT_KG
    / "thresholds.csv",
    index=False
)

DIRECT_KG_EPISODES.to_csv(
    OUT_DIRECT_KG
    / "episode_metrics.csv",
    index=False
)

DIRECT_KG_SEED_METHOD.to_csv(
    OUT_DIRECT_KG
    / "seed_method_summary.csv",
    index=False
)

DIRECT_KG_SUMMARY.to_csv(
    OUT_DIRECT_KG
    / "five_seed_summary.csv",
    index=False
)

DIRECT_KG_COMPARISONS.to_csv(
    OUT_DIRECT_KG
    / "paired_seed_differences.csv",
    index=False
)

DIRECT_KG_PATIENT_SEED.to_csv(
    OUT_DIRECT_KG
    / "patient_seed_metrics.csv",
    index=False
)

DIRECT_KG_PATIENT_SUMMARY.to_csv(
    OUT_DIRECT_KG
    / "patient_summary.csv",
    index=False
)


# ============================================================
# 20. FINAL REPORT
# ============================================================

print(
    "\n"
    +
    "=" * 100
)

print(
    "FINAL DIRECT KG-GUIDED PREDICTION EXPERIMENT COMPLETE"
)

print(
    "=" * 100
)


print(
    f"Runtime: "
    f"{(time.time() - experiment_start) / 60.0:.1f} minutes"
)


print(
    "\nFIVE-SEED SUMMARY"
)

display(
    DIRECT_KG_SUMMARY
)


print(
    "\nPAIRED DIFFERENCES"
)

display(
    DIRECT_KG_COMPARISONS
)


print(
    "\nPATIENT-LEVEL SUMMARY"
)

display(
    DIRECT_KG_PATIENT_SUMMARY
)


print(
    "\nTHRESHOLDS"
)

display(
    DIRECT_KG_THRESHOLDS
)


print(
    "\nSaved to:"
)

print(
    OUT_DIRECT_KG
)
